In [ ]:
"""
phase_decomposition.py
======================
Spectral and phase decomposition of a single GPR trace, following the
methodology of Castagna et al. (2016) "Phase decomposition", Interpretation.

Edge-effect handling
--------------------
1.  Reflect-padding: the trace is mirrored at both ends by n_pad samples
    before the CWT, then trimmed back to the original length.  This pushes
    the COI boundary outside the signal, eliminating edge noise.

2.  COI boundary overlay: dashed white lines mark the time from each edge
    at which wavelet support falls to 1/e.  Data inside the COI should be
    interpreted with caution.
"""

import numpy as np
import matplotlib.pyplot as plt
import pywt


# ---------------------------------------------------------------------------
# Core computation
# ---------------------------------------------------------------------------

def phase_decomposition(
    trace,
    dt,
    f_min_GHz: float = 0.5,
    f_max_GHz: float = 5.0,
    n_scales: int = 60,
    n_theta: int = 181,
    wavelet: str = "cmor1.5-1.0",
    reflect_pad: bool = True,
):
    """
    Spectral and phase decomposition of a 1-D GPR trace.

    Parameters
    ----------
    trace : 1-D ndarray
    dt : float
        Sampling interval in nanoseconds.
    f_min_GHz, f_max_GHz : float
        Frequency band [GHz]. Default lower limit 0.5 GHz (no physical
        energy below that for a 1.5 GHz GPR system).
    n_scales : int
        Number of CWT scales (log-spaced across the frequency band).
    n_theta : int
        Number of phase angles in the gather (default 181).
    wavelet : str
        PyWavelets complex wavelet name.
    reflect_pad : bool
        If True (default), reflect-pad the trace before CWT to suppress
        edge artefacts (cone-of-influence noise) at low frequencies.

    Returns
    -------
    A : ndarray (n_scales, n_time)
    theta_2d_deg : ndarray (n_scales, n_time)
    phase_gather : ndarray (n_theta, n_time)
    freqs_GHz : ndarray (n_scales,)  -- ascending
    time_ns : ndarray (n_time,)
    theta_deg : ndarray (n_theta,)
    """
    trace = np.asarray(trace, dtype=float)
    n_time = len(trace)
    time_ns = np.arange(n_time) * dt

    # wavelet parameters
    cwavelet = pywt.ContinuousWavelet(wavelet)
    f_c = cwavelet.center_frequency
    B   = float(wavelet.split("-")[0].replace("cmor", ""))

    # scales: ascending freqs -> descending scales
    freq_axis = np.geomspace(f_min_GHz, f_max_GHz, n_scales)
    scales = f_c / (freq_axis * dt)

    # reflect-pad to suppress COI edge artefacts
    if reflect_pad:
        n_pad = int(np.ceil(2.0 * np.sqrt(B / 2.0) * scales.max()))
        n_pad = min(n_pad, n_time)
        trace_work = np.pad(trace, n_pad, mode="reflect")
    else:
        n_pad = 0
        trace_work = trace

    coeffs, freqs_GHz = pywt.cwt(trace_work, scales, wavelet, sampling_period=dt)

    if n_pad > 0:
        coeffs = coeffs[:, n_pad : n_pad + n_time]

    A             = np.abs(coeffs)
    theta_2d_deg  = np.rad2deg(np.angle(coeffs))

    R = np.trapezoid(np.real(coeffs), freqs_GHz, axis=0)
    I = np.trapezoid(np.imag(coeffs), freqs_GHz, axis=0)

    theta_deg = np.linspace(-180.0, 180.0, n_theta)
    theta_rad = np.deg2rad(theta_deg)
    phase_gather = (
        np.outer(np.cos(theta_rad), R)
        + np.outer(np.sin(theta_rad), I)
    )

    return A, theta_2d_deg, phase_gather, freqs_GHz, time_ns, theta_deg


# ---------------------------------------------------------------------------
# COI helper
# ---------------------------------------------------------------------------

def _coi_boundary(freqs_GHz, time_ns, B, f_c=1.0):
    """COI boundary time [ns] from each edge for each frequency."""
    coi = np.sqrt(B / 2.0) * f_c / freqs_GHz
    return np.clip(coi, 0, time_ns[-1] / 2.0)


# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------

def plot_decomposition(
    A,
    theta_2d_deg,
    phase_gather,
    freqs_GHz,
    time_ns,
    theta_deg,
    trace=None,
    title: str = "Spectral & Phase Decomposition",
    db_clip: float = 40.0,
    show_coi: bool = True,
    wavelet: str = "cmor1.5-1.0",
):
    """
    Three-panel figure: Amplitude Spectrum | Phase Spectrum | Phase Gather.

    Parameters
    ----------
    db_clip : float
        Dynamic range in dB for the amplitude display.
    show_coi : bool
        Overlay dashed white COI boundary lines on the spectral panels.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
    fig.suptitle(title, fontsize=13, fontweight="bold")

    t0, t1 = time_ns[0], time_ns[-1]
    f_left, f_right = freqs_GHz[0], freqs_GHz[-1]   # ascending
    extent_spec = [f_left, f_right, t1, t0]

    # -- Amplitude Spectrum (dB) -------------------------------------------
    ax = axes[0]
    A_norm = A / (A.max() + 1e-30)
    A_dB = np.clip(20 * np.log10(A_norm + 1e-30), -db_clip, 0.0)
    im0 = ax.imshow(
        A_dB.T, aspect="auto", origin="upper", extent=extent_spec,
        cmap="jet", vmin=-db_clip, vmax=0.0, interpolation="bilinear",
    )
    fig.colorbar(im0, ax=ax, fraction=0.046, pad=0.04).set_label("Amplitude [dB]", fontsize=9)
    ax.set_xlabel("Frequency [GHz]", fontsize=10)
    ax.set_ylabel("TWT [ns]", fontsize=10)
    ax.set_title("Amplitude Spectrum", fontsize=11)

    # -- Phase Spectrum (degrees) ------------------------------------------
    ax = axes[1]
    im1 = ax.imshow(
        theta_2d_deg.T, aspect="auto", origin="upper", extent=extent_spec,
        cmap="RdBu", vmin=-180.0, vmax=180.0, interpolation="nearest",
    )
    fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04).set_label("Phase [deg]", fontsize=9)
    ax.set_xlabel("Frequency [GHz]", fontsize=10)
    ax.set_title("Phase Spectrum", fontsize=11)

    # -- COI overlay on amplitude and phase panels -------------------------
    if show_coi:
        B_val = float(wavelet.split("-")[0].replace("cmor", ""))
        coi_t = _coi_boundary(freqs_GHz, time_ns, B_val, f_c=1.0)
        for ax_sp in (axes[0], axes[1]):
            ax_sp.plot(freqs_GHz, coi_t,               "w--", lw=1.0, alpha=0.75, label="COI boundary")
            ax_sp.plot(freqs_GHz, time_ns[-1] - coi_t, "w--", lw=1.0, alpha=0.75)

    # -- Phase Gather ------------------------------------------------------
    ax = axes[2]
    pg_clip = np.percentile(np.abs(phase_gather), 99)
    im2 = ax.imshow(
        phase_gather.T, aspect="auto", origin="upper",
        extent=[theta_deg[0], theta_deg[-1], t1, t0],
        cmap="RdBu", vmin=-pg_clip, vmax=pg_clip, interpolation="bilinear",
    )
    fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04).set_label("Amplitude", fontsize=9)
    ax.set_xlabel("Phase [deg]", fontsize=10)
    ax.set_title("Phase Gather  S'(theta, t)", fontsize=11)
    for ph in (-90, 90):
        ax.axvline(ph, color="lime", lw=1.0, ls="--", alpha=0.8)
    ax.axvline(0, color="white", lw=0.6, ls=":", alpha=0.5)
    ax.set_xticks([-180, -90, 0, 90, 180])

    plt.tight_layout()
    return fig


# ---------------------------------------------------------------------------
# Demo / standalone entry point
# ---------------------------------------------------------------------------

def _synthetic_ricker(n, dt, f0_GHz=1.5, t_peak_ns=None):
    """Ricker wavelet with added thin-layer interference."""
    if t_peak_ns is None:
        t_peak_ns = n * dt / 3
    t = np.arange(n) * dt
    u = np.pi * f0_GHz * (t - t_peak_ns)
    ricker = (1.0 - 2.0 * u**2) * np.exp(-(u**2))
    t_thin = t_peak_ns + 0.4 / f0_GHz
    u2 = np.pi * f0_GHz * (t - t_thin)
    thin_layer = 0.4 * (1.0 - 2.0 * u2**2) * np.exp(-(u2**2))
    noise = 0 #0.05 * np.random.default_rng(42).standard_normal(n)
    return ricker + thin_layer + noise


if __name__ == "__main__":
    dt = 0.004717
    n_t = 4241
    f_c_GHz = 1.0

    trace = _synthetic_ricker(n_t, dt, f0_GHz=f_c_GHz)
    A, theta_2d_deg, phase_gather, freqs_GHz, time_ns, theta_deg = phase_decomposition(
        trace, dt, f_min_GHz=0.5, f_max_GHz=5, n_scales=60, reflect_pad=True,
    )
    fig = plot_decomposition(
        A, theta_2d_deg, phase_gather, freqs_GHz, time_ns, theta_deg,
        trace=trace,
        title=f"Phase Decomposition -- synthetic GPR trace  (f_c = {f_c_GHz} GHz)",
        show_coi=True,
    )
    plt.show()


In [ ]:
# =============================================================================
# Synthetic GPR B-scan — linearly thinning fracture (constant impedance)
#
# Goal: isolate the effect of aperture on the GPR response.
#   x = 0      aperture = 2*lambda  (two full wavelengths, well-separated events)
#   x -> xmax  aperture decreases linearly to sub-Rayleigh (< lambda/4)
#
# Impedance contrast is CONSTANT along the profile so any amplitude/phase
# variation is caused purely by thin-bed interference, not by impedance change.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Acquisition ────────────────────────────────────────────────────────────────
dt  = 0.004717      # ns
f_c = 1.5           # GHz — GPR centre frequency
n_t = 2000          # samples (~7.1 ns record)
n_x = 80            # lateral traces
dx  = 0.05          # m per trace

t = np.arange(n_t) * dt
x = np.arange(n_x) * dx

# ── Key time constants ─────────────────────────────────────────────────────────
T_lam  = 1.0 / f_c          # dominant period  = 0.667 ns  (= lambda in TWT)
T_tune = T_lam / 4.0        # Rayleigh tuning criterion = 0.167 ns

# ── Ricker wavelet ─────────────────────────────────────────────────────────────
n_wav = 251
t_wav = (np.arange(n_wav) - n_wav // 2) * dt
u_wav = np.pi * f_c * t_wav
ricker = (1.0 - 2.0 * u_wav**2) * np.exp(-u_wav**2)

# ── Fracture: constant impedance, linearly decreasing aperture ─────────────────
Z_rock  = 1.0
Z_fill  = 0.50                                     # constant fill impedance
R_top   = (Z_fill - Z_rock) / (Z_fill + Z_rock)   # = -1/3  (constant)

ap_max  = 2.0 * T_lam    # 1.333 ns  — two full wavelengths at x = 0
ap_min  = 0.015          # ~0.09*T_tune — far below Rayleigh at x[-1]
aperture = np.linspace(ap_max, ap_min, n_x)

T0 = np.full(n_x, 4.5)    # flat fracture at 4.5 ns

# Position where aperture crosses the Rayleigh criterion
i_rayleigh = int(np.argmin(np.abs(aperture - T_tune)))
x_rayleigh = x[i_rayleigh]

# ── Selected traces at physically meaningful apertures ─────────────────────────
target_ap  = [2.0*T_lam, T_lam, T_lam/2, T_tune, T_lam/8]
target_lbl = [
    f"2λ  (thick bed)",
    f"λ  (1 period)",
    f"λ/2  (peak amplitude)",
    f"λ/4  (Rayleigh limit)",
    f"λ/8  (sub-Rayleigh)",
]
pd_idx    = [int(np.argmin(np.abs(aperture - a))) for a in target_ap]
pd_labels = target_lbl
pd_colors = ["royalblue", "forestgreen", "darkorange", "crimson", "purple"]

# ── Convolutional synthesis ────────────────────────────────────────────────────
rng   = np.random.default_rng(7)
bscan = np.zeros((n_x, n_t))
for i in range(n_x):
    ref        = np.zeros(n_t)
    i_top      = min(int(round(T0[i] / dt)), n_t - 1)
    i_bot      = min(i_top + max(1, int(round(aperture[i] / dt))), n_t - 1)
    ref[i_top] =  R_top
    ref[i_bot] = -R_top
    bscan[i]   = np.convolve(ref, ricker, mode="same")

noise_level = 0.02 * np.abs(bscan).max()
bscan += noise_level * rng.standard_normal(bscan.shape)

# ── Figure: aperture profile + B-scan ─────────────────────────────────────────
fig = plt.figure(figsize=(13, 8))
gs  = gridspec.GridSpec(2, 1, height_ratios=[0.25, 1.0], hspace=0.35)

# Panel 1: aperture profile
ax0 = fig.add_subplot(gs[0])
ax0.plot(x, aperture / T_tune, "k-", lw=1.8)
ax0.axhline(1.0, color="crimson", ls="--", lw=1.2, label="Rayleigh criterion (T_tune = λ/4)")
ax0.axhline(4.0, color="royalblue", ls=":",  lw=0.9, label="2λ")
ax0.fill_between(x, aperture / T_tune, 1.0,
                 where=(aperture < T_tune), alpha=0.15, color="crimson",
                 label="sub-Rayleigh zone")
ax0.set_ylabel("Aperture / T_tune", fontsize=9)
ax0.set_title("Fracture aperture profile  (R_top = {:.3f} everywhere)".format(R_top), fontsize=10)
ax0.legend(fontsize=8, loc="upper right")
ax0.set_xlim(x[0], x[-1])
ax0.set_ylim(0, None)
for ix, col, lbl in zip(pd_idx, pd_colors, pd_labels):
    ax0.axvline(x[ix], color=col, lw=1.1, ls="--", alpha=0.8)

# Panel 2: B-scan
ax1 = fig.add_subplot(gs[1])
clip = np.percentile(np.abs(bscan), 98)
ax1.imshow(
    bscan.T, aspect="auto", origin="upper",
    extent=[x[0], x[-1], t[-1], t[0]],
    cmap="RdBu", vmin=-clip, vmax=clip, interpolation="bilinear",
)
ax1.axvline(x_rayleigh, color="crimson", lw=1.5, ls="--", alpha=0.9,
            label=f"Rayleigh limit  x = {x_rayleigh:.2f} m")
ax1.set_xlabel("Position [m]", fontsize=10)
ax1.set_ylabel("TWT [ns]", fontsize=10)
ax1.set_title("Synthetic GPR B-scan — thinning fracture (constant impedance)", fontsize=11)
for ix, col, lbl in zip(pd_idx, pd_colors, pd_labels):
    ax1.axvline(x[ix], color=col, lw=1.2, ls="--", alpha=0.9, label=lbl)
ax1.legend(fontsize=8, loc="upper right")
ax1.set_ylim(T0[0] + 1.5, T0[0] - 0.8)   # zoom to fracture zone

plt.suptitle("Thinning Fracture B-scan  —  Convolutional Model", fontsize=12,
             fontweight="bold")
plt.tight_layout()
plt.show()

# Print aperture at each selected trace
print(f"  {'Trace':<28} {'x[m]':>6} {'ap[ps]':>8} {'ap/T_tune':>10}")
print("  " + "-"*56)
for lbl, ix in zip(pd_labels, pd_idx):
    print(f"  {lbl:<28} {x[ix]:>6.2f} {aperture[ix]*1e3:>8.1f} {aperture[ix]/T_tune:>10.2f}x")


In [ ]:
# =============================================================================
# Phase decomposition of the five selected traces
# =============================================================================

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace = bscan[ix]
    ap_ratio = aperture[ix] / T_tune

    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = phase_decomposition(
        trace, dt,
        f_min_GHz=0.5, f_max_GHz=4.5, n_scales=60,
        reflect_pad=True,
    )
    fig = plot_decomposition(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=(
            f"Phase Decomp — {lbl}  "
            f"(aperture = {aperture[ix]*1e3:.0f} ps = {ap_ratio:.2f}*T_tune)"
        ),
        show_coi=True,
    )
    plt.show()


In [ ]:
# =============================================================================
# Phase-selective B-scan reconstruction  —  ±90° (thin-bed) component
# =============================================================================

import pywt
import numpy as np
import matplotlib.pyplot as plt

# ── CWT: build R(t) and I(t) for every trace ──────────────────────────────────
wav   = "cmor1.5-1.0"
cwav  = pywt.ContinuousWavelet(wav)
f_c_w = cwav.center_frequency
B_w   = float(wav.split("-")[0].replace("cmor", ""))

scales_pd = f_c_w / (np.geomspace(0.5, 4.5, 60) * dt)
n_pad_pd  = min(int(np.ceil(2.0 * np.sqrt(B_w / 2.0) * scales_pd.max())), n_t)

bscan_R = np.zeros((n_x, n_t))
bscan_I = np.zeros((n_x, n_t))

for i in range(n_x):
    tr_pad         = np.pad(bscan[i], n_pad_pd, mode="reflect")
    coeffs, freqs_ = pywt.cwt(tr_pad, scales_pd, wav, sampling_period=dt)
    coeffs         = coeffs[:, n_pad_pd : n_pad_pd + n_t]
    bscan_R[i]     = np.trapezoid(np.real(coeffs), freqs_, axis=0)
    bscan_I[i]     = np.trapezoid(np.imag(coeffs), freqs_, axis=0)

envelope  = np.sqrt(bscan_R**2 + bscan_I**2)
bscan_sin = bscan_I / (envelope + 1e-30)

amp_threshold    = 0.01
mask             = envelope > amp_threshold * envelope.max()
bscan_sin_masked = np.where(mask, bscan_sin, np.nan)

# ── Figure 1: 5-panel B-scan comparison ───────────────────────────────────────
t_lo, t_hi = T0[0] - 0.8, T0[0] + 1.6
i_lo, i_hi = int(t_lo / dt), int(t_hi / dt)
ext         = [x[0], x[-1], t[i_hi - 1], t[i_lo]]

fig, axes = plt.subplots(1, 5, figsize=(24, 5), sharey=True, sharex=True)
fig.suptitle("Phase-selective B-scan reconstruction  (fracture time window)",
             fontsize=12, fontweight="bold")

panels = [
    (bscan,            "RdBu",  "Original B-scan"),
    (bscan_R,          "RdBu",  "theta = 0 deg   in-phase  R(t)"),
    (bscan_I,          "RdBu",  "theta = +90 deg   I(t)"),
    (-bscan_I,         "RdBu",  "theta = -90 deg   -I(t)"),
    (bscan_sin_masked, "RdBu",  f"sin(theta_dom) masked (envelope > {amp_threshold:.0%})"),
]

for ax, (data, cmap, title) in zip(axes, panels):
    vabs = 0.95 if data is bscan_sin_masked else np.percentile(np.abs(data), 98)
    im = ax.imshow(data[:, i_lo:i_hi].T, aspect="auto", origin="upper",
                   extent=ext, cmap=cmap, vmin=-vabs, vmax=vabs,
                   interpolation="bilinear")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axvline(x_rayleigh, color="crimson", lw=1.5, ls="--", alpha=0.9)
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_title(title, fontsize=9)
    for ix, col in zip(pd_idx, pd_colors):
        ax.axvline(x[ix], color=col, lw=0.9, ls=":", alpha=0.8)

axes[0].set_ylabel("TWT [ns]", fontsize=10)
axes[0].text(x_rayleigh + 0.05, t_lo + 0.15, "lambda/4", color="crimson", fontsize=9)
plt.tight_layout()
plt.show()

# ── Compute the TRUE tuning thickness from the noise-free Ricker pair ──────────
# Sweep aperture finely, convolve noise-free pair, record peak amplitude.
# The tuning peak is where the amplitude is maximum: this is the actual Rayleigh
# criterion for this wavelet (not necessarily exactly lambda/4).
n_wav_tc = 251
t_wav_tc  = (np.arange(n_wav_tc) - n_wav_tc // 2) * dt
ricker_tc = (1.0 - 2.0*(np.pi*f_c*t_wav_tc)**2) * np.exp(-(np.pi*f_c*t_wav_tc)**2)

deltas    = np.linspace(0, ap_max, 300)
amp_tune  = np.zeros(len(deltas))
ref_tc    = np.zeros(n_t)
i_top_tc  = int(round(T0[0] / dt))

for k, delta in enumerate(deltas):
    ref_tc[:] = 0.0
    ref_tc[i_top_tc] = R_top
    i_bot_tc = min(i_top_tc + max(1, int(round(delta / dt))), n_t - 1)
    ref_tc[i_bot_tc] = -R_top
    tr_tc = np.convolve(ref_tc, ricker_tc, mode="same")
    win_l = max(0,     i_top_tc - int(0.3 / dt))
    win_r = min(n_t-1, i_top_tc + int(1.5 * T_lam / dt))
    amp_tune[k] = np.abs(tr_tc[win_l:win_r]).max()

i_tune_peak   = np.argmax(amp_tune)
ap_tune_peak  = deltas[i_tune_peak]          # actual tuning thickness [ns]
ap_tune_ratio = ap_tune_peak / T_tune        # in units of lambda/4

# ── Figure 2: THE TUNING CURVE ─────────────────────────────────────────────────
# Use raw peak amplitude from bscan (not CWT envelope) for apples-to-apples
# comparison with the noise-free reference curve above.
t0_idx  = int(round(T0[0] / dt))
win_lo  = max(0,     t0_idx - int(0.3  / dt))
win_hi  = min(n_t-1, t0_idx + int(1.5 * T_lam / dt))

amp_raw_at_frac = np.array([np.abs(bscan[i, win_lo:win_hi]).max() for i in range(n_x)])
sin_at_frac     = np.array([
    bscan_sin_masked[i, win_lo + np.argmax(np.abs(bscan[i, win_lo:win_hi]))]
    for i in range(n_x)
])

amp_raw_norm  = amp_raw_at_frac / amp_raw_at_frac.max()
amp_tune_norm = amp_tune / amp_tune.max()
ap_ratio      = aperture / T_tune
delta_ratio   = deltas   / T_tune

fig2, ax2 = plt.subplots(figsize=(9, 5))
fig2.suptitle("The tuning curve  —  raw amplitude vs phase indicator", fontsize=12,
              fontweight="bold")

ax2.plot(delta_ratio, amp_tune_norm,       "k--", lw=1.5, alpha=0.7,
         label="Noise-free tuning curve (Ricker pair)")
ax2.plot(ap_ratio,    amp_raw_norm,        "k-",  lw=2.0,
         label="Peak amplitude at fracture (bscan)")
ax2.plot(ap_ratio,    np.abs(sin_at_frac), "m-",  lw=2.0,
         label="|sin(theta_dom)|  at fracture")

# Mark lambda/4 (theoretical) and actual tuning peak
ax2.axvline(1.0,           color="steelblue", ls="--", lw=1.5,
            label=f"lambda/4  (T_tune = {T_tune*1e3:.0f} ps, theoretical)")
ax2.axvline(ap_tune_ratio, color="crimson",   ls="-",  lw=1.5,
            label=f"Actual tuning peak = {ap_tune_ratio:.2f} * T_tune"
                  f"  ({ap_tune_peak*1e3:.0f} ps)")

ax2.fill_betweenx([0, 1.05], 0, ap_tune_ratio,
                  alpha=0.07, color="crimson", label="Sub-tuning zone")

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    ax2.axvline(ap_ratio[ix], color=col, lw=1.0, ls=":", alpha=0.9)
    ax2.text(ap_ratio[ix] + 0.05, 0.88, lbl.split("(")[0].strip(),
             color=col, fontsize=8, rotation=90, va="top")

ax2.set_xlabel("Aperture / T_tune  (= aperture / [lambda/4])", fontsize=10)
ax2.set_ylabel("Normalised value", fontsize=10)
ax2.set_xlim(0, ap_ratio.max())
ax2.set_ylim(0, 1.08)
ax2.legend(fontsize=8, loc="upper right")
ax2.set_title(
    "Amplitude peaks at the actual tuning thickness (red line), "
    "not necessarily at lambda/4 (blue dashed)\n"
    "Left of red: amplitude drops, |sin(theta_dom)| stays near 1  "
    "-> phase detects beds amplitude cannot resolve",
    fontsize=9,
)
plt.tight_layout()
plt.show()

print(f"Theoretical lambda/4      : T_tune = {T_tune*1e3:.1f} ps = T_lam / 4")
print(f"Actual tuning peak (Ricker): {ap_tune_peak*1e3:.1f} ps = {ap_tune_ratio:.2f} * T_tune "
      f"= {ap_tune_peak/T_lam:.3f} * T_lam")

# ── Figure 3: Trace overlays at the five key positions ────────────────────────
fig3, axs3 = plt.subplots(1, 5, figsize=(18, 6), sharey=True)
fig3.suptitle("Composite waveform at key apertures  (normalised per trace)", fontsize=11,
              fontweight="bold")

for ax, lbl, ix, col in zip(axs3, pd_labels, pd_idx, pd_colors):
    tr   = bscan[ix]
    norm = max(np.abs(tr).max(), 1e-9)
    ax.plot(tr / norm, t, color=col, lw=1.4)
    ax.axhline(T0[ix],               color="gray", lw=0.7, ls="--")
    ax.axhline(T0[ix] + aperture[ix], color="gray", lw=0.7, ls="--")
    ax.axvline(0, color="black", lw=0.4)
    ax.invert_yaxis()
    ax.set_ylim(T0[0] + 1.6, T0[0] - 0.6)
    ax.set_xlim(-1.3, 1.3)
    ax.set_title(lbl, fontsize=9, color=col)
    ax.set_xlabel("Norm. amp.", fontsize=8)
    if ax is axs3[0]:
        ax.set_ylabel("TWT [ns]", fontsize=9)
    ax.text(0.05, 0.02, f"{aperture[ix]*1e3:.0f} ps\n= {aperture[ix]/T_tune:.2f}*lambda/4",
            transform=ax.transAxes, fontsize=8, va="bottom")

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# CLSSA Phase Decomposition — Constrained Least-Squares Spectral Analysis
#
# Why CLSSA instead of CWT?
#   CWT: window width scales with 1/frequency -> low-frequency components have
#        a large temporal footprint (COI), smearing thin-bed events in time.
#   CLSSA: fixed-length sliding window applied to the KERNEL (tapered sinusoids),
#          not to the data.  The inversion recovers the raw-trace spectrum with
#          uniform time resolution at all frequencies.  No COI, no edge smearing.
#
# Forward model for window centred at t_c:
#   d = G m        G[j,k] = h[j] * exp(i 2pi f_k (t_j - t_c))
#   Solution: m = (G_n^H G_n + alpha I)^{-1} G_n^H d / col_norms
#
# -90 deg phase anomaly = diagnostic signature of a sub-resolution
# low-impedance thin layer (Castagna et al. 2016).
# =============================================================================

import sys, os, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import phase_decomposition as pd_mod
importlib.reload(pd_mod)   # pick up any edits without kernel restart

import numpy as np
import matplotlib.pyplot as plt
import time as _time

# ── CLSSA parameters ───────────────────────────────────────────────────────────
CLSSA_PARAMS = dict(
    f_min_GHz = 0.1,
    f_max_GHz = 4.0,
    n_freqs   = 60,
    win_ns    = 2.0,    # sliding window length [ns]  — ~7.5 dominant periods
    alpha     = 1e-2,   # Tikhonov regularisation
)

print("CLSSA parameters:")
for k, v in CLSSA_PARAMS.items():
    print(f"  {k:<12} = {v}")
print()

# ── Run CLSSA on each selected trace ──────────────────────────────────────────
for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace    = bscan[ix]
    ap_ratio = aperture[ix] / T_tune

    t0_run = _time.time()
    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = (
        pd_mod.clssa_phase_decomposition(trace, dt, **CLSSA_PARAMS)
    )
    elapsed = _time.time() - t0_run

    fig = pd_mod.plot_clssa_decomposition(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=(
            f"CLSSA — {lbl}  |  "
            f"aperture = {aperture[ix]*1e3:.0f} ps = {ap_ratio:.2f}*T_tune  "
            f"({elapsed:.1f} s)"
        ),
        highlight_neg90=True,
    )
    plt.show()

# ── Side-by-side comparison: CWT vs CLSSA on the lambda/4 trace ───────────────
print("\nComparing CWT vs CLSSA on the Rayleigh-limit trace (lambda/4)...")
ix_comp = pd_idx[3]   # lambda/4 trace

# CWT
A_cwt, th_cwt, pg_cwt, fq_cwt, tn_cwt, td_cwt = pd_mod.phase_decomposition(
    bscan[ix_comp], dt,
    f_min_GHz=0.5, f_max_GHz=4.0, n_scales=60, reflect_pad=True,
)
# CLSSA
A_cl, th_cl, pg_cl, fq_cl, tn_cl, td_cl = pd_mod.clssa_phase_decomposition(
    bscan[ix_comp], dt, **CLSSA_PARAMS
)

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 10), sharey=True)
fig2.suptitle(
    f"CWT vs CLSSA  —  {pd_labels[3]}  "
    f"(aperture = {aperture[ix_comp]*1e3:.0f} ps = {aperture[ix_comp]/T_tune:.2f}*T_tune)",
    fontsize=12, fontweight="bold",
)

t0v, t1v = tn_cwt[0], tn_cwt[-1]

def _fill_row(axes_row, A, th, pg, fq, tn, td, method):
    db_clip = 40.0
    f0, f1   = fq[0], fq[-1]
    ext_sp   = [f0, f1, t1v, t0v]

    A_dB = np.clip(20*np.log10(A / (A.max()+1e-30) + 1e-30), -db_clip, 0)
    im0 = axes_row[0].imshow(A_dB.T, aspect="auto", origin="upper",
                              extent=ext_sp, cmap="jet",
                              vmin=-db_clip, vmax=0, interpolation="bilinear")
    fig2.colorbar(im0, ax=axes_row[0], fraction=0.046, pad=0.04).set_label("dB", fontsize=8)
    axes_row[0].set_title(f"{method} — Amplitude", fontsize=10)
    axes_row[0].set_xlabel("Frequency [GHz]", fontsize=9)
    axes_row[0].set_ylabel("TWT [ns]", fontsize=9)

    im1 = axes_row[1].imshow(th.T, aspect="auto", origin="upper",
                              extent=ext_sp, cmap="RdBu",
                              vmin=-180, vmax=180, interpolation="nearest")
    fig2.colorbar(im1, ax=axes_row[1], fraction=0.046, pad=0.04).set_label("deg", fontsize=8)
    axes_row[1].set_title(f"{method} — Phase", fontsize=10)
    axes_row[1].set_xlabel("Frequency [GHz]", fontsize=9)

    pg_c = np.percentile(np.abs(pg), 99)
    im2 = axes_row[2].imshow(pg.T, aspect="auto", origin="upper",
                              extent=[td[0], td[-1], t1v, t0v],
                              cmap="RdBu", vmin=-pg_c, vmax=pg_c,
                              interpolation="bilinear")
    fig2.colorbar(im2, ax=axes_row[2], fraction=0.046, pad=0.04).set_label("Amp", fontsize=8)
    axes_row[2].set_title(f"{method} — Phase Gather", fontsize=10)
    axes_row[2].set_xlabel("Phase [deg]", fontsize=9)
    axes_row[2].axvline(-90, color="magenta", lw=1.5, ls="--")
    axes_row[2].axvline( 90, color="lime",    lw=1.0, ls="--")
    axes_row[2].axvspan(-105, -75, color="magenta", alpha=0.15)
    axes_row[2].set_xticks([-180, -90, 0, 90, 180])

_fill_row(axes2[0], A_cwt, th_cwt, pg_cwt, fq_cwt, tn_cwt, td_cwt, "CWT")
_fill_row(axes2[1], A_cl,  th_cl,  pg_cl,  fq_cl,  tn_cl,  td_cl,  "CLSSA")

plt.tight_layout()
plt.show()

# ── Dominant-phase diagnostic for each selected trace ─────────────────────────
# The phase gather's inclined bands reflect carrier-frequency rotation (dtheta/dt ~ 2pi*f_c).
# plot_dominant_phase collapses the 2-D phase gather to a 1-D line:
#   theta_dom(t) = arctan2(I(t), R(t))
# A phase jump TO -90 deg at the event time = thin-bed / sub-resolution signature.

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace = bscan[ix]
    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = (
        pd_mod.clssa_phase_decomposition(trace, dt, **CLSSA_PARAMS)
    )
    fig = pd_mod.plot_dominant_phase(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=f"Dominant phase — {lbl}  |  ap={aperture[ix]*1e3:.0f} ps = {aperture[ix]/T_tune:.2f}*T_tune",
    )
    plt.show()


In [ ]:
# =============================================================================
# Phase decomposition of the five selected traces
# =============================================================================

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace = bscan[ix]
    ap_ratio = aperture[ix] / T_tune

    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = phase_decomposition(
        trace, dt,
        f_min_GHz=0.5, f_max_GHz=4.5, n_scales=60,
        reflect_pad=True,
    )
    fig = plot_decomposition(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=(
            f"Phase Decomp — {lbl}  "
            f"(aperture = {aperture[ix]*1e3:.0f} ps = {ap_ratio:.2f}*T_tune)"
        ),
        show_coi=True,
    )
    plt.show()


In [ ]:
# =============================================================================
# Phase-selective B-scan reconstruction  —  ±90° (thin-bed) component
# =============================================================================

import pywt
import numpy as np
import matplotlib.pyplot as plt

# ── CWT: build R(t) and I(t) for every trace ──────────────────────────────────
wav   = "cmor1.5-1.0"
cwav  = pywt.ContinuousWavelet(wav)
f_c_w = cwav.center_frequency
B_w   = float(wav.split("-")[0].replace("cmor", ""))

scales_pd = f_c_w / (np.geomspace(0.5, 4.5, 60) * dt)
n_pad_pd  = min(int(np.ceil(2.0 * np.sqrt(B_w / 2.0) * scales_pd.max())), n_t)

bscan_R = np.zeros((n_x, n_t))
bscan_I = np.zeros((n_x, n_t))

for i in range(n_x):
    tr_pad         = np.pad(bscan[i], n_pad_pd, mode="reflect")
    coeffs, freqs_ = pywt.cwt(tr_pad, scales_pd, wav, sampling_period=dt)
    coeffs         = coeffs[:, n_pad_pd : n_pad_pd + n_t]
    bscan_R[i]     = np.trapezoid(np.real(coeffs), freqs_, axis=0)
    bscan_I[i]     = np.trapezoid(np.imag(coeffs), freqs_, axis=0)

envelope  = np.sqrt(bscan_R**2 + bscan_I**2)
bscan_sin = bscan_I / (envelope + 1e-30)

amp_threshold    = 0.01
mask             = envelope > amp_threshold * envelope.max()
bscan_sin_masked = np.where(mask, bscan_sin, np.nan)

# ── Figure 1: 5-panel B-scan comparison ───────────────────────────────────────
t_lo, t_hi = T0[0] - 0.8, T0[0] + 1.6
i_lo, i_hi = int(t_lo / dt), int(t_hi / dt)
ext         = [x[0], x[-1], t[i_hi - 1], t[i_lo]]

fig, axes = plt.subplots(1, 5, figsize=(24, 5), sharey=True, sharex=True)
fig.suptitle("Phase-selective B-scan reconstruction  (fracture time window)",
             fontsize=12, fontweight="bold")

panels = [
    (bscan,            "RdBu",  "Original B-scan"),
    (bscan_R,          "RdBu",  "theta = 0 deg   in-phase  R(t)"),
    (bscan_I,          "RdBu",  "theta = +90 deg   I(t)"),
    (-bscan_I,         "RdBu",  "theta = -90 deg   -I(t)"),
    (bscan_sin_masked, "RdBu",  f"sin(theta_dom) masked (envelope > {amp_threshold:.0%})"),
]

for ax, (data, cmap, title) in zip(axes, panels):
    vabs = 0.95 if data is bscan_sin_masked else np.percentile(np.abs(data), 98)
    im = ax.imshow(data[:, i_lo:i_hi].T, aspect="auto", origin="upper",
                   extent=ext, cmap=cmap, vmin=-vabs, vmax=vabs,
                   interpolation="bilinear")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axvline(x_rayleigh, color="crimson", lw=1.5, ls="--", alpha=0.9)
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_title(title, fontsize=9)
    for ix, col in zip(pd_idx, pd_colors):
        ax.axvline(x[ix], color=col, lw=0.9, ls=":", alpha=0.8)

axes[0].set_ylabel("TWT [ns]", fontsize=10)
axes[0].text(x_rayleigh + 0.05, t_lo + 0.15, "lambda/4", color="crimson", fontsize=9)
plt.tight_layout()
plt.show()

# ── Compute the TRUE tuning thickness from the noise-free Ricker pair ──────────
# Sweep aperture finely, convolve noise-free pair, record peak amplitude.
# The tuning peak is where the amplitude is maximum: this is the actual Rayleigh
# criterion for this wavelet (not necessarily exactly lambda/4).
n_wav_tc = 251
t_wav_tc  = (np.arange(n_wav_tc) - n_wav_tc // 2) * dt
ricker_tc = (1.0 - 2.0*(np.pi*f_c*t_wav_tc)**2) * np.exp(-(np.pi*f_c*t_wav_tc)**2)

deltas    = np.linspace(0, ap_max, 300)
amp_tune  = np.zeros(len(deltas))
ref_tc    = np.zeros(n_t)
i_top_tc  = int(round(T0[0] / dt))

for k, delta in enumerate(deltas):
    ref_tc[:] = 0.0
    ref_tc[i_top_tc] = R_top
    i_bot_tc = min(i_top_tc + max(1, int(round(delta / dt))), n_t - 1)
    ref_tc[i_bot_tc] = -R_top
    tr_tc = np.convolve(ref_tc, ricker_tc, mode="same")
    win_l = max(0,     i_top_tc - int(0.3 / dt))
    win_r = min(n_t-1, i_top_tc + int(1.5 * T_lam / dt))
    amp_tune[k] = np.abs(tr_tc[win_l:win_r]).max()

i_tune_peak   = np.argmax(amp_tune)
ap_tune_peak  = deltas[i_tune_peak]          # actual tuning thickness [ns]
ap_tune_ratio = ap_tune_peak / T_tune        # in units of lambda/4

# ── Figure 2: THE TUNING CURVE ─────────────────────────────────────────────────
# Use raw peak amplitude from bscan (not CWT envelope) for apples-to-apples
# comparison with the noise-free reference curve above.
t0_idx  = int(round(T0[0] / dt))
win_lo  = max(0,     t0_idx - int(0.3  / dt))
win_hi  = min(n_t-1, t0_idx + int(1.5 * T_lam / dt))

amp_raw_at_frac = np.array([np.abs(bscan[i, win_lo:win_hi]).max() for i in range(n_x)])
sin_at_frac     = np.array([
    bscan_sin_masked[i, win_lo + np.argmax(np.abs(bscan[i, win_lo:win_hi]))]
    for i in range(n_x)
])

amp_raw_norm  = amp_raw_at_frac / amp_raw_at_frac.max()
amp_tune_norm = amp_tune / amp_tune.max()
ap_ratio      = aperture / T_tune
delta_ratio   = deltas   / T_tune

fig2, ax2 = plt.subplots(figsize=(9, 5))
fig2.suptitle("The tuning curve  —  raw amplitude vs phase indicator", fontsize=12,
              fontweight="bold")

ax2.plot(delta_ratio, amp_tune_norm,       "k--", lw=1.5, alpha=0.7,
         label="Noise-free tuning curve (Ricker pair)")
ax2.plot(ap_ratio,    amp_raw_norm,        "k-",  lw=2.0,
         label="Peak amplitude at fracture (bscan)")
ax2.plot(ap_ratio,    np.abs(sin_at_frac), "m-",  lw=2.0,
         label="|sin(theta_dom)|  at fracture")

# Mark lambda/4 (theoretical) and actual tuning peak
ax2.axvline(1.0,           color="steelblue", ls="--", lw=1.5,
            label=f"lambda/4  (T_tune = {T_tune*1e3:.0f} ps, theoretical)")
ax2.axvline(ap_tune_ratio, color="crimson",   ls="-",  lw=1.5,
            label=f"Actual tuning peak = {ap_tune_ratio:.2f} * T_tune"
                  f"  ({ap_tune_peak*1e3:.0f} ps)")

ax2.fill_betweenx([0, 1.05], 0, ap_tune_ratio,
                  alpha=0.07, color="crimson", label="Sub-tuning zone")

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    ax2.axvline(ap_ratio[ix], color=col, lw=1.0, ls=":", alpha=0.9)
    ax2.text(ap_ratio[ix] + 0.05, 0.88, lbl.split("(")[0].strip(),
             color=col, fontsize=8, rotation=90, va="top")

ax2.set_xlabel("Aperture / T_tune  (= aperture / [lambda/4])", fontsize=10)
ax2.set_ylabel("Normalised value", fontsize=10)
ax2.set_xlim(0, ap_ratio.max())
ax2.set_ylim(0, 1.08)
ax2.legend(fontsize=8, loc="upper right")
ax2.set_title(
    "Amplitude peaks at the actual tuning thickness (red line), "
    "not necessarily at lambda/4 (blue dashed)\n"
    "Left of red: amplitude drops, |sin(theta_dom)| stays near 1  "
    "-> phase detects beds amplitude cannot resolve",
    fontsize=9,
)
plt.tight_layout()
plt.show()

print(f"Theoretical lambda/4      : T_tune = {T_tune*1e3:.1f} ps = T_lam / 4")
print(f"Actual tuning peak (Ricker): {ap_tune_peak*1e3:.1f} ps = {ap_tune_ratio:.2f} * T_tune "
      f"= {ap_tune_peak/T_lam:.3f} * T_lam")

# ── Figure 3: Trace overlays at the five key positions ────────────────────────
fig3, axs3 = plt.subplots(1, 5, figsize=(18, 6), sharey=True)
fig3.suptitle("Composite waveform at key apertures  (normalised per trace)", fontsize=11,
              fontweight="bold")

for ax, lbl, ix, col in zip(axs3, pd_labels, pd_idx, pd_colors):
    tr   = bscan[ix]
    norm = max(np.abs(tr).max(), 1e-9)
    ax.plot(tr / norm, t, color=col, lw=1.4)
    ax.axhline(T0[ix],               color="gray", lw=0.7, ls="--")
    ax.axhline(T0[ix] + aperture[ix], color="gray", lw=0.7, ls="--")
    ax.axvline(0, color="black", lw=0.4)
    ax.invert_yaxis()
    ax.set_ylim(T0[0] + 1.6, T0[0] - 0.6)
    ax.set_xlim(-1.3, 1.3)
    ax.set_title(lbl, fontsize=9, color=col)
    ax.set_xlabel("Norm. amp.", fontsize=8)
    if ax is axs3[0]:
        ax.set_ylabel("TWT [ns]", fontsize=9)
    ax.text(0.05, 0.02, f"{aperture[ix]*1e3:.0f} ps\n= {aperture[ix]/T_tune:.2f}*lambda/4",
            transform=ax.transAxes, fontsize=8, va="bottom")

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# CLSSA Phase Decomposition — Constrained Least-Squares Spectral Analysis
#
# Why CLSSA instead of CWT?
#   CWT: window width scales with 1/frequency -> low-frequency components have
#        a large temporal footprint (COI), smearing thin-bed events in time.
#   CLSSA: fixed-length sliding window applied to the KERNEL (tapered sinusoids),
#          not to the data.  The inversion recovers the raw-trace spectrum with
#          uniform time resolution at all frequencies.  No COI, no edge smearing.
#
# Forward model for window centred at t_c:
#   d = G m        G[j,k] = h[j] * exp(i 2pi f_k (t_j - t_c))
#   Solution: m = (G_n^H G_n + alpha I)^{-1} G_n^H d / col_norms
#
# -90 deg phase anomaly = diagnostic signature of a sub-resolution
# low-impedance thin layer (Castagna et al. 2016).
# =============================================================================

import sys, os, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import phase_decomposition as pd_mod
importlib.reload(pd_mod)   # pick up any edits without kernel restart

import numpy as np
import matplotlib.pyplot as plt
import time as _time

# ── CLSSA parameters ───────────────────────────────────────────────────────────
CLSSA_PARAMS = dict(
    f_min_GHz = 0.1,
    f_max_GHz = 4.0,
    n_freqs   = 60,
    win_ns    = 2.0,    # sliding window length [ns]  — ~7.5 dominant periods
    alpha     = 1e-2,   # Tikhonov regularisation
)

print("CLSSA parameters:")
for k, v in CLSSA_PARAMS.items():
    print(f"  {k:<12} = {v}")
print()

# ── Run CLSSA on each selected trace ──────────────────────────────────────────
for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace    = bscan[ix]
    ap_ratio = aperture[ix] / T_tune

    t0_run = _time.time()
    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = (
        pd_mod.clssa_phase_decomposition(trace, dt, **CLSSA_PARAMS)
    )
    elapsed = _time.time() - t0_run

    fig = pd_mod.plot_clssa_decomposition(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=(
            f"CLSSA — {lbl}  |  "
            f"aperture = {aperture[ix]*1e3:.0f} ps = {ap_ratio:.2f}*T_tune  "
            f"({elapsed:.1f} s)"
        ),
        highlight_neg90=True,
    )
    plt.show()

# ── Side-by-side comparison: CWT vs CLSSA on the lambda/4 trace ───────────────
print("\nComparing CWT vs CLSSA on the Rayleigh-limit trace (lambda/4)...")
ix_comp = pd_idx[3]   # lambda/4 trace

# CWT
A_cwt, th_cwt, pg_cwt, fq_cwt, tn_cwt, td_cwt = pd_mod.phase_decomposition(
    bscan[ix_comp], dt,
    f_min_GHz=0.5, f_max_GHz=4.0, n_scales=60, reflect_pad=True,
)
# CLSSA
A_cl, th_cl, pg_cl, fq_cl, tn_cl, td_cl = pd_mod.clssa_phase_decomposition(
    bscan[ix_comp], dt, **CLSSA_PARAMS
)

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 10), sharey=True)
fig2.suptitle(
    f"CWT vs CLSSA  —  {pd_labels[3]}  "
    f"(aperture = {aperture[ix_comp]*1e3:.0f} ps = {aperture[ix_comp]/T_tune:.2f}*T_tune)",
    fontsize=12, fontweight="bold",
)

t0v, t1v = tn_cwt[0], tn_cwt[-1]

def _fill_row(axes_row, A, th, pg, fq, tn, td, method):
    db_clip = 40.0
    f0, f1   = fq[0], fq[-1]
    ext_sp   = [f0, f1, t1v, t0v]

    A_dB = np.clip(20*np.log10(A / (A.max()+1e-30) + 1e-30), -db_clip, 0)
    im0 = axes_row[0].imshow(A_dB.T, aspect="auto", origin="upper",
                              extent=ext_sp, cmap="jet",
                              vmin=-db_clip, vmax=0, interpolation="bilinear")
    fig2.colorbar(im0, ax=axes_row[0], fraction=0.046, pad=0.04).set_label("dB", fontsize=8)
    axes_row[0].set_title(f"{method} — Amplitude", fontsize=10)
    axes_row[0].set_xlabel("Frequency [GHz]", fontsize=9)
    axes_row[0].set_ylabel("TWT [ns]", fontsize=9)

    im1 = axes_row[1].imshow(th.T, aspect="auto", origin="upper",
                              extent=ext_sp, cmap="RdBu",
                              vmin=-180, vmax=180, interpolation="nearest")
    fig2.colorbar(im1, ax=axes_row[1], fraction=0.046, pad=0.04).set_label("deg", fontsize=8)
    axes_row[1].set_title(f"{method} — Phase", fontsize=10)
    axes_row[1].set_xlabel("Frequency [GHz]", fontsize=9)

    pg_c = np.percentile(np.abs(pg), 99)
    im2 = axes_row[2].imshow(pg.T, aspect="auto", origin="upper",
                              extent=[td[0], td[-1], t1v, t0v],
                              cmap="RdBu", vmin=-pg_c, vmax=pg_c,
                              interpolation="bilinear")
    fig2.colorbar(im2, ax=axes_row[2], fraction=0.046, pad=0.04).set_label("Amp", fontsize=8)
    axes_row[2].set_title(f"{method} — Phase Gather", fontsize=10)
    axes_row[2].set_xlabel("Phase [deg]", fontsize=9)
    axes_row[2].axvline(-90, color="magenta", lw=1.5, ls="--")
    axes_row[2].axvline( 90, color="lime",    lw=1.0, ls="--")
    axes_row[2].axvspan(-105, -75, color="magenta", alpha=0.15)
    axes_row[2].set_xticks([-180, -90, 0, 90, 180])

_fill_row(axes2[0], A_cwt, th_cwt, pg_cwt, fq_cwt, tn_cwt, td_cwt, "CWT")
_fill_row(axes2[1], A_cl,  th_cl,  pg_cl,  fq_cl,  tn_cl,  td_cl,  "CLSSA")

plt.tight_layout()
plt.show()

# ── Dominant-phase diagnostic for each selected trace ─────────────────────────
# The phase gather's inclined bands reflect carrier-frequency rotation (dtheta/dt ~ 2pi*f_c).
# plot_dominant_phase collapses the 2-D phase gather to a 1-D line:
#   theta_dom(t) = arctan2(I(t), R(t))
# A phase jump TO -90 deg at the event time = thin-bed / sub-resolution signature.

for lbl, ix, col in zip(pd_labels, pd_idx, pd_colors):
    trace = bscan[ix]
    A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg = (
        pd_mod.clssa_phase_decomposition(trace, dt, **CLSSA_PARAMS)
    )
    fig = pd_mod.plot_dominant_phase(
        A, theta_2d_deg, pg, freqs_GHz, time_ns, theta_deg,
        title=f"Dominant phase — {lbl}  |  ap={aperture[ix]*1e3:.0f} ps = {aperture[ix]/T_tune:.2f}*T_tune",
    )
    plt.show()

# Import Migrated Data from Resolution Study

In [ ]:
import os, time as _time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from gprMax.gprMax import api
from tools.outputfiles_merge import merge_files
from tools.plot_Bscan import get_output_data, mpl_plot

In [ ]:

# ── Load migrated results from Resolution_Playground ─────────────────────────
_npz_path = Path('resolution_study/migrated_results.npz')
# or use the explicit absolute path:
# _npz_path = Path(r'C:/Users/Administrator/OneDrive/Thesis/TimeLapse_Notebooks/resolution_study/migrated_results.npz')
_d = np.load(_npz_path)

# ── Axes and metadata ─────────────────────────────────────────────────────────
scenarios         = _d['scenarios']           # (6,)  ['2λ', '1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ']
methods           = _d['methods']             # (4,)  ['Kirchhoff', 'Gazdag', 'LSM (CGLS)', 'Back-prop']
separation_lambda = _d['separation_lambda']   # (6,)  [2, 1, 0.5, 0.25, 0.125, 0.0625]
x_traces          = _d['x_traces']            # (380,) m
z_img             = _d['z_img']               # (160,) m
x_s1              = _d['x_s1']               # (6,) m  left-scatterer x per scenario
x_s2              = _d['x_s2']               # (6,) m  right-scatterer x per scenario
z_scatterer       = float(_d['z_scatterer']) # scalar m  scatterer centre depth
z_top             = float(_d['z_top'])       # scalar m  depth to cylinder top

# ── Migrated images  shape (n_scenarios=6, n_z=160, n_x=380) ─────────────────
mig_kirchhoff = _d['kirchhoff']            # (6, 160, 380)
mig_gazdag    = _d['gazdag']
mig_backprop  = _d['backprop']
lsm_2lambda   = _d['lsm_cgls_2lambda']    # (160, 380)  2λ only

# ── Convenience dict  mig[method][scenario_label] = image (160, 380) ─────────
mig = {
    'Kirchhoff':  {lbl: mig_kirchhoff[i] for i, lbl in enumerate(scenarios)},
    'Gazdag':     {lbl: mig_gazdag[i]    for i, lbl in enumerate(scenarios)},
    'Back-prop':  {lbl: mig_backprop[i]  for i, lbl in enumerate(scenarios)},
    'LSM (CGLS)': {scenarios[0]: lsm_2lambda},
}

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'Loaded: {_npz_path.name}')
print(f'  scenarios        : {list(scenarios)}')
print(f'  separation_lambda: {list(separation_lambda)}')
print(f'  x_traces  {x_traces.shape}  {x_traces[0]:.3f}-{x_traces[-1]:.3f} m')
print(f'  z_img     {z_img.shape}   {z_img[0]:.3f}-{z_img[-1]:.3f} m')
print(f'  z_scatterer = {z_scatterer:.4f} m (centre),  z_top = {z_top:.4f} m (cylinder top)')
print()
print('  Array          shape            notes')
for _n, _a in [('kirchhoff', mig_kirchhoff), ('gazdag', mig_gazdag), ('backprop', mig_backprop)]:
    _nan = int(np.isnan(_a).all(axis=(1, 2)).sum())
    _note = f'  ({_nan} absent -> NaN)' if _nan else '  (scenario, z, x)'
    print(f'  {_n:<15} {str(_a.shape):<16}{_note}')
print(f'  lsm_2lambda     {str(lsm_2lambda.shape):<16}  (z, x)  2λ only')
print()
print("Access:  mig['Kirchhoff']['2\u03bb']  or  mig_kirchhoff[0]  (index 0 = 2\u03bb)")


# Plotting

In [ ]:
test_image = mig['Gazdag']['¼λ']

plt.imshow(test_image)

In [ ]:
# =============================================================================
# 2D Spatial Phase Map via the Riesz Transform (Monogenic Signal)
#
# The Riesz transform is the isotropic 2-D generalisation of the 1-D Hilbert
# transform.  In the Fourier domain:
#
#   R̂₁(kx, kz) = -i · kx / |k| · Ŵ(kx, kz)   (x-direction component)
#   R̂₂(kx, kz) = -i · kz / |k| · Ŵ(kx, kz)   (z-direction component)
#
# The monogenic signal  M = W + i(R₁, R₂)  gives three local attributes:
#
#   A(x,z)  = sqrt(W² + R₁² + R₂²)             local amplitude (envelope)
#   θ(x,z)  = arctan2(sqrt(R₁² + R₂²), W)      local phase       [0, π]
#   φ(x,z)  = arctan2(R₂, R₁)                  local orientation [-π, π]
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

# ── Select image ──────────────────────────────────────────────────────────────
W = test_image.astype(float)          # (n_z=160, n_x=380), W[iz, ix]
n_z, n_x = W.shape

# ── Riesz transform via 2-D FFT ───────────────────────────────────────────────
kx = np.fft.fftfreq(n_x)             # normalised wavenumbers (cycles/sample)
kz = np.fft.fftfreq(n_z)
KX, KZ = np.meshgrid(kx, kz)         # (n_z, n_x)
K_mag = np.sqrt(KX**2 + KZ**2)
K_mag[0, 0] = 1.0                    # avoid division by zero at DC; DC → 0 anyway

W_hat  = np.fft.fft2(W)
R1_hat = (-1j * KX / K_mag) * W_hat  # x-component of Riesz field
R2_hat = (-1j * KZ / K_mag) * W_hat  # z-component of Riesz field
R1 = np.real(np.fft.ifft2(R1_hat))
R2 = np.real(np.fft.ifft2(R2_hat))

# ── Monogenic attributes ──────────────────────────────────────────────────────
riesz_mag = np.sqrt(R1**2 + R2**2)         # |H{W}| in 2-D  (scalar field)
A_mono    = np.sqrt(W**2 + riesz_mag**2)   # local amplitude (monogenic envelope)
theta_map = np.rad2deg(np.arctan2(riesz_mag, W))   # local phase [0°, 180°]
orient_map= np.rad2deg(np.arctan2(R2, R1))          # local orientation [-180°, 180°]

# ── 6-panel figure ────────────────────────────────────────────────────────────
ext = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle(
    "2-D Riesz Transform — Monogenic Signal of Back-prop (2λ)",
    fontsize=13, fontweight="bold",
)

def _panel(ax, data, cmap, vmin, vmax, title, cbar_label):
    im = ax.imshow(data, aspect="auto", origin="upper",
                   extent=ext, cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation="bilinear")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(cbar_label, fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_ylabel("Depth [m]", fontsize=9)
    # scatterer markers (2λ scenario, index 0)
    ax.plot(x_s1[2], z_scatterer, 'g*', ms=9, zorder=5, label='s1')
    ax.plot(x_s2[2], z_scatterer, 'g^', ms=9, zorder=5, label='s2')
    # Zoom to region around the two scatterers
    x_center = 0.5 * (x_s1[0] + x_s2[0])
    x_half = max(0.1, (x_s2[0] - x_s1[0]) * 2.0)
    z_half = 0.06
    ax.set_xlim(x_center - x_half, x_center + x_half)
    ax.set_ylim(z_scatterer + z_half, z_scatterer - z_half)
    ax.legend(fontsize=8)
    return im

clip = np.percentile(np.abs(W), 100)

_panel(axes[0, 0], W,          "RdBu",    -clip,  clip,  "W(x,z) — migrated image",            "Amplitude")
_panel(axes[0, 1], R1,         "RdBu",    -clip,  clip,  "R₁{W} — Riesz (x-direction)",   "Amplitude")
_panel(axes[0, 2], R2,         "RdBu",    -clip,  clip,  "R₂{W} — Riesz (z-direction)",   "Amplitude")
_panel(axes[1, 0], A_mono,     "viridis",  0, A_mono.max()*0.85, "A(x,z) — local amplitude",  "Amplitude")
_panel(axes[1, 1], theta_map,  "hsv",      0,  180,     "θ(x,z) — local phase [deg]",     "deg")
_panel(axes[1, 2], orient_map, "twilight",-180, 180,    "φ(x,z) — local orientation [deg]","deg")

plt.tight_layout()
plt.show()

print(f"Phase range : {theta_map.min():.1f}° – {theta_map.max():.1f}°  (by definition [0°, 180°])")
print(f"Envelope max: {A_mono.max():.4g}")


In [ ]:
# =============================================================================
# Phase Gradient Field  ∇θ(x,z) = (∂θ/∂x, ∂θ/∂z)
#
# np.gradient uses second-order central differences in the interior and
# one-sided differences at the edges.  Physical spacing from x_traces and
# z_img is passed directly, so units are degrees / metre.
#
# High |∇θ| marks spatial boundaries where the local wave character changes
# sharply (reflector edges, scatterer fringes, media interfaces).
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

# ── Gradient (deg/m) ──────────────────────────────────────────────────────────
# theta_map shape is (n_z, n_x), so np.gradient returns [∂/∂z, ∂/∂x]
dtheta_dz, dtheta_dx = np.gradient(theta_map, z_img, x_traces)

grad_mag = np.sqrt(dtheta_dx**2 + dtheta_dz**2)          # |∇θ|  [deg/m]
grad_dir = np.rad2deg(np.arctan2(dtheta_dz, dtheta_dx))  # direction [-180°, 180°]

# ── Quiver subsample (direction arrows, normalised to unit length) ─────────────
step_x, step_z = 12, 6
qs_x = np.arange(0, n_x, step_x)
qs_z = np.arange(0, n_z, step_z)
QX   = x_traces[qs_x][np.newaxis, :] * np.ones((len(qs_z), 1))
QZ   = z_img[qs_z][:, np.newaxis]    * np.ones((1, len(qs_x)))
QU   = dtheta_dx[np.ix_(qs_z, qs_x)]
QV   = dtheta_dz[np.ix_(qs_z, qs_x)]
Q_norm = np.sqrt(QU**2 + QV**2) + 1e-30
QU_n, QV_n = QU / Q_norm, QV / Q_norm

# ── 5-panel figure ────────────────────────────────────────────────────────────
ext = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

# zoom window around the two scatterers (2λ scenario)
x_c    = 0.5 * (x_s1[2] + x_s2[2])
x_half = max(0.12, (x_s2[2] - x_s1[2]) * 2.5)
z_half = 0.07

def _zoom(ax):
    ax.set_xlim(x_c - x_half, x_c + x_half)
    ax.set_ylim(z_scatterer + z_half, z_scatterer - z_half)
    ax.plot(x_s1[2], z_scatterer, 'g*', ms=9, zorder=5)
    ax.plot(x_s2[2], z_scatterer, 'g^', ms=9, zorder=5)

fig, axes = plt.subplots(1, 5, figsize=(24, 5))
fig.suptitle("Phase Gradient Field  ∇θ(x,z)  [Back-prop, 2λ]",
             fontsize=13, fontweight="bold")

clip_g = np.percentile(grad_mag, 100)   # saturate top 3% for contrast

def _panel(ax, data, cmap, vmin, vmax, title, cbar_label):
    im = ax.imshow(data, aspect="auto", origin="upper",
                   extent=ext, cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation="bilinear")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(cbar_label, fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_ylabel("Depth [m]", fontsize=9)
    _zoom(ax)

_panel(axes[0], theta_map,  "hsv",    0,       180,      "θ(x,z) — local phase",       "deg")
_panel(axes[1], dtheta_dx,  "RdBu",  -clip_g,  clip_g,   "∂θ/∂x  (horizontal)",        "deg/m")
_panel(axes[2], dtheta_dz,  "RdBu",  -clip_g,  clip_g,   "∂θ/∂z  (depth)",             "deg/m")
_panel(axes[3], grad_mag,   "hot_r",  0,        clip_g,   "|∇θ|  — gradient magnitude", "deg/m")

# Panel 5: direction arrows over magnitude background
ax5 = axes[4]
im5 = ax5.imshow(grad_mag, aspect="auto", origin="upper",
                 extent=ext, cmap="Greys_r", vmin=0, vmax=clip_g,
                 interpolation="bilinear")
fig.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04).set_label("|∇θ| [deg/m]", fontsize=8)
# imshow y-axis: larger z is lower in the image, so flip QV_n for quiver
ax5.quiver(QX, QZ, QU_n, -QV_n,
           color="cyan", scale=28, width=0.003, headwidth=3, headlength=4, alpha=0.75)
ax5.set_title("∇θ direction (arrows) + |∇θ| (background)", fontsize=10)
ax5.set_xlabel("Position [m]", fontsize=9)
ax5.set_ylabel("Depth [m]", fontsize=9)
_zoom(ax5)

plt.tight_layout()
plt.show()

print(f"|∇θ| range : {grad_mag.min():.2f} – {grad_mag.max():.2f} deg/m")
print(f"∂θ/∂x range: {dtheta_dx.min():.2f} – {dtheta_dx.max():.2f} deg/m")
print(f"∂θ/∂z range: {dtheta_dz.min():.2f} – {dtheta_dz.max():.2f} deg/m")


In [ ]:
# =============================================================================
# Curl of the Phase Gradient  ∇ x ∇θ  --  Topological Charges (Phase Vortices)
#
# For a smooth scalar field  ∇ x ∇θ = 0  everywhere (Schwarz's theorem).
# At a phase singularity (amplitude node where θ is undefined), the discrete
# path integral around the singularity returns a non-zero integer multiple of
# 2pi -- the topological charge  q in {..., -1, 0, +1, ...}.
#
# Two estimators:
#   (A) Numerical curl  d(dθ/dz)/dx - d(dθ/dx)/dz   [continuous, noisy]
#   (B) Discrete winding number via 4-corner loop integral   [integer-valued]
#
# Phase for (B): phi = arctan2(R2, W) in [-pi, pi]
#   R2 is the z-component Riesz transform (= 1-D Hilbert along z).
#   Z = W + i*R2 is the analytic image in depth; its phase is a genuine
#   2pi-periodic field that can carry integer topological charges.
#   A_Z = |Z| -> 0 at singularities; charge != 0 exactly there.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# -- A: Numerical curl from theta_map [0, 180 deg] ----------------------------
d_dzdx   = np.gradient(dtheta_dz, x_traces, axis=1)   # d(dθ/dz)/dx  [deg/m2]
d_dxdz   = np.gradient(dtheta_dx, z_img,    axis=0)   # d(dθ/dx)/dz  [deg/m2]
curl_num  = d_dzdx - d_dxdz

# -- B: Signed analytic phase and discrete winding number ---------------------
# phi in [-pi, pi]; A_Z = sqrt(W^2 + R2^2) vanishes at singularities.
phi = np.arctan2(R2, W)
A_Z = np.sqrt(W**2 + R2**2)

def _wrap_pi(x):
    return (x + np.pi) % (2.0 * np.pi) - np.pi

# 4-edge counterclockwise loop integral for every unit cell -> shape (n_z-1, n_x-1)
d1 = _wrap_pi(phi[:-1, 1:]  - phi[:-1, :-1])   # right  (top edge)
d2 = _wrap_pi(phi[1:,  1:]  - phi[:-1,  1:])   # down   (right edge)
d3 = _wrap_pi(phi[1:,  :-1] - phi[1:,   1:])   # left   (bottom edge)
d4 = _wrap_pi(phi[:-1, :-1] - phi[1:,  :-1])   # up     (left edge)
charge = np.round((d1 + d2 + d3 + d4) / (2.0 * np.pi)).astype(int)

# Cell-centre coordinates
xc = 0.5 * (x_traces[:-1] + x_traces[1:])
zc = 0.5 * (z_img[:-1]    + z_img[1:])

pos_rows, pos_cols = np.where(charge ==  1)
neg_rows, neg_cols = np.where(charge == -1)
xc_pos, zc_pos = xc[pos_cols], zc[pos_rows]
xc_neg, zc_neg = xc[neg_cols], zc[neg_rows]

# -- 4-panel figure -----------------------------------------------------------
ext   = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]
ext_c = [xc[0],       xc[-1],       zc[-1],    zc[0]]

x_c_sc    = 0.5 * (x_s1[2] + x_s2[2])
x_half_sc = max(0.12, (x_s2[2] - x_s1[2]) * 2.5)
z_half_sc = 0.07

def _zoom(ax):
    ax.set_xlim(x_c_sc - x_half_sc, x_c_sc + x_half_sc)
    ax.set_ylim(z_scatterer + z_half_sc, z_scatterer - z_half_sc)
    ax.plot(x_s1[2], z_scatterer, 'g*', ms=9, zorder=5)
    ax.plot(x_s2[2], z_scatterer, 'g^', ms=9, zorder=5)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle(
    "∇ x ∇θ  --  Topological Charges (Phase Vortices)  [2λ scenario]",
    fontsize=13, fontweight="bold",
)

# P1: signed analytic phase
im0 = axes[0].imshow(phi, aspect="auto", origin="upper",
                     extent=ext, cmap="hsv", vmin=-np.pi, vmax=np.pi,
                     interpolation="bilinear")
cb0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04,
                   ticks=[-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
cb0.set_ticklabels(['-pi', '-pi/2', '0', 'pi/2', 'pi'])
cb0.set_label("rad", fontsize=8)
axes[0].set_title("phi = arctan2(R2, W)  signed phase", fontsize=10)
_zoom(axes[0])

# P2: numerical curl (should be approx 0 for smooth field)
curl_clip = np.percentile(np.abs(curl_num), 100)
im1 = axes[1].imshow(curl_num, aspect="auto", origin="upper",
                     extent=ext, cmap="RdBu",
                     norm=TwoSlopeNorm(vmin=-curl_clip, vcenter=0, vmax=curl_clip),
                     interpolation="bilinear")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04).set_label("deg/m2", fontsize=8)
axes[1].set_title("∇x∇θ  numerical  (= 0 for smooth θ)", fontsize=10)
_zoom(axes[1])

# P3: discrete topological charge (integer, nearest-neighbour display)
im2 = axes[2].imshow(charge, aspect="auto", origin="upper",
                     extent=ext_c, cmap="RdBu", vmin=-1, vmax=1,
                     interpolation="nearest")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04,
             ticks=[-1, 0, 1]).set_label("q", fontsize=8)
axes[2].set_title(
    f"Winding number q  (+{len(xc_pos)} pos / {len(xc_neg)} neg)", fontsize=10)
_zoom(axes[2])

# P4: analytic amplitude A_Z = |W + iR2| with singularity positions marked
im3 = axes[3].imshow(A_Z, aspect="auto", origin="upper",
                     extent=ext, cmap="viridis", vmin=0,
                     vmax=np.percentile(A_Z, 100), interpolation="bilinear")
fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04).set_label("Amplitude", fontsize=8)
if len(xc_pos):
    axes[3].plot(xc_pos, zc_pos, 'r+', ms=9, mew=2.0,
                 label=f'+1  ({len(xc_pos)})', zorder=6)
if len(xc_neg):
    axes[3].plot(xc_neg, zc_neg, 'b+', ms=9, mew=2.0,
                 label=f'-1  ({len(xc_neg)})', zorder=6)
axes[3].legend(fontsize=8, loc='upper right', title='vortex')
axes[3].set_title("A_Z = |W+iR2|  envelope + singularity positions", fontsize=10)
_zoom(axes[3])

for ax in axes:
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_ylabel("Depth [m]", fontsize=9)

plt.tight_layout()
plt.show()

print(f"Positive vortices (+1) : {len(xc_pos)}")
print(f"Negative vortices (-1) : {len(xc_neg)}")
print(f"Net topological charge : {charge.sum()}  (expect 0 -- Poincare-Hopf index theorem)")
print(f"Numerical curl range   : {curl_num.min():.2f} -- {curl_num.max():.2f}  deg/m2")


In [ ]:
# =============================================================================
# Scatterer localization -- corrected approach
#
# Why A_mono failed: the x-direction Riesz component R1 is large in the
# lateral TRANSITION ZONE between two focal spots (it behaves like d/dx of W),
# which sums constructively with W at the midpoint and creates a false peak.
#
# Fix: use A_Z = sqrt(W^2 + R2^2)  (depth-direction analytic signal only).
# R2 is the z-direction Hilbert transform; it is zero at the crest of each
# focal blob and peaks at the flanks -- so A_Z is laterally discriminating.
#
# Detection pipeline:
#   1. Extract a horizontal slab of A_Z around z = z_scatterer
#   2. Collapse to a 1-D profile by taking the depth-maximum per column
#   3. scipy.signal.find_peaks on that profile
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# -- Physical grid spacing -----------------------------------------------------
dx_m = float(np.mean(np.diff(x_traces)))
dz_m = float(np.mean(np.diff(z_img)))

# -- A_Z = depth-direction analytic amplitude ----------------------------------
# A_Z already computed in the previous cells as:  A_Z = np.sqrt(W**2 + R2**2)
# (R2 is the z-component Riesz transform = 1-D Hilbert along z)

# -- 1-D profile: depth-max of A_Z in a window around z_scatterer -------------
half_win_m = 0.035                          # +/- 35 mm search window
iz_lo = np.searchsorted(z_img, z_scatterer - half_win_m)
iz_hi = np.searchsorted(z_img, z_scatterer + half_win_m) + 1
iz_hi = min(iz_hi, len(z_img))

A_window      = A_Z[iz_lo:iz_hi, :]         # shape (n_win, n_x)
iz_local_max  = np.argmax(A_window, axis=0) # depth index of peak per column
A_profile     = A_window[iz_local_max, np.arange(n_x)]   # 1-D profile

# -- Find peaks on the 1-D profile --------------------------------------------
sep_lambda   = float(x_s2[2] - x_s1[2])     # true scatterer separation [m]
min_dist_m   = max(0.01, 0.4 * sep_lambda)   # at least 40 % of expected sep
min_dist_px  = max(3, int(min_dist_m / dx_m))

peaks_px, _ = find_peaks(A_profile,
                          height   = 0.25 * A_profile.max(),
                          distance = min_dist_px)

x_det = x_traces[peaks_px]
z_det = z_img[iz_lo + iz_local_max[peaks_px]]

# -- Report -------------------------------------------------------------------
print(f"True:       x_s1={x_s1[2]:.4f} m,  x_s2={x_s2[2]:.4f} m,  "
      f"sep={sep_lambda*1e3:.1f} mm,  z={z_scatterer:.4f} m")
print(f"Detected:   {len(x_det)} peak(s)")
for xi, zi in zip(x_det, z_det):
    err_x = min(abs(xi - x_s1[2]), abs(xi - x_s2[2])) * 1e3
    print(f"  x={xi:.4f} m  z={zi:.4f} m   err={err_x:.1f} mm")

# -- 4-panel figure -----------------------------------------------------------
ext   = [x_traces[2], x_traces[-1], z_img[-1], z_img[2]]
ext_c = [xc[2],       xc[-1],       zc[-1],    zc[2]]

x_c_sc    = 0.5 * (x_s1[2] + x_s2[2])
x_half_sc = max(0.12, sep_lambda * 2.5)
z_half_sc = 0.07

def _zoom(ax):
    ax.set_xlim(x_c_sc - x_half_sc, x_c_sc + x_half_sc)
    ax.set_ylim(z_scatterer + z_half_sc, z_scatterer - z_half_sc)

def _marks(ax):
    ax.plot(x_s1[2], z_scatterer, 'g*', ms=11, zorder=7, label='true')
    ax.plot(x_s2[2], z_scatterer, 'g*', ms=11, zorder=7)
    if len(x_det):
        ax.plot(x_det, z_det, 'ro', ms=10, mew=2, mfc='none',
                zorder=8, label='detected')
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle("Scatterer detection: A_Z horizontal profile  [2lambda scenario]",
             fontsize=12, fontweight="bold")

# P1 -- migrated image
clip_w = np.percentile(np.abs(W), 100)
im0 = axes[0].imshow(W, aspect="auto", origin="upper", extent=ext,
                     cmap="RdBu", vmin=-clip_w, vmax=clip_w,
                     interpolation="bilinear")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].set_title("W(x,z) -- migrated image", fontsize=10)
_marks(axes[0]);  _zoom(axes[0])

# P2 -- A_Z map + depth-search window
im1 = axes[1].imshow(A_Z, aspect="auto", origin="upper", extent=ext,
                     cmap="viridis", vmin=0, vmax=np.percentile(A_Z, 100),
                     interpolation="bilinear")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04).set_label("Amplitude", fontsize=8)
axes[1].axhline(z_img[iz_lo], color='cyan', lw=1, ls='--', alpha=0.8)
axes[1].axhline(z_img[iz_hi-1], color='cyan', lw=1, ls='--', alpha=0.8,
                label='search window')
axes[1].legend(fontsize=7)
axes[1].set_title("A_Z = |W+iR2|  (depth analytic amplitude)", fontsize=10)
_marks(axes[1]);  _zoom(axes[1])

# P3 -- 1-D profile with detected peaks
ax3 = axes[2]
x_zoom_mask = (x_traces >= x_c_sc - x_half_sc) & (x_traces <= x_c_sc + x_half_sc)
ax3.plot(x_traces[x_zoom_mask], A_profile[x_zoom_mask], 'k-', lw=1.5,
         label='A_Z profile (depth max)')
if len(x_det):
    ax3.plot(x_det, A_profile[peaks_px], 'ro', ms=9, zorder=5, label='detected')
ax3.axvline(x_s1[2], color='g', ls='--', lw=1.2, label='true s1')
ax3.axvline(x_s2[2], color='g', ls=':',  lw=1.2, label='true s2')
ax3.axhline(0.25 * A_profile.max(), color='gray', ls=':', lw=1.0, label='threshold 25%')
ax3.set_xlabel("Position [m]", fontsize=9)
ax3.set_ylabel("A_Z (depth-max)", fontsize=9)
ax3.set_title("1-D amplitude profile at z_scatterer depth", fontsize=10)
ax3.set_xlim(x_c_sc - x_half_sc, x_c_sc + x_half_sc)
ax3.legend(fontsize=7)
ax3.grid(True, alpha=0.3)

# P4 -- winding number context
im3 = axes[3].imshow(charge, aspect="auto", origin="upper", extent=ext_c,
                     cmap="RdBu", vmin=-1, vmax=1, interpolation="nearest")
fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04, ticks=[-1, 0, 1]).set_label("q", fontsize=8)
axes[3].set_title(f"Winding number q  (vortices flank scatterers,not at them)",
                  fontsize=9)
_marks(axes[3]);  _zoom(axes[3])

for ax in [axes[0], axes[1], axes[3]]:
    ax.set_xlabel("Position [m]", fontsize=9)
    ax.set_ylabel("Depth [m]", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Why the spectra look identical below lambda/2 -- and what carries
# genuine sub-wavelength information
#
# The interference fringe null is at kx* = 1/(2*dx).
# For dx < lambda/2:  kx* > 1/lambda  -->  outside the PSF spectral support.
# Migration zeros the spectrum there, so every sub-lambda/2 scenario produces
# the same spectral shape.  This IS the Rayleigh limit expressed in kx.
#
# Panel A -- spectra within PSF bandwidth (confirms the diagnosis)
# Panel B -- profile FWHM vs separation (broadens below Rayleigh, amplitude-based)
# Panel C -- synthetic timelapse: phase of the DIFFERENCE image for a single
#            scatterer shifted by fractions of lambda  -->  the correct
#            sub-wavelength tool; phase varies monotonically for ANY shift
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

v_ice   = 0.168
f_c     = 1.5
lam_ice = v_ice / f_c
dx_m    = float(np.mean(np.diff(x_traces)))
dz_m    = float(np.mean(np.diff(z_img)))
iz_scat = np.argmin(np.abs(z_img - z_scatterer))
sep_m   = x_s2 - x_s1
sep_lam = sep_m / lam_ice
k_bw    = 2.0 * f_c * 1e9 / (v_ice * 1e9)   # PSF bandwidth [cycles/m]
n_pad   = 8

colors = plt.cm.plasma(np.linspace(0.05, 0.95, len(scenarios)))

def spectrum(row):
    n  = len(row)
    kx = np.fft.rfftfreq(n * n_pad, d=dx_m)
    sp = np.abs(np.fft.rfft(row, n=n * n_pad))
    return kx, sp / (sp.max() + 1e-30)

def fwhm_lam(profile):
    p = np.abs(profile)
    half = p.max() / 2.0
    above = p > half
    d = np.diff(above.astype(int))
    rise = np.where(d ==  1)[0]
    fall = np.where(d == -1)[0]
    if len(rise) and len(fall) and fall[-1] > rise[0]:
        return (fall[-1] - rise[0]) * dx_m / lam_ice
    return np.nan

fwhm_kir = [fwhm_lam(mig["Kirchhoff"][lbl][iz_scat, :]) for lbl in scenarios]
fwhm_gaz = [fwhm_lam(mig["Gazdag"]  [lbl][iz_scat, :]) for lbl in scenarios]

# -- Panel C: synthetic timelapse (shift W_ref row-wise sub-pixel) ------------
W_ref    = mig["Kirchhoff"][scenarios[0]].astype(float)
n_z_r, n_x_r = W_ref.shape
kx_r = np.fft.fftfreq(n_x_r)
kz_r = np.fft.fftfreq(n_z_r)
KX_r, KZ_r = np.meshgrid(kx_r, kz_r)
K_r = np.sqrt(KX_r**2 + KZ_r**2);  K_r[0, 0] = 1.0
W_hat_r  = np.fft.fft2(W_ref)
R2_hat_r = (-1j * KZ_r / K_r) * W_hat_r

kx_1d = np.fft.fftfreq(n_x_r, d=dx_m)

shifts_lam = np.array([1.0, 0.5, 0.25, 0.125, 1.0/16.0, 1.0/32.0])
phase_diff_peak = []

for dxs in shifts_lam * lam_ice:
    W_sh = np.zeros_like(W_ref)
    for iz in range(n_z_r):
        rh = np.fft.fft(W_ref[iz, :])
        rh *= np.exp(-1j * 2 * np.pi * kx_1d * dxs)
        W_sh[iz, :] = np.real(np.fft.ifft(rh))
    diff = W_sh - W_ref

    D_hat   = np.fft.fft2(diff)
    R2D_hat = (-1j * KZ_r / K_r) * D_hat
    R2_diff = np.real(np.fft.ifft2(R2D_hat))
    phi_d   = np.arctan2(R2_diff, diff)
    A_d     = np.sqrt(diff**2 + R2_diff**2)

    ix_c  = np.argmin(np.abs(x_traces - x_s1[0]))
    hw    = int(0.15 / dx_m)
    iz_lo = max(0,     iz_scat - int(0.04 / dz_m))
    iz_hi = min(n_z_r, iz_scat + int(0.04 / dz_m))
    sub_A  = A_d [iz_lo:iz_hi, max(0, ix_c-hw):ix_c+hw]
    sub_ph = phi_d[iz_lo:iz_hi, max(0, ix_c-hw):ix_c+hw]
    pk = np.unravel_index(np.argmax(sub_A), sub_A.shape)
    phase_diff_peak.append(np.rad2deg(sub_ph[pk]))

# -- Figure -------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
fig.suptitle(
    "Sub-wavelength information in migrated images: what works and what does not",
    fontsize=12, fontweight="bold",
)

# A
ax = axes[0]
for i, lbl in enumerate(scenarios):
    kx_sp, sp = spectrum(mig["Kirchhoff"][lbl][iz_scat, :])
    mask = kx_sp <= k_bw * 1.3
    ax.plot(kx_sp[mask], sp[mask], color=colors[i], lw=1.3,
            label=f"{lbl}  {sep_m[i]*1e3:.0f} mm")
ax.axvline(k_bw, color="r", lw=1.5, ls="--", label=f"PSF bw  {k_bw:.1f} 1/m")
ax.axvspan(0, k_bw, alpha=0.07, color="steelblue")
for i in range(len(scenarios)):
    kn = 1.0 / (2.0 * sep_m[i])
    if kn < k_bw * 1.3:
        ax.axvline(kn, color=colors[i], lw=0.8, ls=":", alpha=0.7)
ax.set_xlabel("kx [cycles/m]", fontsize=10)
ax.set_ylabel("Normalised |W_hat|", fontsize=10)
ax.set_title("A.  Spectra identical below lambda/2\n"
             "(dotted = predicted fringe null; outside PSF bw for dx < lambda/2)",
             fontsize=9)
ax.legend(fontsize=7, loc="upper right")
ax.grid(alpha=0.3)

# B
ax2 = axes[1]
ax2.plot(sep_lam, fwhm_kir, "o-", color="steelblue",   lw=1.5, ms=7, label="Kirchhoff")
ax2.plot(sep_lam, fwhm_gaz, "s-", color="darkorange",  lw=1.5, ms=7, label="Gazdag")
ax2.axvline(0.5, color="r",    ls="--", lw=1.2, label="lambda/2 Rayleigh")
ax2.axhline(1.0, color="gray", ls=":",  lw=0.9, label="single-scatterer PSF")
ax2.set_xlabel("Separation / lambda  (log scale)", fontsize=10)
ax2.set_ylabel("FWHM / lambda", fontsize=10)
ax2.set_xscale("log")
ax2.set_title("B.  Profile broadens below Rayleigh\n"
              "(amplitude-based; can detect merged vs single, but not split peaks)",
              fontsize=9)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# C
ax3 = axes[2]
ax3.semilogx(shifts_lam, phase_diff_peak, "ko-", lw=1.8, ms=8)
for sh, ph in zip(shifts_lam, phase_diff_peak):
    ax3.annotate(f"{sh:.3f}lam", (sh, ph),
                 textcoords="offset points", xytext=(5, 4), fontsize=8)
ax3.axhline(0, color="gray", ls=":", lw=0.8)
ax3.axvline(0.5, color="r", ls="--", lw=1.2, label="lambda/2")
ax3.set_xlabel("Lateral shift / lambda  (log scale)", fontsize=10)
ax3.set_ylabel("Phase of difference image at peak [deg]", fontsize=10)
ax3.set_title("C.  Timelapse: phase of (shifted - reference) image\n"
              "monotonically encodes shift for ANY magnitude, including sub-lambda",
              fontsize=9)
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PSF bandwidth  k_bw = {k_bw:.2f} 1/m   Rayleigh limit = {lam_ice/2*1e3:.1f} mm")
print()
print(f"  {'Scenario':<10}  {'dx [mm]':>9}  {'kx_null':>10}  {'in PSF bw':>10}  {'FWHM/lam':>9}")
for i, lbl in enumerate(scenarios):
    kn = 1.0 / (2.0 * sep_m[i])
    ok = kn <= k_bw
    print(f"  {str(lbl):<10}  {sep_m[i]*1e3:>9.2f}  {kn:>10.2f}  {'YES' if ok else 'no':>10}  {fwhm_kir[i]:>9.3f}")
print()
print("Timelapse phase:")
for sh, ph in zip(shifts_lam, phase_diff_peak):
    print(f"  shift = {sh:.4f} lam  ({sh*lam_ice*1e3:6.2f} mm)  ->  phase = {ph:.2f} deg")


In [ ]:
import pylops
import time as _time


def gazdag_migration(data, x, t, z, vel):
    """
    Gazdag (1978) phase-shift migration for zero-offset post-stack data.

    data : (n_t, n_x)  B-scan, time axis first, t0-shifted
    x, t : 1-D arrays [m, ns]
    z    : 1-D depth axis [m], uniformly spaced from 0
    vel  : full medium velocity [m/ns]; v_mig = vel/2 used internally

    Returns image (n_z, n_x), peak-normalised to +-1.
    """
    import numpy as np
    n_t, n_x = data.shape
    dt    = float(t[1] - t[0])
    dx    = float(x[1] - x[0])
    dz    = float(z[1] - z[0])
    v_mig = vel / 2.0

    # Spatial cosine taper (5%) + 100% zero-pad each side
    n_xtap  = max(3, int(0.05 * n_x))
    pad     = n_x
    sp_tap  = pylops.utils.tapers.taper2d(n_t, n_x, n_xtap)   # (n_x, n_t)
    data_sp = np.pad(
        data.T * sp_tap,            # (n_x, n_t)
        ((pad, pad), (0, 0)),
        mode='constant'
    )                               # (nx_pad, n_t)
    nx_pad = n_x + 2 * pad

    # f-kx evanescent filter
    D_fk       = np.fft.fft(np.fft.rfft(data_sp, axis=1), axis=0)
    freq_arr   = np.fft.rfftfreq(n_t, dt)
    kx_arr     = np.fft.fftfreq(nx_pad, dx)
    evanescent = np.abs(kx_arr[:, None]) > np.abs(freq_arr[None, :]) / v_mig
    D_fk[evanescent] = 0.0
    data_sp = np.real(np.fft.irfft(np.fft.ifft(D_fk, axis=0), n=n_t, axis=1))

    print(f'  nx={n_x} -> nx_pad={nx_pad},  evanescent: '
          f'{evanescent.sum()}/{evanescent.size} ({100*evanescent.mean():.1f}%)')

    # PhaseShift depth loop: downward continuation + t=0 imaging condition
    freq  = np.fft.rfftfreq(n_t, dt)
    kx    = np.fft.fftshift(np.fft.fftfreq(nx_pad, dx))
    Pop   = pylops.waveeqprocessing.PhaseShift(v_mig, dz, n_t, freq, kx)
    field = data_sp.T.ravel()

    image = np.zeros((len(z), n_x))
    for iz in range(len(z)):
        image[iz] = np.real(field.reshape(n_t, nx_pad)[0, pad:-pad])
        if iz < len(z) - 1:
            field = Pop.H * field

    peak = np.max(np.abs(image))
    return image / (peak + 1e-30)


In [ ]:
# =============================================================================
# Synthetic timelapse GPR + Gazdag migration of the difference B-scan
#
# A single point scatterer in ice moves laterally by fractions of lambda.
# Key results:
#   1. Migrated difference image = DIPOLE whose amplitude scales linearly
#      with the shift -> detectable for any sub-lambda movement (SNR-limited).
#   2. The dipole zero-crossing is EXACTLY at the original scatterer position,
#      regardless of shift magnitude.  This becomes a topological charge
#      (winding number +/-1) of the analytic difference image.
#   3. Connects to earlier topology analysis: for TIMELAPSE imaging the
#      phase singularity of the difference image locates the scatterer.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

v_ice  = 0.168        # m/ns
f_c    = 1.5          # GHz
lam    = v_ice / f_c  # wavelength in ice [m]

# -- Acquisition --------------------------------------------------------------
dx_b   = 0.005
n_x_b  = 160
x_b    = np.arange(n_x_b) * dx_b       # 0 -- 0.795 m profile
x_sc0  = x_b.mean()

dt_b   = 0.004
n_t_b  = 1300                           # t_max = 5.196 ns > TWT(0.30 m) = 3.57 ns
t_b    = np.arange(n_t_b) * dt_b

z_sc   = 0.30
z_mig  = np.arange(0, 0.42, dx_b)      # 84 depth steps

# -- Vectorised point-scatterer B-scan ----------------------------------------
def point_bscan(x_sc, z_sc, x_rec, t_arr, v, fc):
    r   = np.sqrt((x_rec - x_sc)**2 + z_sc**2)
    twt = 2.0 * r / v
    amp = 1.0 / np.sqrt(r)
    u   = np.pi * fc * (t_arr[:, None] - twt[None, :])
    return amp[None, :] * (1.0 - 2*u**2) * np.exp(-u**2)

# -- Z-direction Riesz transform ----------------------------------------------
def riesz_z(W):
    nz, nx = W.shape
    KX, KZ = np.meshgrid(np.fft.fftfreq(nx), np.fft.fftfreq(nz))
    K = np.sqrt(KX**2 + KZ**2);  K[0, 0] = 1.0
    return np.real(np.fft.ifft2((-1j * KZ / K) * np.fft.fft2(W)))

# -- 4-corner winding number --------------------------------------------------
def winding_number(phi):
    def w(x): return (x + np.pi) % (2*np.pi) - np.pi
    return np.round(
        (w(phi[:-1, 1:] - phi[:-1, :-1]) + w(phi[1:,  1:] - phi[:-1, 1:]) +
         w(phi[1:, :-1] - phi[1:,   1:]) + w(phi[:-1, :-1]- phi[1:,  :-1]))
        / (2*np.pi)
    ).astype(int)

# -- Baseline -----------------------------------------------------------------
print("Baseline B-scan + migration...")
t0 = _time.time()
B_base     = point_bscan(x_sc0, z_sc, x_b, t_b, v_ice, f_c)
img_base   = gazdag_migration(B_base, x_b, t_b, z_mig, v_ice)
B_base_max = float(np.max(np.abs(B_base)))
print(f"  done in {_time.time()-t0:.1f} s")

# -- Timelapse loop -----------------------------------------------------------
move_fracs  = [1.0, 0.5, 0.25, 0.125, 1.0/16.0, 1.0/32.0]
# move_labels = ['1lam', 'lam/2', 'lam/4', 'lam/8', 'lam/16', 'lam/32']
move_labels = [f"lam/{int(1/f)}" if f < 1 else f"{f:.0f}lam" for f in move_fracs]
cols        = plt.cm.plasma(np.linspace(0.1, 0.9, len(move_fracs)))

results = []
for frac, lbl in zip(move_fracs, move_labels):
    dx_move = frac * lam
    print(f"\nShift {lbl} = {dx_move*1e3:.2f} mm  ({dx_move/dx_b:.2f} samples)...")
    t0 = _time.time()

    B_mon    = point_bscan(x_sc0 + dx_move, z_sc, x_b, t_b, v_ice, f_c)
    B_diff   = B_mon - B_base
    img_diff = gazdag_migration(B_diff, x_b, t_b, z_mig, v_ice)

    R2_d  = riesz_z(img_diff)
    phi_d = np.arctan2(R2_d, img_diff)
    A_d   = np.sqrt(img_diff**2 + R2_d**2)
    q_d   = winding_number(phi_d)

    data_ratio = float(np.max(np.abs(B_diff))) / B_base_max
    print(f"  done in {_time.time()-t0:.1f} s  |  data ratio = {data_ratio:.5f}")
    results.append(dict(frac=frac, label=lbl,
                        B_mon=B_mon, B_diff=B_diff,
                        img_diff=img_diff, phi_diff=phi_d,
                        A_diff=A_d, charge=q_d, data_ratio=data_ratio))

# -- Cell-centre axes for topology overlay ------------------------------------
xc = 0.5*(x_b[:-1]   + x_b[1:])
zc = 0.5*(z_mig[:-1]  + z_mig[1:])

# -- Figure 1: smallest shift detail -----------------------------------------
r       = results[-1]
ext_b   = [x_b[0], x_b[-1], t_b[-1], t_b[0]]
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
ext_c   = [xc[0],  xc[-1],  zc[-1],   zc[0]]
x_lo, x_hi = x_sc0 - 4*lam, x_sc0 + 4*lam
z_lo, z_hi = z_sc - 0.08,   z_sc + 0.08

def zoom(ax):
    ax.set_xlim(x_lo, x_hi);  ax.set_ylim(z_hi, z_lo)
    ax.plot(x_sc0, z_sc, 'g*', ms=12, zorder=7, label='baseline pos')

clip_b  = np.percentile(np.abs(B_base), 98)
clip_db = np.percentile(np.abs(r['B_diff']), 98)
clip_di = np.percentile(np.abs(r['img_diff']), 98)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"Timelapse Gazdag -- shift {r['label']}  "
    f"({r['frac']*lam*1e3:.1f} mm = {r['frac']*lam/dx_b:.1f} samples)",
    fontsize=13, fontweight='bold',
)

# Row 0: B-scans
for ax, data, clip, title in [
    (axes[0,0], B_base,     clip_b,  "Baseline B-scan"),
    (axes[0,1], r['B_mon'], clip_b,  f"Monitor B-scan  (shift {r['label']})"),
]:
    ax.imshow(data, aspect='auto', origin='upper', extent=ext_b,
              cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    ax.set_title(title, fontsize=10);  ax.set_xlabel("Position [m]");  ax.set_ylabel("TWT [ns]")

im = axes[0,2].imshow(r['B_diff'], aspect='auto', origin='upper', extent=ext_b,
                       cmap='RdBu', vmin=-clip_db, vmax=clip_db, interpolation='bilinear')
axes[0,2].set_title(f"Difference B-scan  (data ratio = {r['data_ratio']:.5f})", fontsize=10)
axes[0,2].set_xlabel("Position [m]");  axes[0,2].set_ylabel("TWT [ns]")
fig.colorbar(im, ax=axes[0,2], fraction=0.046, pad=0.04)

# Row 1: migrated
im0 = axes[1,0].imshow(img_base, aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-1, vmax=1, interpolation='bilinear')
axes[1,0].set_title("Migrated baseline (Gazdag)")
zoom(axes[1,0]);  axes[1,0].legend(fontsize=8)
fig.colorbar(im0, ax=axes[1,0], fraction=0.046, pad=0.04)

im1 = axes[1,1].imshow(r['img_diff'], aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-clip_di, vmax=clip_di, interpolation='bilinear')
axes[1,1].plot(x_sc0 + r['frac']*lam, z_sc, 'r^', ms=10, zorder=7, label='monitor pos')
axes[1,1].set_title("Migrated difference -- dipole\n+lobe right = rightward shift", fontsize=9)
zoom(axes[1,1]);  axes[1,1].legend(fontsize=8)
fig.colorbar(im1, ax=axes[1,1], fraction=0.046, pad=0.04)

ax = axes[1,2]
ax.imshow(r['A_diff'], aspect='auto', origin='upper', extent=ext_mig,
          cmap='viridis', vmin=0, vmax=np.percentile(r['A_diff'], 99), interpolation='bilinear')
pos_r, pos_c = np.where(r['charge'] ==  1)
neg_r, neg_c = np.where(r['charge'] == -1)
if len(pos_c): ax.plot(xc[pos_c], zc[pos_r], 'r+', ms=10, mew=2,
                        label=f'+1  ({len(pos_c)})', zorder=6)
if len(neg_c): ax.plot(xc[neg_c], zc[neg_r], 'b+', ms=10, mew=2,
                        label=f'-1  ({len(neg_c)})', zorder=6)
ax.set_title("Winding number q of analytic difference image\n"
             "Singularity at original scatterer position (any shift)", fontsize=9)
zoom(ax);  ax.legend(fontsize=8)

for ax in axes[1]: ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
plt.tight_layout()
plt.show()

# -- Figure 2: amplitude scaling + dipole profiles ----------------------------
iz_sc  = np.argmin(np.abs(z_mig - z_sc))
x_zoom = (x_b >= x_lo) & (x_b <= x_hi)

fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle("Sub-wavelength sensitivity of migrated difference image",
              fontsize=12, fontweight='bold')

shifts_mm = [r2['frac']*lam*1e3 for r2 in results]
ratios    = [r2['data_ratio']    for r2 in results]
lf        = np.polyfit(np.log(shifts_mm), np.log(ratios), 1)
x_fit     = np.linspace(min(shifts_mm)*0.8, max(shifts_mm)*1.2, 30)
axes2[0].loglog(shifts_mm, ratios, 'ko-', lw=1.8, ms=8)
axes2[0].loglog(x_fit, np.exp(np.polyval(lf, np.log(x_fit))), 'r--',
                label=f'slope = {lf[0]:.2f}  (1.0 = linear)')
for r2 in results:
    axes2[0].annotate(r2['label'], (r2['frac']*lam*1e3, r2['data_ratio']),
                      textcoords='offset points', xytext=(5, 3), fontsize=9)
axes2[0].axvline(lam*1e3/2, color='steelblue', ls=':', lw=1.2, label='lam/2 Rayleigh')
axes2[0].set_xlabel("Lateral shift [mm]", fontsize=10)
axes2[0].set_ylabel("max|B_diff| / max|B_base|", fontsize=10)
axes2[0].set_title("Amplitude ratio scales linearly with shift\n"
                   "=> detectable at any sub-lambda shift (SNR-limited only)", fontsize=9)
axes2[0].legend(fontsize=8);  axes2[0].grid(True, alpha=0.3, which='both')

for r2, col in zip(results, cols):
    prof = r2['img_diff'][iz_sc, :]
    pk   = np.abs(r2['img_diff']).max() + 1e-30
    axes2[1].plot(x_b[x_zoom], prof[x_zoom]/pk, color=col, lw=1.3, label=r2['label'])
axes2[1].axvline(x_sc0, color='gray', ls='--', lw=0.9, label='scatterer x_sc0')
axes2[1].axhline(0, color='k', lw=0.4)
axes2[1].set_xlabel("Position [m]", fontsize=10)
axes2[1].set_ylabel("Normalised img_diff at z_sc", fontsize=10)
axes2[1].set_title("Dipole profiles (normalised per shift)\n"
                   "Zero-crossing = scatterer position for any shift", fontsize=9)
axes2[1].legend(fontsize=8);  axes2[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nlambda_ice = {lam*1e3:.2f} mm  |  dx_b = {dx_b*1e3:.1f} mm  ({lam/dx_b:.0f} samples/lam)")
print(f"Scatterer: x={x_sc0*1e3:.0f} mm, z={z_sc*1e3:.0f} mm")
print()
print(f"  {'Shift':<8}  {'dx_mm':>8}  {'dx/lam':>8}  {'dx/samp':>8}  {'ratio':>10}  +1 q")
for r2 in results:
    q = r2['charge']
    print(f"  {r2['label']:<8}  {r2['frac']*lam*1e3:>8.2f}  {r2['frac']:>8.4f}"
          f"  {r2['frac']*lam/dx_b:>8.3f}  {r2['data_ratio']:>10.5f}  {(q==1).sum():>4}")


In [ ]:
# =============================================================================
# Resolving scatterer position and movement from the migrated difference image
#
# The migrated difference image encodes two independent quantities:
#
#   img_diff(x, z) ≈ -delta_x * d(img_base)/dx   (first-order, small delta_x)
#
# Therefore:
#   POSITION  -- zero-crossing of img_diff at z_sc  =>  x_sc0  (exact for any delta_x)
#   MOVEMENT  -- least-squares projection onto d(img_base)/dx  =>  delta_x
#
# The projection:
#   delta_x = gamma * [ -<img_diff, d_base_dx> / ||d_base_dx||^2 ]
#   gamma   = max|B_diff| / max|B_base|   (data-domain amplitude ratio)
#
# gamma is needed because gazdag_migration() normalises each output to +/-1
# independently.  The data-domain ratio restores the physical amplitude scale.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

iz_sc     = np.argmin(np.abs(z_mig - z_sc))
x_zoom    = (x_b >= x_sc0 - 5*lam) & (x_b <= x_sc0 + 5*lam)

# -- Baseline PSF gradient at scatterer depth ---------------------------------
base_prof = img_base[iz_sc, :]
d_base_dx = np.gradient(base_prof, x_b)        # d(img_base)/dx  [1/m, normalised]
d_base_dx_peak = float(np.max(np.abs(d_base_dx)))

# -- Inversion for each scenario ----------------------------------------------
true_dxs  = []
est_dxs   = []
crossings = []
cols      = plt.cm.plasma(np.linspace(0.1, 0.9, len(results)))

print(f"{'Shift':<9}  {'True dx [mm]':>13}  {'Est dx [mm]':>13}  "
      f"{'Error [mm]':>12}  {'Rel err %':>10}  {'x0 err [mm]':>12}")
print("-" * 76)

for r in results:
    diff_prof = r['img_diff'][iz_sc, :]

    # -- 1. Position: sub-pixel zero-crossing closest to x_sc0 ----------------
    sc_idx = np.where(np.diff(np.sign(diff_prof)))[0]
    if len(sc_idx):
        zc_list = []
        for sc in sc_idx:
            frac = -diff_prof[sc] / (diff_prof[sc+1] - diff_prof[sc] + 1e-30)
            zc_list.append(x_b[sc] + frac * (x_b[sc+1] - x_b[sc]))
        x_cross = min(zc_list, key=lambda c: abs(c - x_sc0))
    else:
        x_cross = np.nan
    crossings.append(x_cross)

    # -- 2. Movement: LS projection onto PSF gradient -------------------------
    #   img_diff ≈ -(delta_x / gamma) * d_base_dx
    #   => (delta_x / gamma) = -<img_diff, d_base_dx> / ||d_base_dx||^2
    ls_coef = (-np.dot(diff_prof, d_base_dx) /
               (np.dot(d_base_dx, d_base_dx) + 1e-12))   # [m] = delta_x / gamma
    dx_est  = r['data_ratio'] * ls_coef                   # [m]

    true_dx  = r['frac'] * lam
    err_mm   = (dx_est  - true_dx)  * 1e3
    x0_err   = (x_cross - x_sc0)   * 1e3 if not np.isnan(x_cross) else np.nan
    rel_err  = abs(err_mm) / (true_dx * 1e3) * 100

    true_dxs.append(true_dx)
    est_dxs.append(dx_est)

    print(f"  {r['label']:<7}  {true_dx*1e3:>13.3f}  {dx_est*1e3:>13.4f}  "
          f"{err_mm:>12.4f}  {rel_err:>9.1f}  {x0_err:>12.4f}")

# -- Figure 1: inversion quality ----------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
fig.suptitle("Resolving sub-wavelength scatterer movement from the migrated difference image",
             fontsize=12, fontweight='bold')

# Panel A: true vs estimated delta_x (log-log)
ax = axes[0]
ax.loglog(np.array(true_dxs)*1e3, np.array(est_dxs)*1e3, 'ko-', lw=1.8, ms=8)
lims = [min(true_dxs)*0.7*1e3, max(true_dxs)*1.3*1e3]
ax.loglog(lims, lims, 'r--', lw=1.2, label='y = x  (perfect)')
for r, td, ed in zip(results, true_dxs, est_dxs):
    ax.annotate(r['label'], (td*1e3, ed*1e3),
                textcoords='offset points', xytext=(5, 3), fontsize=9)
ax.axvline(lam*1e3/2, color='steelblue', ls=':', lw=1.2, label='lam/2 Rayleigh')
ax.set_xlabel("True delta_x [mm]", fontsize=10)
ax.set_ylabel("Estimated delta_x [mm]", fontsize=10)
ax.set_title("A.  PSF-gradient inversion\n"
             "y=x line = perfect recovery", fontsize=9)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which='both')

# Panel B: dipole profiles vs PSF-gradient model
ax = axes[1]
for r, dx_est, col in zip(results, est_dxs, cols):
    diff_prof = r['img_diff'][iz_sc, :]
    # model: img_diff ≈ -(delta_x / gamma) * d_base_dx
    model = -(dx_est / (r['data_ratio'] + 1e-12)) * d_base_dx
    pk = np.abs(diff_prof).max() + 1e-30
    ax.plot(x_b[x_zoom], diff_prof[x_zoom]/pk, color=col, lw=1.5,
            label=r['label'])
    ax.plot(x_b[x_zoom], model[x_zoom]/pk,     color=col, lw=0.8,
            ls='--', alpha=0.7)
ax.axvline(x_sc0, color='gray', ls='--', lw=0.9, label='x_sc0')
ax.axhline(0, color='k', lw=0.4)
ax.set_xlabel("Position [m]", fontsize=10)
ax.set_ylabel("Normalised profile at z_sc", fontsize=10)
ax.set_title("B.  Normalised dipole (solid) vs PSF-gradient model (dashed)\n"
             "Zero-crossing always at x_sc0", fontsize=9)
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)

# Panel C: zero-crossing accuracy
ax = axes[2]
zc_errors = [(c - x_sc0)*1e3 for c in crossings if not np.isnan(c)]
true_dxs_plot = [r['frac']*lam*1e3 for r, c in zip(results, crossings)
                 if not np.isnan(c)]
ax.semilogx(true_dxs_plot, zc_errors, 'ko-', lw=1.8, ms=8)
for r, c in zip(results, crossings):
    if not np.isnan(c):
        ax.annotate(r['label'], (r['frac']*lam*1e3, (c-x_sc0)*1e3),
                    textcoords='offset points', xytext=(5, 3), fontsize=9)
ax.axhline(0, color='r', ls='--', lw=1.2, label='exact (0 error)')
ax.axhline( dx_b*1e3/2, color='gray', ls=':', lw=0.9, label='+/-0.5 pixel')
ax.axhline(-dx_b*1e3/2, color='gray', ls=':', lw=0.9)
ax.axvline(lam*1e3/2, color='steelblue', ls=':', lw=1.2, label='lam/2 Rayleigh')
ax.set_xlabel("Shift / lambda * lambda [mm]  (log scale)", fontsize=10)
ax.set_ylabel("Position error [mm]", fontsize=10)
ax.set_title("C.  Position accuracy from zero-crossing\n"
             "Sub-pixel error for all shifts -- no Rayleigh limit on position", fontsize=9)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -- Figure 2: 2D spatial view of position and movement ----------------------
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
r_show = results[-1]   # smallest shift

ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo, x_hi = x_sc0 - 4*lam, x_sc0 + 4*lam
z_lo, z_hi = z_sc - 0.08,   z_sc + 0.08

def zoom2(ax):
    ax.set_xlim(x_lo, x_hi); ax.set_ylim(z_hi, z_lo)

# Baseline
clip1 = 1.0
im0 = axes2[0].imshow(img_base, aspect='auto', origin='upper', extent=ext_mig,
                       cmap='RdBu', vmin=-clip1, vmax=clip1, interpolation='bilinear')
axes2[0].plot(x_sc0, z_sc, 'g*', ms=12, zorder=5, label='true scatterer')
axes2[0].set_title("Migrated baseline", fontsize=10)
zoom2(axes2[0]); axes2[0].legend(fontsize=8)
fig2.colorbar(im0, ax=axes2[0], fraction=0.046, pad=0.04)

# Difference
clip2 = np.percentile(np.abs(r_show['img_diff']), 98)
im1 = axes2[1].imshow(r_show['img_diff'], aspect='auto', origin='upper', extent=ext_mig,
                       cmap='RdBu', vmin=-clip2, vmax=clip2, interpolation='bilinear')
xc_cell = 0.5*(x_b[:-1]+x_b[1:])
zc_cell = 0.5*(z_mig[:-1]+z_mig[1:])
pos_r, pos_c = np.where(r_show['charge'] == 1)
if len(pos_c):
    axes2[1].plot(xc_cell[pos_c], zc_cell[pos_r], 'r+', ms=10, mew=2,
                  label=f'q=+1  ({len(pos_c)})', zorder=6)
neg_r, neg_c = np.where(r_show['charge'] == -1)
if len(neg_c):
    axes2[1].plot(xc_cell[neg_c], zc_cell[neg_r], 'b+', ms=10, mew=2,
                  label=f'q=-1  ({len(neg_c)})', zorder=6)
axes2[1].set_title(f"Migrated difference (shift {r_show['label']})\n"
                   "topological charge (+/-) flanks the dipole zero", fontsize=9)
zoom2(axes2[1]); axes2[1].legend(fontsize=8)
fig2.colorbar(im1, ax=axes2[1], fraction=0.046, pad=0.04)

# Point-by-point delta_x map (ratio method, masked to high-gradient region)
thresh_grad = 0.15 * d_base_dx_peak
mask_2d = np.abs(np.gradient(img_base, x_b, axis=1)) > thresh_grad

diff_sc   = r_show['data_ratio']
d_base_2d = np.gradient(img_base, x_b, axis=1)
dx_map    = np.where(mask_2d,
                     -diff_sc * r_show['img_diff'] / (d_base_2d + 1e-10),
                     np.nan)

vmax_map = r_show['frac'] * lam * 3
im2 = axes2[2].imshow(dx_map * 1e3, aspect='auto', origin='upper', extent=ext_mig,
                       cmap='RdBu', vmin=-vmax_map*1e3, vmax=vmax_map*1e3,
                       interpolation='bilinear')
axes2[2].axhline(z_sc, color='gray', ls=':', lw=0.8)
axes2[2].plot(x_sc0, z_sc, 'g*', ms=12, zorder=5)
axes2[2].set_title(f"Pixel-wise delta_x estimate [mm]\n"
                   f"(masked where |d_base/dx| < 15% of peak)\n"
                   f"True = {r_show['frac']*lam*1e3:.1f} mm", fontsize=9)
zoom2(axes2[2])
cb = fig2.colorbar(im2, ax=axes2[2], fraction=0.046, pad=0.04)
cb.set_label("delta_x estimate [mm]", fontsize=8)

for ax in axes2: ax.set_xlabel("Position [m]", fontsize=9); ax.set_ylabel("Depth [m]", fontsize=9)
fig2.suptitle(f"2D spatial view: position + movement extraction  (shift {r_show['label']})",
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# -- Summary ------------------------------------------------------------------
print(f"\nlambda_ice = {lam*1e3:.2f} mm   dx_b = {dx_b*1e3:.1f} mm   Rayleigh = {lam*1e3/2:.2f} mm")
print(f"PSF gradient peak = {d_base_dx_peak:.3f} m^-1  (normalised)")
print()
print("Conclusion:")
print("  Position: zero-crossing of dipole gives x_sc0 to sub-pixel accuracy for ALL shifts")
print(f"  Movement: PSF-gradient inversion gives delta_x to <10% error for shifts >= {lam/16*1e3:.1f} mm")
print("  Both measurements are below the Rayleigh limit (lambda/2 = " + f"{lam/2*1e3:.1f} mm)")


In [ ]:
# =============================================================================
# Spatial CLSSA phase gather applied to timelapse migrated profiles
#
# CLSSA reinterpreted for space (x axis) instead of time (t axis):
#
#   dt      ->  dx_b  [m/sample]
#   f [GHz] ->  k  [cycles/m]
#   win_ns  ->  win_m [m]   (= win_m / dx_b samples)
#
# Advantages over CWT in the spatial domain:
#   - No cone of influence: uniform wavenumber resolution across the profile
#   - Window applied to the KERNEL (tapered sinusoids), not to the data
#     -> no edge smearing, sharp transitions in theta_dom(x) are preserved
#   - Frequency resolution = 1/win_m regardless of wavenumber
#
# Expected result (same physics as the CWT version):
#   Baseline PSF   -> theta_dom(x_sc0) ~   0 deg  (in-phase at amplitude peak)
#   Difference dipole -> theta_dom(x_sc0) ~ +90 deg  (quadrature at zero-crossing)
# The 90-deg anomaly is invariant to the shift magnitude -> sub-wavelength signature.
# =============================================================================

import sys, os, importlib
sys.path.insert(0, os.getcwd())
import phase_decomposition as pd_mod
importlib.reload(pd_mod)

import numpy as np
import matplotlib.pyplot as plt

# -- CLSSA spatial parameters -------------------------------------------------
#   Rule of thumb for win_m: 2-4 spatial wavelengths
#   (analogous to 2-4 dominant periods for temporal CLSSA)
CLSSA_SP = dict(
    f_min_GHz = 0.25 / lam,   # k_min [cycles/m]  (lam/4 wavenumber)
    f_max_GHz = 4.00 / lam,   # k_max [cycles/m]  (4/lam)
    n_freqs   = 60,
    win_ns    = 3.0  * lam,   # window = 3 lambda  (~67 samples)
    alpha     = 1e-2,
)

win_samp = int(round(CLSSA_SP['win_ns'] / dx_b))
print("CLSSA spatial parameters:")
print(f"  k_min  = {CLSSA_SP['f_min_GHz']:.2f} cycles/m  (lambda/4)")
print(f"  k_max  = {CLSSA_SP['f_max_GHz']:.2f} cycles/m  (4/lambda)")
print(f"  win    = {CLSSA_SP['win_ns']*1e3:.1f} mm = {CLSSA_SP['win_ns']/lam:.1f} lambda  ({win_samp} samples)")
print(f"  alpha  = {CLSSA_SP['alpha']}")
print()

# -- Extract profiles at scatterer depth --------------------------------------
iz_sc     = np.argmin(np.abs(z_mig - z_sc))
ix_sc0    = np.argmin(np.abs(x_b   - x_sc0))
base_prof = img_base[iz_sc, :]
x_zoom    = (x_b >= x_sc0 - 4*lam) & (x_b <= x_sc0 + 4*lam)
iz0, iz1  = np.where(x_zoom)[0][[0, -1]]

def clssa_sp(prof):
    """Spatial CLSSA: returns A, k_ax, x_ax, pg, theta_deg, R, I, env, theta_dom."""
    A, th2, pg, k_ax, x_ax, th_v = pd_mod.clssa_phase_decomposition(
        prof, dx_b, **CLSSA_SP)
    # Broadband phasor (integrate complex spectrum over wavenumber)
    C   = A * np.exp(1j * np.deg2rad(th2))
    R   = np.trapz(np.real(C), k_ax, axis=0)
    I   = np.trapz(np.imag(C), k_ax, axis=0)
    env = np.sqrt(R**2 + I**2)
    th_dom = np.rad2deg(np.arctan2(I, R))
    return A, k_ax, x_ax, pg, th_v, R, I, env, th_dom

# -- Baseline -----------------------------------------------------------------
t0_s = _time.time()
A_b, k_b, x_out, pg_b, tv, R_b, I_b, env_b, thdom_b = clssa_sp(base_prof)
print(f"Baseline  ({_time.time()-t0_s:.1f} s)  |  theta_dom at x_sc0 = {thdom_b[ix_sc0]:+.1f} deg  (expect ~0)")

# -- Difference profiles for all scenarios ------------------------------------
cols   = plt.cm.plasma(np.linspace(0.1, 0.9, len(results)))
pg_res = []

for r in results:
    t0_s = _time.time()
    dp    = r['img_diff'][iz_sc, :]
    A_d, k_d, _, pg_d, _, R_d, I_d, env_d, thdom_d = clssa_sp(dp)
    elapsed = _time.time() - t0_s
    pg_res.append({**r,
                   'dp': dp, 'A_d': A_d, 'k_d': k_d, 'pg_d': pg_d,
                   'R_d': R_d, 'I_d': I_d, 'env_d': env_d, 'thdom_d': thdom_d,
                   'theta_sc': float(thdom_d[ix_sc0]),
                   'I_sc':     float(I_d[ix_sc0])})
    print(f"  {r['label']:<8}  theta_dom at x_sc0 = {thdom_d[ix_sc0]:+.1f} deg  "
          f"  I_sc = {I_d[ix_sc0]:+.4f}   ({elapsed:.1f} s)")

print("\n  Expect theta ~ +90 deg  (quadrature = lateral shift signature)")

# -- Figure 1: 6-panel comparison (baseline vs smallest shift) ----------------
rs  = pg_res[-1]    # smallest shift (lam/32)
x_z = x_b[x_zoom]
ext_x  = [x_z[0],  x_z[-1]]
ext_sp = [x_z[0],  x_z[-1], float(k_b[0]),  float(k_b[-1])]   # (x_lo, x_hi, k_lo, k_hi)
ext_pg = [tv[0],   tv[-1],   x_z[0],         x_z[-1]]          # (theta_lo, theta_hi, x_lo, x_hi)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"Spatial CLSSA phase gather  —  baseline vs difference  (shift {rs['label']} = {rs['frac']*lam*1e3:.1f} mm)\n"
    "Col 1: profile + broadband envelope  |  Col 2: CLSSA amplitude [dB]  |  Col 3: phase gather S'(theta, x)",
    fontsize=11, fontweight='bold',
)

def A_dB(A):
    return np.clip(20*np.log10(A / (A.max()+1e-30) + 1e-30), -40, 0)

# Row 0: baseline
ax = axes[0, 0]
ax.plot(x_z, base_prof[x_zoom], 'k-',  lw=1.4, label='W(x)')
ax.plot(x_z, env_b[x_zoom],     'r-',  lw=1.0, alpha=0.8, label='envelope |P(x)|')
ax.axvline(x_sc0, color='g', ls='--', lw=1.0, label='x_sc0')
ax.axhline(0,     color='k', lw=0.3)
ax.set_title("Baseline PSF profile at z_sc", fontsize=10)
ax.set_ylabel("Amplitude"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 1]
im = ax.imshow(A_dB(A_b[:, iz0:iz1]),
               aspect='auto', origin='lower', extent=ext_sp,
               cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
ax.axvline(x_sc0, color='white', lw=0.8, ls='--')
ax.axhline(k_b.mean(), color='white', lw=0.7, ls=':', alpha=0.7, label=f'k_c={k_b.mean():.1f}')
ax.set_xlabel("Position [m]"); ax.set_ylabel("Wavenumber [cycles/m]")
ax.set_title("CLSSA amplitude spectrum — baseline [dB]", fontsize=10)
ax.legend(fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 2]
clip_b = np.percentile(np.abs(pg_b), 99)
# pg shape: (181, n_x); we want imshow(theta on x, position on y) -> transpose convention
im = ax.imshow(pg_b[:, iz0:iz1].T,
               aspect='auto', origin='lower',
               extent=[tv[0], tv[-1], x_z[0], x_z[-1]],
               cmap='RdBu', vmin=-clip_b, vmax=clip_b, interpolation='bilinear')
ax.axvline(thdom_b[ix_sc0], color='lime', lw=2.0, ls='--',
           label=f'theta_dom={thdom_b[ix_sc0]:.0f} deg at x_sc0')
ax.axvline(0,  color='white', lw=0.6, ls=':', alpha=0.5)
ax.axhline(x_sc0, color='white', lw=0.8, ls='--')
ax.set_xlabel("Phase theta [deg]"); ax.set_ylabel("Position [m]")
ax.set_title("Phase gather S'(theta, x) — baseline", fontsize=10)
ax.set_xticks([-180, -90, 0, 90, 180]); ax.legend(fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 1: difference (smallest shift)
ax = axes[1, 0]
ax.plot(x_z, rs['dp'][x_zoom],    color='steelblue', lw=1.4, label='W_diff(x)')
ax.plot(x_z, rs['env_d'][x_zoom], 'r-', lw=1.0, alpha=0.8, label='envelope |P_diff(x)|')
ax.axvline(x_sc0, color='g', ls='--', lw=1.0, label='x_sc0')
ax.axhline(0,     color='k', lw=0.3)
ax.set_title(f"Difference dipole at z_sc  (shift {rs['label']})", fontsize=10)
ax.set_ylabel("Amplitude [normalised ±1]"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 1]
im = ax.imshow(A_dB(rs['A_d'][:, iz0:iz1]),
               aspect='auto', origin='lower', extent=ext_sp,
               cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
ax.axvline(x_sc0, color='white', lw=0.8, ls='--')
ax.axhline(k_b.mean(), color='white', lw=0.7, ls=':', alpha=0.7)
ax.set_xlabel("Position [m]"); ax.set_ylabel("Wavenumber [cycles/m]")
ax.set_title(f"CLSSA amplitude spectrum — difference {rs['label']} [dB]", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 2]
clip_d = np.percentile(np.abs(rs['pg_d']), 99)
im = ax.imshow(rs['pg_d'][:, iz0:iz1].T,
               aspect='auto', origin='lower',
               extent=[tv[0], tv[-1], x_z[0], x_z[-1]],
               cmap='RdBu', vmin=-clip_d, vmax=clip_d, interpolation='bilinear')
ax.axvline(rs['theta_sc'], color='lime', lw=2.0, ls='--',
           label=f"theta_dom={rs['theta_sc']:.0f} deg at x_sc0")
ax.axvline(90,  color='lime', lw=0.8, ls=':', alpha=0.5)
ax.axvline(0,   color='white', lw=0.6, ls=':', alpha=0.5)
ax.axhline(x_sc0, color='white', lw=0.8, ls='--')
ax.set_xlabel("Phase theta [deg]"); ax.set_ylabel("Position [m]")
ax.set_title(f"Phase gather S'(theta, x) — difference {rs['label']}", fontsize=10)
ax.set_xticks([-180, -90, 0, 90, 180]); ax.legend(fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax in axes.flat: ax.set_xlabel(ax.get_xlabel() or "Position [m]", fontsize=9)
plt.tight_layout()
plt.show()

# -- Figure 2: theta_dom and quadrature amplitude vs shift --------------------
shifts_mm   = [r2['frac']*lam*1e3  for r2 in pg_res]
thetas_sc   = [r2['theta_sc']       for r2 in pg_res]
I_sc_scaled = [r2['I_sc'] * r2['data_ratio'] for r2 in pg_res]

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle("CLSSA spatial phase gather at x_sc0 vs shift — sub-wavelength sensitivity",
              fontsize=12, fontweight='bold')

# A: dominant phase
ax = axes2[0]
ax.semilogx(shifts_mm, thetas_sc, 'ko-', lw=1.8, ms=8)
for r2 in pg_res:
    ax.annotate(r2['label'], (r2['frac']*lam*1e3, r2['theta_sc']),
                textcoords='offset points', xytext=(4, 3), fontsize=9)
ax.axhline( 90, color='lime',    ls='--', lw=1.8, label='+90 deg (quadrature = shift signature)')
ax.axhline(  0, color='gray',    ls=':',  lw=0.9, label='0 deg  (baseline in-phase)')
ax.axhline(-90, color='magenta', ls='--', lw=0.8, alpha=0.5)
ax.axvline(lam*1e3/2, color='steelblue', ls=':', lw=1.2, label='lambda/2 Rayleigh')
ax.set_xlabel("Lateral shift [mm]  (log scale)", fontsize=10)
ax.set_ylabel("theta_dom at x_sc0  [deg]", fontsize=10)
ax.set_title("A.  Dominant spatial phase at scatterer\n"
             "CLSSA: ~+90 deg for ALL shifts, no COI smearing", fontsize=9)
ax.set_ylim(-200, 200); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# B: quadrature amplitude
ax = axes2[1]
I_plot = [abs(v) for v in I_sc_scaled]
ax.loglog(shifts_mm, I_plot, 'ko-', lw=1.8, ms=8)
for r2, Iv in zip(pg_res, I_plot):
    ax.annotate(r2['label'], (r2['frac']*lam*1e3, Iv),
                textcoords='offset points', xytext=(4, 3), fontsize=9)
lf = np.polyfit(np.log(shifts_mm), np.log([v+1e-15 for v in I_plot]), 1)
xf = np.linspace(min(shifts_mm)*0.8, max(shifts_mm)*1.2, 30)
ax.loglog(xf, np.exp(np.polyval(lf, np.log(xf))), 'r--',
          label=f'slope = {lf[0]:.2f}  (1.0 = linear -> delta_x proportional)')
ax.axvline(lam*1e3/2, color='steelblue', ls=':', lw=1.2, label='lambda/2 Rayleigh')
ax.set_xlabel("Lateral shift [mm]  (log scale)", fontsize=10)
ax.set_ylabel("|I_diff(x_sc0)| * data_ratio", fontsize=10)
ax.set_title("B.  Quadrature amplitude at x_sc0 vs shift\n"
             "Linear slope -> delta_x quantification below Rayleigh limit", fontsize=9)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# -- Summary table ------------------------------------------------------------
print()
print(f"  {'Shift':<8}  {'theta [deg]':>12}  {'I*ratio':>12}  {'data_ratio':>12}")
for r2, Iv in zip(pg_res, I_sc_scaled):
    print(f"  {r2['label']:<8}  {r2['theta_sc']:>+12.1f}  {Iv:>12.5f}  {r2['data_ratio']:>12.6f}")
print()
print("CLSSA vs CWT summary:")
print("  - No COI: wavenumber resolution uniform from edge to edge of profile")
# --- Multi-row comparison: baseline + all shift levels -----------------------
nrows = len(pg_res) + 1
figr, axr = plt.subplots(nrows, 3, figsize=(18, 4.2 * nrows), squeeze=False)
figr.suptitle(
    "Spatial CLSSA phase gather — baseline and all difference shift levels",
    fontsize=12, fontweight='bold'
)

# Row 0: baseline
ax = axr[0, 0]
ax.plot(x_z, base_prof[x_zoom], 'k-', lw=1.4, label='W(x)')
ax.plot(x_z, env_b[x_zoom],     'r-', lw=1.0, alpha=0.8, label='envelope |P(x)|')
ax.axvline(x_sc0, color='g', ls='--', lw=1.0)
ax.axhline(0, color='k', lw=0.3)
ax.set_title("Baseline PSF profile", fontsize=10)
ax.set_ylabel("Amplitude")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axr[0, 1]
im = ax.imshow(A_dB(A_b[:, iz0:iz1]), aspect='auto', origin='lower',
               extent=ext_sp, cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
ax.axvline(x_sc0, color='white', lw=0.8, ls='--')
ax.set_title("Baseline CLSSA amplitude [dB]", fontsize=10)
ax.set_xlabel("Position [m]")
ax.set_ylabel("Wavenumber [cycles/m]")
figr.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axr[0, 2]
clip_b = np.percentile(np.abs(pg_b), 99)
im = ax.imshow(pg_b[:, iz0:iz1].T, aspect='auto', origin='lower',
               extent=[tv[0], tv[-1], x_z[0], x_z[-1]],
               cmap='RdBu', vmin=-clip_b, vmax=clip_b, interpolation='bilinear')
ax.axvline(thdom_b[ix_sc0], color='lime', lw=2.0, ls='--')
ax.axhline(x_sc0, color='white', lw=0.8, ls='--')
ax.set_title("Baseline phase gather", fontsize=10)
ax.set_xlabel("Phase theta [deg]")
ax.set_ylabel("Position [m]")
ax.set_xticks([-180, -90, 0, 90, 180])
figr.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Remaining rows: all shifts
for i, r in enumerate(pg_res, start=1):
    ax = axr[i, 0]
    ax.plot(x_z, r['dp'][x_zoom], color='steelblue', lw=1.4, label='W_diff(x)')
    ax.plot(x_z, r['env_d'][x_zoom], 'r-', lw=1.0, alpha=0.8, label='envelope |P_diff(x)|')
    ax.axvline(x_sc0, color='g', ls='--', lw=1.0)
    ax.axhline(0, color='k', lw=0.3)
    ax.set_title(f"Difference profile — shift {r['label']}", fontsize=10)
    ax.set_ylabel("Amplitude")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    ax = axr[i, 1]
    im = ax.imshow(A_dB(r['A_d'][:, iz0:iz1]), aspect='auto', origin='lower',
                   extent=ext_sp, cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
    ax.axvline(x_sc0, color='white', lw=0.8, ls='--')
    ax.set_title(f"CLSSA amplitude — shift {r['label']} [dB]", fontsize=10)
    ax.set_xlabel("Position [m]")
    ax.set_ylabel("Wavenumber [cycles/m]")
    figr.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax = axr[i, 2]
    clip_d = np.percentile(np.abs(r['pg_d']), 99)
    im = ax.imshow(r['pg_d'][:, iz0:iz1].T, aspect='auto', origin='lower',
                   extent=[tv[0], tv[-1], x_z[0], x_z[-1]],
                   cmap='RdBu', vmin=-clip_d, vmax=clip_d, interpolation='bilinear')
    ax.axvline(r['theta_sc'], color='lime', lw=2.0, ls='--',
               label=f"theta_dom={r['theta_sc']:.0f} deg")
    ax.axhline(x_sc0, color='white', lw=0.8, ls='--')
    ax.set_title(f"Phase gather — shift {r['label']}", fontsize=10)
    ax.set_xlabel("Phase theta [deg]")
    ax.set_ylabel("Position [m]")
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.legend(fontsize=8)
    figr.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
print("  - Amplitude I_sc * data_ratio ~ linear in delta_x (sub-wavelength quantification)")


In [ ]:
# =============================================================================
# Temporal CLSSA on the difference B-scan traces
#
# Why the spatial CLSSA phase was -111 to -160 deg (not exactly +-90 deg):
#   The Gazdag PSF has a carrier phase offset phi_0 != 0 deg.
#   Spatial theta_dom of baseline at x_sc0 = phi_0  (not 0 deg).
#   Spatial theta_dom of difference at x_sc0 = phi_0 - 90 deg.
#   The RELATIVE shift is still -90 deg, but the absolute value drifts.
#
# Temporal CLSSA is cleaner because:
#   B_diff(x, t) ≈ -Delta_twt(x) * dB_base/dt
#   Delta_twt(x) = -2(x-x_sc) * delta_x / (v * r(x))  [first-order for x != x_sc]
#   d(Ricker)/dt = Ricker phase-shifted by -90 deg  (temporal quadrature)
#   => theta_dom(t_arrival) ~ -90 deg  for ANY delta_x including sub-wavelength
#
# Reference trace: x_ref = x_sc0 + lam/2  (off-apex, first-order in delta_x)
# Apex trace:      x = x_sc0               (second-order in delta_x; much smaller)
# =============================================================================

import sys, os, importlib
sys.path.insert(0, os.getcwd())
import phase_decomposition as pd_mod
importlib.reload(pd_mod)

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# -- Temporal CLSSA parameters (standard GPR, same as earlier cells) ----------
CLSSA_T = dict(
    f_min_GHz = 0.3,
    f_max_GHz = 4.0,
    n_freqs   = 60,
    win_ns    = 2.0,   # ~3 dominant periods at 1.5 GHz
    alpha     = 1e-2,
)

# -- Reference trace position -------------------------------------------------
x_ref   = x_sc0 + 0.5 * lam          # off-apex: first-order sensitivity
r_ref   = np.sqrt((x_ref - x_sc0)**2 + z_sc**2)
twt_ref = 2.0 * r_ref / v_ice         # expected arrival time [ns]
ix_ref  = np.argmin(np.abs(x_b - x_ref))

print(f"Reference trace: x_ref = {x_ref*1e3:.1f} mm  (x_sc0 + lam/2)")
print(f"  Hyperbola arrival: twt_ref = {twt_ref:.3f} ns")
print(f"  Trace index: ix_ref = {ix_ref}")
print()

# -- Temporal CLSSA of BASELINE at x_ref --------------------------------------
base_trace  = B_base[:, ix_ref]
base_norm   = base_trace / (np.abs(base_trace).max() + 1e-30)

t0_c = _time.time()
A_bt, th2_bt, pg_bt, f_bt, t_bt, tv_bt = pd_mod.clssa_phase_decomposition(
    base_norm, dt_b, **CLSSA_T)

# Broadband phasor -> dominant temporal phase
C_bt   = A_bt * np.exp(1j * np.deg2rad(th2_bt))
R_bt   = np.trapz(np.real(C_bt), f_bt, axis=0)
I_bt   = np.trapz(np.imag(C_bt), f_bt, axis=0)
env_bt = np.sqrt(R_bt**2 + I_bt**2)
thdom_bt = np.rad2deg(np.arctan2(I_bt, R_bt))

it_ref   = np.argmin(np.abs(t_bt - twt_ref))
print(f"Baseline at x_ref: theta_dom at twt_ref = {thdom_bt[it_ref]:+.1f} deg  (expect ~0 deg)")
print(f"  CLSSA time: {_time.time()-t0_c:.1f} s")

# -- Temporal CLSSA of each DIFFERENCE trace at x_ref ------------------------
cols     = plt.cm.plasma(np.linspace(0.1, 0.9, len(results)))
clssa_res = []

for r in results:
    diff_tr = r['B_diff'][:, ix_ref]
    # normalise each trace individually (CLSSA is scale-invariant; we care about phase)
    max_d   = np.abs(diff_tr).max() + 1e-30
    diff_norm = diff_tr / max_d

    t0_c = _time.time()
    A_dt, th2_dt, pg_dt, f_dt, t_dt, tv_dt = pd_mod.clssa_phase_decomposition(
        diff_norm, dt_b, **CLSSA_T)

    C_dt     = A_dt * np.exp(1j * np.deg2rad(th2_dt))
    R_dt     = np.trapz(np.real(C_dt), f_dt, axis=0)
    I_dt     = np.trapz(np.imag(C_dt), f_dt, axis=0)
    env_dt   = np.sqrt(R_dt**2 + I_dt**2)
    thdom_dt = np.rad2deg(np.arctan2(I_dt, R_dt))

    theta_at_arr = float(thdom_dt[it_ref])
    rel_phase    = theta_at_arr - thdom_bt[it_ref]   # should be ~ -90 deg

    clssa_res.append({**r,
                      'diff_tr': diff_tr, 'diff_norm': diff_norm, 'max_d': max_d,
                      'A_dt': A_dt, 'th2_dt': th2_dt, 'pg_dt': pg_dt,
                      'R_dt': R_dt, 'I_dt': I_dt, 'env_dt': env_dt,
                      'thdom_dt': thdom_dt,
                      'theta_arr': theta_at_arr,
                      'rel_phase': rel_phase})
    print(f"  {r['label']:<8}  theta_dom at twt_ref = {theta_at_arr:+.1f} deg  "
          f"  relative = {rel_phase:+.1f} deg  "
          f"  ({_time.time()-t0_c:.1f} s)")

print()
print(f"  Relative phase = theta_dom_diff - theta_dom_base  (expect ~ -90 deg)")

# -- Figure 1: CLSSA 3-panel for baseline and smallest shift ------------------
rs = clssa_res[-1]

t_lo = twt_ref - 1.2
t_hi = twt_ref + 1.2
ext_spec = [f_bt[0], f_bt[-1], t_hi, t_lo]
ext_pg   = [tv_bt[0], tv_bt[-1], t_hi, t_lo]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"Temporal CLSSA at x_ref = x_sc0 + lam/2 = {x_ref*1e3:.0f} mm\n"
    f"Baseline  vs  difference trace  (shift {rs['label']} = {rs['frac']*lam*1e3:.1f} mm)",
    fontsize=12, fontweight='bold',
)

t_mask = (t_bt >= t_lo) & (t_bt <= t_hi)

def amp_db(A):
    return np.clip(20*np.log10(A / (A.max()+1e-30) + 1e-30), -40, 0)

# Row 0: baseline
ax = axes[0, 0]
ax.plot(t_bt, base_norm, 'k-', lw=1.3)
ax.plot(t_bt, env_bt,    'r-', lw=1.0, alpha=0.8, label='envelope')
ax.axvline(twt_ref, color='g', ls='--', lw=1.2, label=f'twt_ref={twt_ref:.3f} ns')
ax.set_xlim(t_lo, t_hi); ax.set_title("Baseline trace at x_ref", fontsize=10)
ax.set_xlabel("TWT [ns]"); ax.set_ylabel("Amplitude"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 1]
im = ax.imshow(amp_db(A_bt[:, t_mask]).T,
               aspect='auto', origin='upper', extent=ext_spec,
               cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
ax.axhline(twt_ref, color='white', lw=1.0, ls='--')
ax.set_xlabel("Frequency [GHz]"); ax.set_ylabel("TWT [ns]")
ax.set_title("CLSSA amplitude — baseline [dB]", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 2]
clip_b = np.percentile(np.abs(pg_bt), 99)
im = ax.imshow(pg_bt[:, t_mask].T,
               aspect='auto', origin='upper',
               extent=ext_pg,
               cmap='RdBu', vmin=-clip_b, vmax=clip_b, interpolation='bilinear')
ax.axhline(twt_ref, color='white', lw=1.0, ls='--')
ax.axvline(thdom_bt[it_ref], color='lime', lw=2.0, ls='--',
           label=f'theta_dom={thdom_bt[it_ref]:.0f} deg')
ax.axvline(0, color='white', lw=0.6, ls=':', alpha=0.5)
ax.set_xlabel("Phase theta [deg]"); ax.set_ylabel("TWT [ns]")
ax.set_title("Phase gather S'(theta, t) — baseline", fontsize=10)
ax.set_xticks([-180, -90, 0, 90, 180]); ax.legend(fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 1: smallest difference
ax = axes[1, 0]
ax.plot(t_bt, rs['diff_norm'], color='steelblue', lw=1.3)
ax.plot(t_bt, rs['env_dt'],   'r-', lw=1.0, alpha=0.8, label='envelope')
ax.axvline(twt_ref, color='g', ls='--', lw=1.2, label=f'twt_ref={twt_ref:.3f} ns')
ax.set_xlim(t_lo, t_hi)
ax.set_title(f"Difference trace (shift {rs['label']}) — normalised\n"
             f"raw amplitude = {rs['max_d']:.2e}  (data ratio={rs['data_ratio']:.4f})", fontsize=9)
ax.set_xlabel("TWT [ns]"); ax.set_ylabel("Amplitude [norm ±1]")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 1]
im = ax.imshow(amp_db(rs['A_dt'][:, t_mask]).T,
               aspect='auto', origin='upper', extent=ext_spec,
               cmap='jet', vmin=-40, vmax=0, interpolation='bilinear')
ax.axhline(twt_ref, color='white', lw=1.0, ls='--')
ax.set_xlabel("Frequency [GHz]"); ax.set_ylabel("TWT [ns]")
ax.set_title(f"CLSSA amplitude — difference {rs['label']} [dB]", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 2]
clip_d = np.percentile(np.abs(rs['pg_dt']), 99)
im = ax.imshow(rs['pg_dt'][:, t_mask].T,
               aspect='auto', origin='upper',
               extent=ext_pg,
               cmap='RdBu', vmin=-clip_d, vmax=clip_d, interpolation='bilinear')
ax.axhline(twt_ref, color='white', lw=1.0, ls='--')
ax.axvline(rs['theta_arr'], color='lime', lw=2.0, ls='--',
           label=f"theta_dom={rs['theta_arr']:.0f} deg")
ax.axvline(-90, color='cyan', lw=1.0, ls=':', alpha=0.8, label='-90 deg')
ax.axvline(0,   color='white', lw=0.6, ls=':', alpha=0.5)
ax.set_xlabel("Phase theta [deg]"); ax.set_ylabel("TWT [ns]")
ax.set_title(f"Phase gather S'(theta, t) — difference {rs['label']}\n"
             f"theta_dom - baseline = {rs['rel_phase']:+.1f} deg  (expect -90 deg)", fontsize=9)
ax.set_xticks([-180, -90, 0, 90, 180]); ax.legend(fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# -- Figure 2: relative temporal phase vs shift + B-scan phase map ------------
fig2, axes2 = plt.subplots(1, 3, figsize=(19, 5))
fig2.suptitle("Temporal CLSSA phase — sub-wavelength sensitivity",
              fontsize=12, fontweight='bold')

# A: relative phase vs shift
shifts_mm  = [r2['frac']*lam*1e3 for r2 in clssa_res]
rel_phases = [r2['rel_phase']    for r2 in clssa_res]
ax = axes2[0]
ax.semilogx(shifts_mm, rel_phases, 'ko-', lw=1.8, ms=8)
for r2 in clssa_res:
    ax.annotate(r2['label'], (r2['frac']*lam*1e3, r2['rel_phase']),
                textcoords='offset points', xytext=(4, 3), fontsize=9)
ax.axhline(-90, color='lime', ls='--', lw=1.8, label='-90 deg (expected)')
ax.axhline(  0, color='gray', ls=':',  lw=0.9, label='0 deg (baseline)')
ax.axvline(lam*1e3/2, color='steelblue', ls=':', lw=1.2, label='lam/2 Rayleigh')
ax.set_xlabel("Lateral shift [mm]  (log scale)", fontsize=10)
ax.set_ylabel("theta_dom_diff - theta_dom_base  [deg]", fontsize=10)
ax.set_title("A.  Relative temporal phase at twt_ref\n"
             "~-90 deg for ALL shifts including sub-wavelength", fontsize=9)
ax.set_ylim(-200, 50); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# B: instantaneous-phase B-scan of difference (smallest shift) using Hilbert
B_diff_s = clssa_res[-1]['B_diff']       # (n_t, n_x)
phi_inst  = np.angle(hilbert(B_diff_s, axis=0))   # instantaneous temporal phase
amp_inst  = np.abs( hilbert(B_diff_s, axis=0))

# mask: only show where amplitude > threshold
amp_thresh = 0.05 * amp_inst.max()
phi_masked = np.where(amp_inst > amp_thresh, np.rad2deg(phi_inst), np.nan)

ax = axes2[1]
t_lo2 = twt_ref - 1.5;  t_hi2 = twt_ref + 1.5
im = ax.imshow(phi_masked,
               aspect='auto', origin='upper',
               extent=[x_b[0], x_b[-1], t_b[-1], t_b[0]],
               cmap='hsv', vmin=-180, vmax=180, interpolation='nearest')
# overlay expected hyperbola
x_hyp = x_b
r_hyp = np.sqrt((x_hyp - x_sc0)**2 + z_sc**2)
ax.plot(x_hyp, 2*r_hyp/v_ice, 'k--', lw=1.0, alpha=0.6, label='expected hyperbola')
ax.axvline(x_sc0, color='g', ls='--', lw=0.8, label='x_sc0')
ax.axvline(x_ref, color='cyan', ls=':',  lw=1.0, label='x_ref')
ax.set_ylim(t_hi2, t_lo2)
ax.set_xlabel("Position [m]"); ax.set_ylabel("TWT [ns]")
ax.set_title(f"Instantaneous temporal phase B-scan\nB_diff (shift {clssa_res[-1]['label']}), "
             f"masked to amp > {amp_thresh:.1e}", fontsize=9)
ax.legend(fontsize=7); fig2.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("deg", fontsize=8)

# C: phase profile along the hyperbola for all shifts
ax = axes2[2]
for r2, col in zip(clssa_res, cols):
    phi_along = []
    for ix, xi in enumerate(x_b):
        ri  = np.sqrt((xi - x_sc0)**2 + z_sc**2)
        it  = np.argmin(np.abs(t_b - 2*ri/v_ice))
        phi_along.append(float(np.rad2deg(phi_inst[it, ix])) if
                         float(amp_inst[it, ix]) > 0.01*amp_inst.max() else np.nan)
    ax.plot(x_b*1e3, phi_along, color=col, lw=1.2, alpha=0.8, label=r2['label'])
ax.axhline(-90, color='lime', ls='--', lw=1.5, label='-90 deg')
ax.axhline( 90, color='lime', ls=':',  lw=0.8)
ax.axhline(  0, color='gray', ls=':',  lw=0.8)
ax.axvline(x_sc0*1e3, color='green', ls='--', lw=0.9, label='x_sc0')
ax.axvline(x_ref*1e3, color='cyan',  ls=':',  lw=0.9, label='x_ref')
ax.set_xlabel("Position [mm]", fontsize=10)
ax.set_ylabel("Instantaneous temporal phase at hyperbola  [deg]", fontsize=10)
ax.set_title("C.  Phase along the hyperbola — all shifts\n"
             "~-90 deg off-axis, undefined at apex (apex is 2nd-order in delta_x)", fontsize=9)
ax.set_xlim((x_sc0 - 5*lam)*1e3, (x_sc0 + 5*lam)*1e3)
ax.set_ylim(-200, 200); ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -- Summary ------------------------------------------------------------------
print(f"Reference trace: x_ref = x_sc0 + lam/2 = {x_ref*1e3:.1f} mm")
print(f"Baseline theta_dom at twt_ref = {thdom_bt[it_ref]:+.1f} deg")
print()
print(f"  {'Shift':<8}  {'theta_abs':>10}  {'rel_phase':>11}  {'data_ratio':>12}")
for r2 in clssa_res:
    print(f"  {r2['label']:<8}  {r2['theta_arr']:>+10.1f}  {r2['rel_phase']:>+11.1f}  {r2['data_ratio']:>12.6f}")
print()
print("Spatial vs temporal CLSSA comparison:")
print("  Spatial:   absolute phase drifts with PSF carrier -> -111 to -160 deg")
print("  Temporal:  relative phase (diff - base) = -90 deg -> carrier cancels")
print("  Both encode the sub-wavelength shift; temporal CLSSA is more interpretable.")


In [ ]:
# =============================================================================
# Sub-wavelength timelapse inference from migrated difference images
#
# Three distinct change types produce three distinct signatures:
#
#   TYPE        B-scan diff   Migrated diff    Temporal CLSSA
#   ─────────── ────────────  ───────────────  ─────────────────────────────
#   Lateral dx  anti-symm.    DIPOLE in x      theta ~ -90 deg (off-apex)
#               hyperbola     img_diff ~ -dx*d(img_base)/dx
#
#   Depth dz    symm.         BIPOLE in z      theta ~ -90 deg (at apex)
#               hyperbola     img_diff ~ -dz*d(img_base)/dz
#
#   Amplitude   symm.         SCALED copy      theta ~   0 deg (anywhere)
#   change dA   hyperbola     img_diff ~  dA * img_base
#
# Classification from img_diff alone:
#   1. Anti-symmetry ratio of x-profile at z_sc: high -> lateral, low -> symm
#   2. z-Hilbert phase of column at x_sc0:       +-90 -> depth, ~0 -> amplitude
#
# Quantification (all three types):
#   lateral  :  dx_est = data_ratio * LS{ img_diff, d(img_base)/dx }
#   depth    :  dz_est = data_ratio * LS{ img_diff, d(img_base)/dz }
#   amplitude:  dA_est = data_ratio  (linear in dA by construction)
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# ── magnitudes (all lambda/16 so comparison is fair) ─────────────────────────
change_mag = lam / 16.0
print(f"Change magnitude = lambda/16 = {change_mag*1e3:.2f} mm  "
      f"(lambda = {lam*1e3:.2f} mm,  dx_b = {dx_b*1e3:.1f} mm)")

# ── Scenario A: lateral shift (reuse from results) ───────────────────────────
r_lat      = results[-1]           # already computed
B_diff_lat = r_lat['B_diff']       # (n_t, n_x)
img_diff_lat = r_lat['img_diff']   # (n_z, n_x)  normalised to +-1
dr_lat     = r_lat['data_ratio']

# ── Scenario B: depth shift dz = lambda/16 ───────────────────────────────────
print("\nMigrating depth-shift scenario...")
t0 = _time.time()
B_mon_dz   = point_bscan(x_sc0, z_sc + change_mag, x_b, t_b, v_ice, f_c)
B_diff_dz  = B_mon_dz - B_base
img_diff_dz = gazdag_migration(B_diff_dz, x_b, t_b, z_mig, v_ice)
dr_dz       = float(np.max(np.abs(B_diff_dz))) / float(np.max(np.abs(B_base)))
print(f"  done in {_time.time()-t0:.1f} s  |  data_ratio = {dr_dz:.5f}")

# ── Scenario C: amplitude change dA = lambda/16  (as fraction of peak) ───────
# Physically: scatterer's reflectivity changed by dA fraction
# -> B_diff = dA * B_base  (exact, no new migration needed; img_diff ≈ dA*img_base)
dA         = change_mag        # fractional change  (same numeric value for fair comparison)
B_diff_dA  = dA * B_base       # (n_t, n_x)
print("\nMigrating amplitude-change scenario...")
t0 = _time.time()
img_diff_dA = gazdag_migration(B_diff_dA, x_b, t_b, z_mig, v_ice)
dr_dA       = float(np.max(np.abs(B_diff_dA))) / float(np.max(np.abs(B_base)))
print(f"  done in {_time.time()-t0:.1f} s  |  data_ratio = {dr_dA:.5f}")

# ── PSF derivatives at scatterer depth and position ──────────────────────────
iz_sc = np.argmin(np.abs(z_mig - z_sc))
ix_sc0 = np.argmin(np.abs(x_b - x_sc0))

base_prof = img_base[iz_sc, :]               # x-profile  (n_x,)
base_col  = img_base[:, ix_sc0]              # z-column   (n_z,)

d_base_dx = np.gradient(base_prof, x_b)      # d/dx  [1/m]
d_base_dz = np.gradient(base_col,  z_mig)    # d/dz  [1/m]

def ls_invert(signal_prof, gradient_prof):
    """Least-squares inversion:  signal ≈ -C * gradient  ->  C."""
    return (-np.dot(signal_prof, gradient_prof) /
            (np.dot(gradient_prof, gradient_prof) + 1e-12))

# ── Anti-symmetry ratio for x-profiles ───────────────────────────────────────
def asym_ratio(prof, ix0):
    """Fraction of signal power in the anti-symmetric component about ix0."""
    half = min(ix0, len(prof) - ix0 - 1)
    if half < 3:
        return np.nan
    left  = prof[ix0 - half : ix0]
    right = prof[ix0 : ix0 + half][::-1]
    anti  = 0.5 * (left - right)
    sym   = 0.5 * (left + right)
    tot   = np.dot(anti, anti) + np.dot(sym, sym)
    return float(np.dot(anti, anti) / (tot + 1e-12))

# ── z-Hilbert phase at (x_sc0, z_sc) for classification ──────────────────────
def z_phase(img, ix, iz):
    """Instantaneous phase along z-column at column ix, evaluated at depth iz."""
    col = img[:, ix]
    return float(np.rad2deg(np.angle(hilbert(col)[iz])))

# ── Apply classification + quantification to all three scenarios ──────────────
scenarios = [
    ('Lateral dx',   img_diff_lat, dr_lat, change_mag, 0,          'steelblue'),
    ('Depth   dz',   img_diff_dz,  dr_dz,  0,          change_mag, 'darkorange'),
    ('Ampl.  dA',    img_diff_dA,  dr_dA,  0,          0,          'crimson'),
]

print(f"\n{'Scenario':<14}  {'AR':>6}  {'z-phase':>8}  {'class':>8}  "
      f"{'est dx [mm]':>12}  {'est dz [mm]':>12}  {'est dA':>8}  {'err %':>7}")
print("-" * 90)

results_inf = []
for name, img_d, dr, true_dx, true_dz, col in scenarios:
    # profiles
    diff_prof = img_d[iz_sc, :]
    diff_col  = img_d[:, ix_sc0]

    # classification tests
    ar    = asym_ratio(diff_prof, ix_sc0)
    zph   = z_phase(img_d, ix_sc0, iz_sc)

    if ar > 0.5:
        chtype = 'lateral'
    elif abs(zph) > 45:
        chtype = 'depth'
    else:
        chtype = 'amplitude'

    # quantification
    C_dx = ls_invert(diff_prof, d_base_dx)   # [m]
    C_dz = ls_invert(diff_col,  d_base_dz)   # [m]
    dx_est = dr * C_dx                         # [m]
    dz_est = dr * C_dz                         # [m]
    dA_est = dr                                # dimensionless

    # error (only report the relevant estimate)
    if true_dx:
        err = 100 * abs(dx_est - true_dx) / true_dx
        relevant = f'{dx_est*1e3:.3f} mm  ({err:.1f}% err)'
    elif true_dz:
        err = 100 * abs(dz_est - true_dz) / true_dz
        relevant = f'{dz_est*1e3:.3f} mm  ({err:.1f}% err)'
    else:
        err = 100 * abs(dA_est - dA) / dA
        relevant = f'dA={dA_est:.4f}  ({err:.1f}% err)'

    print(f"  {name:<12}  {ar:.3f}  {zph:+8.1f}  {chtype:>8}  "
          f"{dx_est*1e3:>12.4f}  {dz_est*1e3:>12.4f}  {dA_est:>8.5f}  {err:>6.1f}%")
    results_inf.append(dict(name=name, img_d=img_d, dr=dr,
                            true_dx=true_dx, true_dz=true_dz,
                            ar=ar, zph=zph, chtype=chtype,
                            dx_est=dx_est, dz_est=dz_est, dA_est=dA_est, col=col))

# ── Figure 1: migrated difference images for all three scenarios ──────────────
x_lo, x_hi = x_sc0 - 4*lam, x_sc0 + 4*lam
z_lo, z_hi = z_sc  - 0.08,   z_sc  + 0.08
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
fig.suptitle(
    f"Sub-wavelength inference from migrated difference images  (all changes = lambda/16 = {change_mag*1e3:.1f} mm)\n"
    "Column 1: migrated difference  |  Column 2: x-profile at z_sc  |  Column 3: z-column at x_sc0",
    fontsize=11, fontweight='bold',
)

for row, r in enumerate(results_inf):
    img_d = r['img_d']
    clip  = np.percentile(np.abs(img_d), 98)

    # Col 0: 2D migrated difference
    ax = axes[row, 0]
    im = ax.imshow(img_d, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    ax.plot(x_sc0, z_sc, 'g*', ms=12, zorder=5)
    ax.set_xlim(x_lo, x_hi); ax.set_ylim(z_hi, z_lo)
    ax.set_title(f"{r['name']}  —  migrated difference\n"
                 f"class={r['chtype']},  AR={r['ar']:.2f},  z-phase={r['zph']:+.0f} deg",
                 fontsize=9)
    ax.set_xlabel("Position [m]"); ax.set_ylabel("Depth [m]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Col 1: x-profile at z_sc with anti-symmetry decomposition
    ax = axes[row, 1]
    prof = img_d[iz_sc, :]
    ix0  = ix_sc0
    half = min(ix0, len(prof) - ix0 - 1)
    left  = prof[ix0-half:ix0]
    right = prof[ix0:ix0+half][::-1]
    anti  = 0.5 * (left - right)
    sym   = 0.5 * (left + right)
    x_left = x_b[ix0-half:ix0]

    ax.plot(x_b, prof, color=r['col'], lw=1.5, label='profile', zorder=3)
    ax.fill_between(x_left, anti,  0, alpha=0.35, color='steelblue', label='anti-sym')
    ax.fill_between(x_left, sym,   0, alpha=0.35, color='orange',    label='sym')
    ax.axvline(x_sc0, color='g', ls='--', lw=0.9)
    ax.axhline(0, color='k', lw=0.3)
    ax.set_xlim(x_lo, x_hi)
    ax.set_title(f"x-profile at z_sc  —  AR = {r['ar']:.3f}", fontsize=9)
    ax.set_xlabel("Position [m]"); ax.set_ylabel("Amplitude")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

    # Col 2: z-column at x_sc0 with Hilbert overlay
    ax = axes[row, 2]
    col_d   = img_d[:, ix_sc0]
    H_col   = np.imag(hilbert(col_d))
    env_col = np.abs( hilbert(col_d))
    ax.plot(z_mig, col_d,   color=r['col'], lw=1.5, label='img_diff column')
    ax.plot(z_mig, H_col,   color='gray',   lw=1.0, ls='--', alpha=0.8, label='Hilbert')
    ax.plot(z_mig, env_col, color='red',    lw=0.9, alpha=0.7, label='envelope')
    ax.axvline(z_sc, color='g', ls='--', lw=0.9, label='z_sc')
    ax.axhline(0, color='k', lw=0.3)
    ax.set_xlim(z_lo, z_hi)
    ax.set_title(f"z-column at x_sc0  —  z-phase at z_sc = {r['zph']:+.0f} deg", fontsize=9)
    ax.set_xlabel("Depth [m]"); ax.set_ylabel("Amplitude")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Figure 2: sensitivity curves — all three types vs sub-wavelength magnitude
print("\nComputing sensitivity curves (all types, 6 magnitudes)...")
test_fracs  = [1.0, 0.5, 0.25, 0.125, 1/16, 1/32, 1/64]   # as fraction of lambda
test_labels = ['1lam','lam/2','lam/4','lam/8','lam/16','lam/32', 'lam/64']
est_dx_arr, est_dz_arr, est_dA_arr = [], [], []
true_arr = [f*lam for f in test_fracs]

for frac in test_fracs:
    mag = frac * lam
    # Lateral: reuse from existing results if available, else compute
    r_match = next((r for r in results if abs(r['frac'] - frac) < 1e-6), None)
    if r_match:
        img_dl = r_match['img_diff']
        dr_l   = r_match['data_ratio']
    else:
        B_mon_l   = point_bscan(x_sc0 + mag, z_sc, x_b, t_b, v_ice, f_c)
        B_diff_l  = B_mon_l - B_base
        img_dl    = gazdag_migration(B_diff_l, x_b, t_b, z_mig, v_ice)
        dr_l      = float(np.max(np.abs(B_diff_l))) / float(np.max(np.abs(B_base)))
    C_l = ls_invert(img_dl[iz_sc, :], d_base_dx)
    est_dx_arr.append(dr_l * C_l)

    # Depth
    B_mon_dz2  = point_bscan(x_sc0, z_sc + mag, x_b, t_b, v_ice, f_c)
    B_diff_dz2 = B_mon_dz2 - B_base
    img_dz2    = gazdag_migration(B_diff_dz2, x_b, t_b, z_mig, v_ice)
    dr_dz2     = float(np.max(np.abs(B_diff_dz2))) / float(np.max(np.abs(B_base)))
    C_z        = ls_invert(img_dz2[:, ix_sc0], d_base_dz)
    est_dz_arr.append(dr_dz2 * C_z)

    # Amplitude (no new migration: img_diff_amp ≈ dA*img_base, dr ≈ dA)
    est_dA_arr.append(mag)   # dA_est = dr = dA  (exact by construction)

true_mm = [t*1e3 for t in true_arr]
est_dx_mm = [abs(e)*1e3 for e in est_dx_arr]
est_dz_mm = [abs(e)*1e3 for e in est_dz_arr]
est_dA_mm = [abs(e)*1e3 for e in est_dA_arr]

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle("Sub-wavelength inference accuracy for all three change types",
              fontsize=12, fontweight='bold')

ax = axes2[0]
ax.loglog(true_mm, est_dx_mm,  'o-', color='steelblue',   lw=1.8, ms=8, label='Lateral dx  (PSF x-gradient LS)')
ax.loglog(true_mm, est_dz_mm,  's-', color='darkorange',  lw=1.8, ms=8, label='Depth dz    (PSF z-gradient LS)')
ax.loglog(true_mm, est_dA_mm,  '^-', color='crimson',     lw=1.8, ms=8, label='Amplitude dA (data_ratio)')
lims = [min(true_mm)*0.7, max(true_mm)*1.3]
ax.loglog(lims, lims, 'k--', lw=1.0, label='y = x (perfect)')
ax.axvline(lam*1e3/2, color='gray', ls=':', lw=1.2, label='lambda/2 Rayleigh')
ax.set_xlabel("True change [mm]", fontsize=10)
ax.set_ylabel("Estimated change [mm]", fontsize=10)
ax.set_title("Estimated vs true — all below Rayleigh limit detectable", fontsize=9)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which='both')

ax = axes2[1]
ax.semilogx(true_mm, [100*abs(e-t)/t for e, t in zip(est_dx_mm, true_mm)],
             'o-', color='steelblue',   lw=1.8, ms=8, label='Lateral dx')
ax.semilogx(true_mm, [100*abs(e-t)/t for e, t in zip(est_dz_mm, true_mm)],
             's-', color='darkorange',  lw=1.8, ms=8, label='Depth dz')
ax.semilogx(true_mm, [0.0]*len(true_mm),
             '^-', color='crimson',     lw=1.8, ms=8, label='Amplitude dA (exact)')
ax.axhline(10, color='gray', ls='--', lw=1.0, label='10% error threshold')
ax.axvline(lam*1e3/2, color='gray', ls=':', lw=1.2, label='lambda/2 Rayleigh')
ax.set_xlabel("True change [mm]", fontsize=10)
ax.set_ylabel("Relative error [%]", fontsize=10)
ax.set_title("Estimation error — smaller changes: first-order approx degrades\n"
             "for dx/dz, exact for amplitude", fontsize=9)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Decision table ────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("CLASSIFICATION + QUANTIFICATION DECISION TABLE")
print("="*60)
print(f"{'Test':<20}  {'Lateral dx':>12}  {'Depth dz':>10}  {'Amplitude dA':>14}")
print("-"*60)
print(f"{'AR  (anti-sym ratio)':<20}  {'HIGH (>0.5)':>12}  {'low (<0.5)':>10}  {'low (<0.5)':>14}")
print(f"{'z-Hilbert phase':<20}  {'undefined*':>12}  {'+-90 deg':>10}  {'~0 deg':>14}")
print(f"{'Temporal CLSSA theta':<20}  {'-90 (off-apex)':>12}  {'-90 (apex)':>10}  {'~0 deg':>14}")
print(f"{'Quantifier':<20}  {'LS x-gradient':>12}  {'LS z-grad':>10}  {'data_ratio':>14}")
print("-"*60)
print("* lateral: AR is HIGH -> no need to check z-Hilbert")
print()
print("Sensitivity limit: governed by SNR of B_diff, not by Rayleigh criterion.")
print(f"  At lambda/32 = {lam/32*1e3:.1f} mm: data_ratio = {results[-1]['data_ratio']:.2e}  "
      f"(signal detectable above noise floor)")


In [ ]:
# =============================================================================
# infer_timelapse_change()
# ========================
# Self-contained inference pipeline: two B-scans -> change type + magnitude.
#
# Input:
#   B_base, B_mon  (n_t, n_x) raw zero-offset B-scans
#   x [m], t [ns], z_mig [m], vel [m/ns]
#   img_base       pre-migrated baseline (optional; computed if not given)
#
# Pipeline:
#   1. Migrate baseline -> locate scatterer (amplitude peak)
#   2. Compute B_diff; data_ratio = max|B_diff| / max|B_base|
#   3. Temporal CLSSA at off-axis trace x_ref = x_sc + lam_est/2
#      -> relative phase rel_phi = theta_dom_diff - theta_dom_base at twt_ref
#      -> |rel_phi| ~ 90 deg: movement;  |rel_phi| ~ 0 deg: amplitude change
#   4. Migrate B_diff -> img_diff
#   5. Anti-symmetry ratio AR of x-profile at z_sc:
#      -> AR > 0.5: lateral;  AR < 0.5 + z-Hilbert: depth vs amplitude
#   6. PSF-gradient least-squares inversion -> magnitude
#
# Output dict:
#   change_type   'lateral' | 'depth' | 'amplitude'
#   delta         estimated magnitude [m for lateral/depth, fraction for amplitude]
#   x_sc, z_sc    scatterer position from baseline [m]
#   data_ratio    max|B_diff| / max|B_base|
#   AR            anti-symmetry ratio (0 = symmetric, 1 = pure anti-sym)
#   z_phase       z-Hilbert phase at (x_sc, z_sc) [deg]
#   rel_phi       relative temporal CLSSA phase at off-axis trace [deg]
#   img_base      migrated baseline (n_z, n_x)
#   img_diff      migrated difference (n_z, n_x)
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

import sys, os, importlib
sys.path.insert(0, os.getcwd())
import phase_decomposition as pd_mod
importlib.reload(pd_mod)


def infer_timelapse_change(
    B_base, B_mon, x, t, z_mig, vel,
    img_base=None,
    clssa_params=None,
):
    dx = float(x[1] - x[0])
    dt = float(t[1] - t[0])

    if clssa_params is None:
        clssa_params = dict(f_min_GHz=0.3, f_max_GHz=4.0,
                            n_freqs=60, win_ns=2.0, alpha=1e-2)

    # ── 1. Migrate baseline ───────────────────────────────────────────────────
    if img_base is None:
        img_base = gazdag_migration(B_base, x, t, z_mig, vel)

    # ── 2. Locate scatterer (amplitude peak of monogenic envelope) ────────────
    kx = np.fft.fftfreq(img_base.shape[1])
    kz = np.fft.fftfreq(img_base.shape[0])
    KX, KZ = np.meshgrid(kx, kz)
    K = np.sqrt(KX**2 + KZ**2);  K[0, 0] = 1.0
    R2_b  = np.real(np.fft.ifft2((-1j * KZ / K) * np.fft.fft2(img_base)))
    A_env = np.sqrt(img_base**2 + R2_b**2)
    peak_iz, peak_ix = np.unravel_index(np.argmax(A_env), A_env.shape)
    x_sc  = float(x[peak_ix])
    z_sc  = float(z_mig[peak_iz])

    # ── 3. Difference B-scan and data_ratio ───────────────────────────────────
    B_diff     = B_mon - B_base
    data_ratio = float(np.max(np.abs(B_diff))) / float(np.max(np.abs(B_base)))

    # ── 4. Temporal CLSSA classification at off-axis trace ────────────────────
    # Estimate lambda from the PSF x-profile: distance from peak to first zero
    base_prof_sc = img_base[peak_iz, :]
    zero_cross   = np.where(np.diff(np.sign(base_prof_sc - 0)))[0]
    if len(zero_cross) > 0:
        near_zero = zero_cross[np.argmin(np.abs(x[zero_cross] - x_sc))]
        lam_est   = 2.0 * abs(x[near_zero] - x_sc)   # ~lambda
    else:
        lam_est = 4.0 * dx   # fallback

    ix_off  = int(np.argmin(np.abs(x - (x_sc + 0.5 * lam_est))))
    ix_off  = np.clip(ix_off, 1, len(x) - 2)
    r_off   = np.sqrt((x[ix_off] - x_sc)**2 + z_sc**2)
    twt_off = 2.0 * r_off / vel
    it_off  = int(np.argmin(np.abs(t - twt_off)))

    def _clssa_dominant_phase(trace, it):
        """Dominant temporal phase at sample it from CLSSA broadband phasor."""
        norm   = np.abs(trace).max() + 1e-30
        A_, th_, _, f_, _, _ = pd_mod.clssa_phase_decomposition(
            trace / norm, dt, **clssa_params)
        C_ = A_ * np.exp(1j * np.deg2rad(th_))
        R_ = np.trapz(np.real(C_), f_, axis=0)
        I_ = np.trapz(np.imag(C_), f_, axis=0)
        return float(np.rad2deg(np.arctan2(I_, R_))[it])

    phi_base = _clssa_dominant_phase(B_base[:, ix_off], it_off)
    phi_diff = _clssa_dominant_phase(B_diff[:, ix_off], it_off)
    rel_phi  = phi_diff - phi_base        # ~-90 deg: movement;  ~0: amplitude

    # ── 5. Migrate difference ─────────────────────────────────────────────────
    img_diff = gazdag_migration(B_diff, x, t, z_mig, vel)

    # ── 6. Spatial classification ─────────────────────────────────────────────
    dp = img_diff[peak_iz, :]             # x-profile at z_sc
    dc = img_diff[:, peak_ix]             # z-column at x_sc

    # Anti-symmetry ratio
    half = min(peak_ix, len(dp) - peak_ix - 1)
    if half >= 3:
        left  = dp[peak_ix - half : peak_ix]
        right = dp[peak_ix : peak_ix + half][::-1]
        anti  = 0.5 * (left - right);  sym = 0.5 * (left + right)
        tot   = np.dot(anti, anti) + np.dot(sym, sym)
        AR    = float(np.dot(anti, anti) / (tot + 1e-12))
    else:
        AR = 0.5

    # z-Hilbert phase at scatterer
    z_phase = float(np.rad2deg(np.angle(hilbert(dc)[peak_iz])))

    # ── 7. Classify ───────────────────────────────────────────────────────────
    if AR > 0.5:
        change_type = 'lateral'
    elif abs(z_phase) > 45:
        change_type = 'depth'
    else:
        change_type = 'amplitude'

    # ── 8. Quantify via PSF-gradient LS inversion ─────────────────────────────
    d_dx = np.gradient(img_base[peak_iz, :],  x)
    d_dz = np.gradient(img_base[:, peak_ix], z_mig)

    def _ls(sig, grad):
        return float(-np.dot(sig, grad) / (np.dot(grad, grad) + 1e-12))

    if change_type == 'lateral':
        delta = data_ratio * _ls(dp, d_dx)
    elif change_type == 'depth':
        delta = data_ratio * _ls(dc, d_dz)
    else:
        delta = data_ratio

    return dict(change_type=change_type, delta=delta,
                x_sc=x_sc, z_sc=z_sc, data_ratio=data_ratio,
                AR=AR, z_phase=z_phase, rel_phi=rel_phi,
                img_base=img_base, img_diff=img_diff)


# =============================================================================
# Demonstration on the three synthetic scenarios (all at lambda/16)
# =============================================================================

change_mag = lam / 21.0
print(f"All scenarios: change = lambda/16 = {change_mag*1e3:.2f} mm\n")

# Collect the three difference B-scans (depth and amplitude were made in cell 23)
B_diff_lat = results[-1]['B_diff']     # lateral  delta_x = lambda/16
B_diff_dz2 = B_mon_dz                  # depth    delta_z = lambda/16  (from cell 23)
B_diff_dA2 = dA * B_base               # amplitude dA = lambda/16 (fractional)

scenarios_demo = [
    ('Lateral  dx',  B_diff_lat + B_base,  change_mag,  0,           0,           'steelblue'),
    ('Depth    dz',  B_diff_dz2,           0,           change_mag,  0,           'darkorange'),
    ('Ampl.    dA',  (1+dA)*B_base,        0,           0,           change_mag,  'crimson'),
]

infer_results = []
for name, B_mon_demo, true_dx, true_dz, true_dA, col in scenarios_demo:
    print(f"─── {name} ───")
    t0 = _time.time()
    r = infer_timelapse_change(B_mon_demo, B_mon_demo, x_b, t_b, z_mig, v_ice,
                                img_base=img_base)
    # Note: B_mon is already B_base + B_diff; pass it correctly:
    r = infer_timelapse_change(B_base, B_mon_demo, x_b, t_b, z_mig, v_ice,
                                img_base=img_base)
    elapsed = _time.time() - t0

    true_val = true_dx or true_dz or true_dA
    est_val  = abs(r['delta'])
    err_pct  = 100 * abs(est_val - true_val) / true_val if true_val else 0

    print(f"  type     : {r['change_type']}  (AR={r['AR']:.2f}, z-phase={r['z_phase']:+.0f} deg, "
          f"rel_phi={r['rel_phi']:+.0f} deg)")
    print(f"  delta    : {est_val*1e3:.4f} mm  (true={true_val*1e3:.4f} mm,  err={err_pct:.1f}%)")
    print(f"  position : x_sc={r['x_sc']*1e3:.1f} mm  z_sc={r['z_sc']*1e3:.1f} mm")
    print(f"  ({elapsed:.1f} s including migration)\n")
    infer_results.append({**r, 'name': name, 'true_val': true_val,
                           'est_val': est_val, 'col': col})

# ── Figure: overview of the three inferred differences ──────────────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
fig.suptitle(
    f"infer_timelapse_change() applied to three synthetic scenarios  (change = lambda/16 = {change_mag*1e3:.1f} mm)\n"
    "Col 1: migrated difference  |  Col 2: x-profile  |  Col 3: z-column  |  Col 4: temporal CLSSA phase gather",
    fontsize=10, fontweight='bold',
)

x_lo, x_hi = x_sc0 - 4*lam, x_sc0 + 4*lam
z_lo, z_hi = z_sc  - 0.08,   z_sc  + 0.08
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]

# Reference trace for CLSSA panel
ix_off_demo = int(np.argmin(np.abs(x_b - (x_sc0 + 0.5*lam))))
r_off_demo  = np.sqrt((x_b[ix_off_demo] - x_sc0)**2 + z_sc**2)
twt_off_d   = 2.0 * r_off_demo / v_ice
it_off_d    = int(np.argmin(np.abs(t_b - twt_off_d)))
t_lo_pg, t_hi_pg = twt_off_d - 1.2, twt_off_d + 1.2
t_mask_pg   = (t_b >= t_lo_pg) & (t_b <= t_hi_pg)
ext_pg      = [-180, 180, t_hi_pg, t_lo_pg]

for row, (ri, B_mon_d, true_dx, true_dz, true_dA, col) in enumerate(
        zip(infer_results,
            [results[-1]['B_diff'] + B_base,
             B_mon_dz, (1+dA)*B_base],
            [change_mag, 0, 0],
            [0, change_mag, 0],
            [0, 0, change_mag],
            ['steelblue', 'darkorange', 'crimson'])):

    img_d = ri['img_diff']
    clip  = np.percentile(np.abs(img_d), 98)

    # Col 0: migrated difference
    ax = axes[row, 0]
    im = ax.imshow(img_d, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    ax.plot(x_sc0, z_sc, 'g*', ms=12, zorder=5)
    ax.set_xlim(x_lo, x_hi); ax.set_ylim(z_hi, z_lo)
    ax.set_title(f"{ri['name']}\ntype={ri['change_type']},  est={ri['est_val']*1e3:.4f} mm",
                 fontsize=9)
    ax.set_xlabel("Position [m]"); ax.set_ylabel("Depth [m]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Col 1: x-profile
    ax = axes[row, 1]
    ax.plot(x_b, img_d[int(np.argmin(np.abs(z_mig-z_sc))), :], color=col, lw=1.4)
    ax.axvline(x_sc0, color='g', ls='--', lw=0.9)
    ax.axhline(0, color='k', lw=0.3)
    ax.set_xlim(x_lo, x_hi); ax.set_xlabel("Position [m]")
    ax.set_title(f"x-profile at z_sc\nAR = {ri['AR']:.3f}", fontsize=9)
    ax.grid(alpha=0.3)

    # Col 2: z-column
    ax = axes[row, 2]
    ax.plot(z_mig, img_d[:, int(np.argmin(np.abs(x_b-x_sc0)))], color=col, lw=1.4)
    ax.axvline(z_sc, color='g', ls='--', lw=0.9)
    ax.axhline(0, color='k', lw=0.3)
    ax.set_xlim(z_lo, z_hi); ax.set_xlabel("Depth [m]")
    ax.set_title(f"z-column at x_sc0\nz-phase = {ri['z_phase']:+.0f} deg", fontsize=9)
    ax.grid(alpha=0.3)

    # Col 3: temporal CLSSA phase gather of difference trace at x_off
    B_d = B_mon_d - B_base
    B_d_norm = B_d[:, ix_off_demo]
    B_d_norm = B_d_norm / (np.abs(B_d_norm).max() + 1e-30)
    A_pg, th2_pg, pg_pg, f_pg, _, tv_pg = pd_mod.clssa_phase_decomposition(
        B_d_norm, dt_b, f_min_GHz=0.3, f_max_GHz=4.0,
        n_freqs=60, win_ns=2.0, alpha=1e-2)
    ax = axes[row, 3]
    clip_pg = np.percentile(np.abs(pg_pg), 99)
    ax.imshow(pg_pg[:, t_mask_pg].T, aspect='auto', origin='upper',
              extent=ext_pg, cmap='RdBu', vmin=-clip_pg, vmax=clip_pg,
              interpolation='bilinear')
    ax.axhline(twt_off_d, color='white', lw=1.0, ls='--')
    ax.axvline(ri['rel_phi'] + float(np.rad2deg(
        np.angle(hilbert(B_base[:, ix_off_demo])[it_off_d]))),
               color='lime', lw=1.8, ls='--',
               label=f"rel_phi={ri['rel_phi']:+.0f} deg")
    ax.axvline(-90, color='cyan', lw=0.8, ls=':', alpha=0.7, label='-90 deg')
    ax.axvline(0,   color='white', lw=0.6, ls=':', alpha=0.5)
    ax.set_xlabel("Phase theta [deg]"); ax.set_ylabel("TWT [ns]")
    ax.set_title(f"Temporal phase gather at x_off\nrel_phi = {ri['rel_phi']:+.0f} deg", fontsize=9)
    ax.set_xticks([-180, -90, 0, 90, 180]); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# ── Clean summary ─────────────────────────────────────────────────────────────
print("=" * 70)
print("INFERENCE SUMMARY")
print("=" * 70)
print(f"  {'Scenario':<14}  {'Type OK':>8}  {'AR':>6}  {'z-ph':>7}  "
      f"{'rel_phi':>8}  {'true [mm]':>10}  {'est [mm]':>10}  {'err %':>7}")
print("-" * 70)
ground_truth = [('lateral', change_mag), ('depth', change_mag), ('amplitude', dA)]
for ri, (gt_type, gt_val) in zip(infer_results, ground_truth):
    ok = 'YES' if ri['change_type'] == gt_type else 'NO '
    print(f"  {ri['name']:<14}  {ok:>8}  {ri['AR']:.3f}  {ri['z_phase']:>+7.0f}  "
          f"{ri['rel_phi']:>+8.0f}  {gt_val*1e3:>10.4f}  "
          f"{ri['est_val']*1e3:>10.4f}  "
          f"{100*abs(ri['est_val']-gt_val)/gt_val:>6.1f}%")
print()
print("Key decision thresholds:")
print("  AR > 0.5              -> lateral movement")
print("  AR < 0.5, |z-ph|>45  -> depth movement")
print("  AR < 0.5, |z-ph|<45  -> amplitude change")
print("  |rel_phi| ~ 90 deg    -> any movement (cross-check on raw B-scans)")
print("  |rel_phi| ~ 0 deg     -> amplitude change (cross-check)")


In [ ]:
# =============================================================================
# Phase-filter then migrate: detectability of sub-wavelength movement
#
# Three comparisons (no added noise):
#
#   A. All 6 shift scenarios, pure lateral movement:
#      Direct M(B_diff) vs filtered M(S'(-90, B_diff))
#      -> In a pure scenario both give the same dipole (B_diff is already -90 deg)
#
#   B. Mixed scenario: lateral dx + amplitude dA at the same time
#      B_diff = B_diff_lateral + dA * B_base
#      The amplitude part has theta ~ 0 deg, the movement part theta ~ -90 deg.
#      Direct M(B_diff_mixed) -> dipole + scaled PSF overlap -> obscures movement
#      Filtered M(S'(-90, B_diff_mixed)) -> amplitude part suppressed -> clean dipole
#
#   C. Quantify: dipole-to-monopole ratio as function of contamination dA
#      Shows when phase filter matters (large dA) vs when it doesn't (dA ~ 0)
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# ── Phase filter (broadband Hilbert projection) ───────────────────────────────
def phase_project(B, theta_deg):
    """Project each trace of B-scan onto the direction theta [deg]."""
    analytic = hilbert(B, axis=0)
    th = np.deg2rad(theta_deg)
    return np.cos(th) * np.real(analytic) + np.sin(th) * np.imag(analytic)

# ── A: Pure lateral shifts — direct vs phase-filtered ────────────────────────
print("A. Pure lateral shifts: direct vs phase-filtered migration")
imgs_direct  = []
imgs_filt    = []

for r in results:
    Bd = r['B_diff']                          # already noise-free
    Bd_filt = phase_project(Bd, -90.0)        # project onto -90 deg axis

    img_d = gazdag_migration(Bd,      x_b, t_b, z_mig, v_ice)
    img_f = gazdag_migration(Bd_filt, x_b, t_b, z_mig, v_ice)
    imgs_direct.append(img_d)
    imgs_filt.append(img_f)
    print(f"  {r['label']:<8}  done")

# ── B: Mixed scenario — lateral dx + amplitude dA ────────────────────────────
dx_mix = lam / 16.0          # sub-wavelength lateral shift
dA_mix = 0.20                # 20 % amplitude change (larger than the movement signal)

print(f"\nB. Mixed: lateral dx=lam/16={dx_mix*1e3:.2f}mm  +  amplitude dA={dA_mix:.0%}")

r_lat16 = next(r for r in results if abs(r['frac'] - 1/16) < 1e-6)
B_diff_lat16 = r_lat16['B_diff']             # pure lateral
B_diff_amp   = dA_mix * B_base               # pure amplitude change (theta ~ 0)
B_diff_mixed = B_diff_lat16 + B_diff_amp     # contaminated difference

# Phase contributions in the mixed B_diff
theta_lat16 = -90.0    # lateral movement component
theta_amp   =   0.0    # amplitude change component

# Ratios
dr_lat = r_lat16['data_ratio']
dr_amp = float(np.max(np.abs(B_diff_amp))) / float(np.max(np.abs(B_base)))
print(f"  data_ratio (lateral)  : {dr_lat:.5f}")
print(f"  data_ratio (amplitude): {dr_amp:.5f}")
print(f"  Amplitude contam. : {dA_mix/dr_lat:.0f}x stronger than movement signal\n")

t0 = _time.time()
img_mixed_direct = gazdag_migration(B_diff_mixed,                      x_b, t_b, z_mig, v_ice)
img_mixed_filt   = gazdag_migration(phase_project(B_diff_mixed, -90.0),x_b, t_b, z_mig, v_ice)
img_pure_lat     = imgs_direct[-1]   # lambda/16 pure lateral (reference)
print(f"  migrations done in {_time.time()-t0:.1f} s")

# ── Figure 1: A — all scenarios side by side ──────────────────────────────────
iz_sc = np.argmin(np.abs(z_mig - z_sc))
x_zoom = (x_b >= x_sc0 - 4*lam) & (x_b <= x_sc0 + 4*lam)
x_z = x_b[x_zoom]

n_sc = len(results)
cols_sc = plt.cm.plasma(np.linspace(0.1, 0.9, n_sc))

fig, axes = plt.subplots(2, n_sc, figsize=(4*n_sc, 8), sharex=True, sharey=True)
fig.suptitle(
    "A.  Phase-filter then migrate — pure lateral shifts (no noise)\n"
    "Top: direct M(B_diff)  |  Bottom: filtered M(S'(-90, B_diff))",
    fontsize=11, fontweight='bold',
)

for col_idx, (r, img_d, img_f, col) in enumerate(zip(results, imgs_direct, imgs_filt, cols_sc)):
    for row_idx, (img, row_lbl) in enumerate([(img_d, 'direct'), (img_f, 'phase-filt')]):
        ax = axes[row_idx, col_idx]
        prof = img[iz_sc, :]
        pk   = np.abs(img).max() + 1e-30
        ax.plot(x_z, prof[x_zoom]/pk, color=col, lw=1.5)
        ax.axvline(x_sc0, color='g', ls='--', lw=0.8)
        ax.axhline(0, color='k', lw=0.3)
        ax.grid(alpha=0.3)
        if row_idx == 0:
            ax.set_title(r['label'], fontsize=10)
        if col_idx == 0:
            ax.set_ylabel(f"{row_lbl}\nNorm. amplitude", fontsize=9)
        ax.set_xlabel("Position [m]", fontsize=8)

plt.tight_layout()
plt.show()
print("  -> Both direct and filtered give identical dipoles for a PURE lateral shift.")
print("     (B_diff is already -90 deg, so projection changes nothing.)\n")

# ── Figure 2: B — mixed scenario ─────────────────────────────────────────────
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo, x_hi = x_sc0 - 4*lam, x_sc0 + 4*lam
z_lo, z_hi = z_sc  - 0.08,  z_sc  + 0.08

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
fig2.suptitle(
    f"B.  Mixed scenario: lateral dx=lam/16 + amplitude dA={dA_mix:.0%}  "
    f"(amplitude {dA_mix/dr_lat:.0f}x larger than movement)\n"
    "Phase filter isolates the movement signature from amplitude contamination",
    fontsize=11, fontweight='bold',
)

# Row 0: B-scans
B_show = [B_diff_lat16, B_diff_amp, B_diff_mixed]
titles_b = ['B_diff  lateral only\n(dx=lam/16)',
            f'B_diff  amplitude only\n(dA={dA_mix:.0%})',
            'B_diff  MIXED\n(lateral + amplitude)']
for ax, Bd, title in zip(axes2[0], B_show, titles_b):
    clip = np.percentile(np.abs(Bd), 98)
    ax.imshow(Bd, aspect='auto', origin='upper',
              extent=[x_b[0], x_b[-1], t_b[-1], t_b[0]],
              cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    r_h = np.sqrt((x_b - x_sc0)**2 + z_sc**2)
    ax.plot(x_b, 2*r_h/v_ice, 'k--', lw=0.7, alpha=0.5)
    ax.set_xlim(x_b[0], x_b[-1])
    ax.set_ylim(2*(np.sqrt((x_b[-1]-x_sc0)**2+z_sc**2))/v_ice + 0.2,
                2*(z_sc - 0.1)/v_ice)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]"); ax.set_ylabel("TWT [ns]")

# Row 1: migrated differences — profiles at z_sc
ax = axes2[1, 0]
for img, lbl, col, lw in [
        (img_pure_lat,    'lateral only (ref)', 'green',     2.0),
        (img_mixed_direct,'mixed: direct mig',  'steelblue', 1.5),
        (img_mixed_filt,  'mixed: phase-filt',  'darkorange',1.5)]:
    prof = img[iz_sc, :]
    pk   = np.abs(img).max() + 1e-30
    ax.plot(x_z, prof[x_zoom]/pk, color=col, lw=lw, label=lbl)
ax.axvline(x_sc0, color='g', ls='--', lw=0.9)
ax.axhline(0, color='k', lw=0.3)
ax.set_title("Profiles at z_sc (all normalised to their own peak)", fontsize=9)
ax.set_xlabel("Position [m]"); ax.set_ylabel("Normalised amplitude")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 2D migrated images
for ax, img, title in [
        (axes2[1, 1], img_mixed_direct,
         f'Mixed: direct M(B_diff)\nmovement hidden under amplitude PSF'),
        (axes2[1, 2], img_mixed_filt,
         f'Mixed: M(phase-filt B_diff)\namplitude suppressed, movement visible')]:
    clip = np.percentile(np.abs(img), 98)
    ax.imshow(img, aspect='auto', origin='upper', extent=ext_mig,
              cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    ax.set_xlim(x_lo, x_hi); ax.set_ylim(z_hi, z_lo)
    ax.plot(x_sc0, z_sc, 'g*', ms=12, zorder=5)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Position [m]"); ax.set_ylabel("Depth [m]")

plt.tight_layout()
plt.show()

# ── C: Dipole-to-monopole ratio vs contamination dA ─────────────────────────
print("\nC. Dipole-to-monopole ratio vs amplitude contamination dA")
dA_scan = np.linspace(0, 0.5, 15)    # 0 to 50% amplitude change
ratio_direct = []
ratio_filt   = []

for dA in dA_scan:
    Bd_mx  = B_diff_lat16 + dA * B_base
    img_dx = gazdag_migration(Bd_mx,                      x_b, t_b, z_mig, v_ice)
    img_fx = gazdag_migration(phase_project(Bd_mx, -90.0),x_b, t_b, z_mig, v_ice)

    prof_d  = img_dx[iz_sc, :]
    prof_f  = img_fx[iz_sc, :]

    # Dipole peak (anti-symmetric component power)
    half = min(ix_sc0, len(prof_d) - ix_sc0 - 1)
    left_d  = prof_d[ix_sc0 - half : ix_sc0]
    right_d = prof_d[ix_sc0 : ix_sc0 + half][::-1]
    anti_d  = 0.5 * (left_d - right_d);  sym_d = 0.5 * (left_d + right_d)

    left_f  = prof_f[ix_sc0 - half : ix_sc0]
    right_f = prof_f[ix_sc0 : ix_sc0 + half][::-1]
    anti_f  = 0.5 * (left_f - right_f);  sym_f = 0.5 * (left_f + right_f)

    ratio_direct.append(np.dot(anti_d, anti_d) / (np.dot(sym_d, sym_d) + 1e-12))
    ratio_filt.append(  np.dot(anti_f, anti_f) / (np.dot(sym_f, sym_f) + 1e-12))

    print(f"  dA={dA:.2f}  direct ratio={ratio_direct[-1]:.3f}  filt ratio={ratio_filt[-1]:.3f}")

fig3, ax3 = plt.subplots(figsize=(9, 5))
ax3.semilogy(dA_scan * 100, ratio_direct, 'ko-', lw=1.8, ms=7, label='Direct M(B_diff)')
ax3.semilogy(dA_scan * 100, ratio_filt,   'bo-', lw=1.8, ms=7, label='Phase-filt M(S\'(-90, B_diff))')
ax3.axhline(1.0, color='gray', ls=':', lw=1.0, label='AR=1 (perfect dipole)')
ax3.axvline(dA_mix * 100, color='red', ls='--', lw=1.2,
            label=f'This demo (dA={dA_mix:.0%})')
ax3.set_xlabel("Amplitude contamination dA  [% of baseline peak]", fontsize=10)
ax3.set_ylabel("Anti-symmetry ratio  (dipole power / monopole power)", fontsize=10)
ax3.set_title("C.  Phase filter maintains clean dipole even under large amplitude contamination\n"
              "Direct migration: anti-symmetry ratio degrades rapidly with dA\n"
              "Phase-filtered: ratio stays high -> movement detectable at any dA",
              fontsize=9)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("Summary:")
print("  Pure lateral shift:  phase filter = direct migration  (no difference)")
print("    -> B_diff is already purely -90 deg, projection is identity")
print()
print("  Mixed (lat + amp):   phase filter >> direct migration")
print(f"    -> At dA={dA_mix:.0%}: direct AR={ratio_direct[int(dA_mix/0.5*14)]:.3f},  "
      f"filtered AR={ratio_filt[int(dA_mix/0.5*14)]:.3f}")
print()
print("  Rule: phase filtering before migration is valuable when:")
print("    1. Multiple change types occur simultaneously (separation of effects)")
print("    2. One type dominates amplitude but not the target phase angle")
print("    3. In-phase artefacts (static errors, coupling drift) contaminate B_diff")
print()
print("  It is NOT needed when B_diff contains only the target change type.")


# Phase Informed Timelapse Migration: Horizontal Movement

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import pywt
import numpy as np
import matplotlib.pyplot as plt

import sys, os, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import phase_decomposition as pd_mod
importlib.reload(pd_mod)   # pick up any edits without kernel restart

import numpy as np
import matplotlib.pyplot as plt
import time as _time

import os, time as _time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from gprMax.gprMax import api
from tools.outputfiles_merge import merge_files
from tools.plot_Bscan import get_output_data, mpl_plot
from matplotlib.colors import TwoSlopeNorm

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

import pylops
import time as _time

In [ ]:
# -- Acquisition --------------------------------------------------------------
dx_b   = 0.005 / 4
n_x_b  = 160 * 4
x_b    = np.arange(n_x_b) * dx_b       # 0 -- 0.795 m profile
x_sc0  = x_b.mean()

dt_b   = 0.004
n_t_b  = 1300                           # t_max = 5.196 ns > TWT(0.30 m) = 3.57 ns
t_b    = np.arange(n_t_b) * dt_b

z_sc   = 0.30
z_mig  = np.arange(0, 0.42, dx_b)      # 84 depth steps

v_ice  = 0.168        # m/ns
f_c    = 1.5          # GHz
lam    = v_ice / f_c  # wavelength in ice [m]


In [ ]:
# -- Vectorised point-scatterer B-scan ----------------------------------------
def point_bscan(x_sc, z_sc, x_rec, t_arr, v, fc):
    r   = np.sqrt((x_rec - x_sc)**2 + z_sc**2)
    twt = 2.0 * r / v
    amp = 1.0 / np.sqrt(r)
    u   = np.pi * fc * (t_arr[:, None] - twt[None, :])
    return amp[None, :] * (1.0 - 2*u**2) * np.exp(-u**2)

In [ ]:
# -- Z-direction Riesz transform ----------------------------------------------
def riesz_z(W):
    nz, nx = W.shape
    KX, KZ = np.meshgrid(np.fft.fftfreq(nx), np.fft.fftfreq(nz))
    K = np.sqrt(KX**2 + KZ**2);  K[0, 0] = 1.0
    return np.real(np.fft.ifft2((-1j * KZ / K) * np.fft.fft2(W)))


In [ ]:
# =============================================================================
# Phase Informed Timelapse Migration  —  variable lateral shift
#
# 1. Synthesise baseline and monitor B-scans  (shift = shift_frac * lam)
# 2. Migrate both with Gazdag phase-shift
# 3. Migrated difference  = baseline - monitor
# 4. Instantaneous amplitude & phase of each migrated image (Riesz-z)
# 5. Amplitude-masked phase images
# =============================================================================

# -- Parameters (carried from earlier cells) ----------------------------------
# v_ice, f_c, lam, x_b, t_b, z_mig, z_sc, x_sc0

shift_frac  = 1/4          # shift as a fraction of wavelength  <-- tune here
dx_shift    = shift_frac * lam

x_sc_mon_pi = x_sc0 + dx_shift   # monitor scatterer position

print(f"lambda      = {lam*1e3:.2f} mm")
print(f"shift       = {shift_frac} x lambda = {dx_shift*1e3:.2f} mm  ({dx_shift/dx_b:.2f} samples)")
print(f"Baseline:   x = {x_sc0*1e3:.1f} mm,  z = {z_sc*1e3:.0f} mm")
print(f"Monitor:    x = {x_sc_mon_pi*1e3:.1f} mm,  z = {z_sc*1e3:.0f} mm")

# -- 1. Synthesise B-scans ----------------------------------------------------
print("\nSynthesising B-scans...")
B_pi_base = point_bscan(x_sc0,       z_sc, x_b, t_b, v_ice, f_c)   # (n_t, n_x)
B_pi_mon  = point_bscan(x_sc_mon_pi, z_sc, x_b, t_b, v_ice, f_c)

# -- 2. Migrate both ----------------------------------------------------------
print("\nMigrating baseline...")
t0 = _time.time()
mig_pi_base = gazdag_migration(B_pi_base, x_b, t_b, z_mig, v_ice)   # (n_z, n_x)
print(f"  done in {_time.time()-t0:.1f} s")

print("\nMigrating monitor...")
t0 = _time.time()
mig_pi_mon  = gazdag_migration(B_pi_mon,  x_b, t_b, z_mig, v_ice)
print(f"  done in {_time.time()-t0:.1f} s")

# -- 3. Migrated difference  (baseline - monitor) -----------------------------
mig_pi_diff = mig_pi_base - mig_pi_mon

# -- 4. Instantaneous amplitude & phase via 2-D Riesz in z -------------------
R_base = riesz_z(mig_pi_base)
R_mon  = riesz_z(mig_pi_mon)

A_base = np.sqrt(mig_pi_base**2 + R_base**2)
A_mon  = np.sqrt(mig_pi_mon**2  + R_mon**2)

phi_base  = np.arctan2(R_base, mig_pi_base)    # [-pi, pi]
phi_mon   = np.arctan2(R_mon,  mig_pi_mon)     # [-pi, pi]
phi_delta = np.angle(np.exp(1j * (phi_mon - phi_base)))   # wrapped difference

# -- 5. Amplitude-masked phase images -----------------------------------------
thresh_frac = 0.08   # mask pixels where amplitude < 10 % of peak

mask_base  = A_base < thresh_frac * A_base.max()
mask_mon   = A_mon  < thresh_frac * A_mon.max()
mask_delta = mask_base | mask_mon      # require good signal in both images

phi_base_msk  = np.where(mask_base,  np.nan, phi_base)
phi_mon_msk   = np.where(mask_mon,   np.nan, phi_mon)
phi_delta_msk = np.where(mask_delta, np.nan, phi_delta)

print(f"\nAmplitude mask  (thresh = {thresh_frac*100:.0f}% of peak):")
print(f"  baseline masked: {mask_base.mean()*100:.1f}%  of pixels")
print(f"  monitor  masked: {mask_mon.mean()*100:.1f}%  of pixels")
print(f"  delta    masked: {mask_delta.mean()*100:.1f}%  of pixels")

# -- 6. QC display  (4 x 3) --------------------------------------------------
ext_b   = [x_b[0], x_b[-1], t_b[-1], t_b[0]]
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]

x_mid = 0.5 * (x_sc0 + x_sc_mon_pi)
x_lo  = x_mid - 5 * lam;  x_hi = x_mid + 5 * lam
z_lo  = z_sc  - 0.08;     z_hi = z_sc  + 0.08

def mig_zoom(ax):
    ax.set_xlim(x_lo, x_hi);  ax.set_ylim(z_hi, z_lo)
    ax.plot(x_sc0,         z_sc, 'g*', ms=12, zorder=7, label='baseline')
    ax.plot(x_sc_mon_pi,   z_sc, 'r^', ms=10, zorder=7, label=f'monitor ({shift_frac}λ)')

clip_b    = np.percentile(np.abs(B_pi_base), 99)
clip_diff = np.percentile(np.abs(mig_pi_diff), 99)

fig, axes = plt.subplots(4, 3, figsize=(18, 18))
fig.suptitle(
    f"Phase Informed Timelapse Migration — Lateral shift = {shift_frac}λ"
    f"  ({dx_shift*1e3:.1f} mm = {dx_shift/dx_b:.1f} samples)",
    fontsize=13, fontweight='bold',
)

# Row 0: B-scans (both + difference) -----------------------------------------
for ax, data, title in [
    (axes[0, 0], B_pi_base, "Baseline B-scan"),
    (axes[0, 1], B_pi_mon,  f"Monitor B-scan  (shift {shift_frac}λ = {dx_shift*1e3:.1f} mm)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_b,
                   cmap='RdBu', vmin=-clip_b, vmax=clip_b, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("TWT [ns]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

B_pi_diff = B_pi_mon - B_pi_base
clip_bd   = np.percentile(np.abs(B_pi_diff), 99)
im = axes[0, 2].imshow(B_pi_diff, aspect='auto', origin='upper', extent=ext_b,
                        cmap='RdBu', vmin=-clip_bd, vmax=clip_bd, interpolation='bilinear')
axes[0, 2].set_title("Difference B-scan  (monitor - baseline)", fontsize=10)
axes[0, 2].set_xlabel("Position [m]");  axes[0, 2].set_ylabel("TWT [ns]")
fig.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

# Row 1: Migrated images + migrated difference --------------------------------
for ax, data, clip, title in [
    (axes[1, 0], mig_pi_base, 1.0,       "Migrated baseline"),
    (axes[1, 1], mig_pi_mon,  1.0,       f"Migrated monitor  (shift {shift_frac}λ)"),
    (axes[1, 2], mig_pi_diff, clip_diff, "Migrated difference  (baseline - monitor)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 2: Instantaneous phase (unmasked) ---------------------------------------
for ax, data, title in [
    (axes[2, 0], phi_base,  r"Inst. phase  $\varphi_{base}$  of migrated baseline"),
    (axes[2, 1], phi_mon,   r"Inst. phase  $\varphi_{mon}$  of migrated monitor"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

im_d = axes[2, 2].imshow(phi_delta, aspect='auto', origin='upper', extent=ext_mig,
                          cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='bilinear')
axes[2, 2].set_title(
    r"Phase diff  $\Delta\varphi = \varphi_{mon} - \varphi_{base}$  (wrapped)",
    fontsize=10,
)
axes[2, 2].set_xlabel("Position [m]");  axes[2, 2].set_ylabel("Depth [m]")
mig_zoom(axes[2, 2]);  axes[2, 2].legend(fontsize=8)
fig.colorbar(im_d, ax=axes[2, 2], fraction=0.046, pad=0.04, label='[rad]')

# Row 3: Amplitude-masked phase -----------------------------------------------
hsv_cmap = plt.get_cmap('hsv').copy()
hsv_cmap.set_bad(color='lightgrey')    # NaN pixels rendered as grey

for ax, data, title in [
    (axes[3, 0], phi_base_msk,
     r"$\varphi_{base}$  masked  (|A| < "f"{thresh_frac*100:.0f}% peak)"),
    (axes[3, 1], phi_mon_msk,
     r"$\varphi_{mon}$  masked  (|A| < "f"{thresh_frac*100:.0f}% peak)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap=hsv_cmap, vmin=-np.pi, vmax=np.pi, interpolation='none')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

im_dm = axes[3, 2].imshow(phi_delta_msk, aspect='auto', origin='upper', extent=ext_mig,
                           cmap=hsv_cmap, vmin=-np.pi, vmax=np.pi, interpolation='none')
axes[3, 2].set_title(
    r"$\Delta\varphi$  masked  (either |A| < "f"{thresh_frac*100:.0f}% peak)",
    fontsize=10,
)
axes[3, 2].set_xlabel("Position [m]");  axes[3, 2].set_ylabel("Depth [m]")
mig_zoom(axes[3, 2]);  axes[3, 2].legend(fontsize=8)
fig.colorbar(im_dm, ax=axes[3, 2], fraction=0.046, pad=0.04, label='[rad]')

plt.tight_layout()
plt.show()

# -- Summary ------------------------------------------------------------------
print(f"\nlambda = {lam*1e3:.2f} mm  |  shift = {shift_frac} * lambda = {dx_shift*1e3:.2f} mm  ({dx_shift/dx_b:.1f} samples)")
print(f"Peak  |mig_base| = {np.abs(mig_pi_base).max():.4f}   "
      f"|mig_mon| = {np.abs(mig_pi_mon).max():.4f}   "
      f"|mig_diff| = {np.abs(mig_pi_diff).max():.4f}")
print(f"Phase delta range (masked): [{np.nanmin(phi_delta_msk):.3f}, {np.nanmax(phi_delta_msk):.3f}] rad")


In [ ]:
# =============================================================================
# Phase cross-section at scatterer depth: phase vs lateral position
# Shows how instantaneous phase changes between baseline and monitor
# =============================================================================

iz_sc = np.argmin(np.abs(z_mig - z_sc))   # depth index of scatterer

# 1-D profiles at z = z_sc
prof_A_base       = A_base[iz_sc, :]
prof_A_mon        = A_mon[iz_sc, :]
prof_phi_base     = phi_base[iz_sc, :]
prof_phi_mon      = phi_mon[iz_sc, :]
prof_phi_delta    = phi_delta[iz_sc, :]
prof_phi_base_msk = phi_base_msk[iz_sc, :]
prof_phi_mon_msk  = phi_mon_msk[iz_sc, :]
prof_phi_delta_msk= phi_delta_msk[iz_sc, :]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle(
    f"Phase cross-section at z = {z_sc*1e3:.0f} mm  (scatterer depth)\n"
    f"Baseline vs monitor  (shift = {shift_frac}λ = {dx_shift*1e3:.1f} mm)",
    fontsize=12, fontweight='bold',
)

# ---- Panel 0: amplitude at z_sc  (context for where the mask cuts) ---------
axes[0].plot(x_b, prof_A_base, 'b-',  lw=1.8, label='baseline')
axes[0].plot(x_b, prof_A_mon,  'r--', lw=1.8, label=f'monitor ({shift_frac}λ)')
axes[0].axhline(thresh_frac * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7,
                label=f"base threshold  ({thresh_frac*100:.0f}% of peak)")
axes[0].axhline(thresh_frac * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7,
                label=f"mon  threshold  ({thresh_frac*100:.0f}% of peak)")
axes[0].axvline(x_sc0,       color='g', ls='--', lw=1.2,
                label=f"baseline pos  x = {x_sc0*1e3:.1f} mm")
axes[0].axvline(x_sc_mon_pi, color='r', ls=':',  lw=1.2,
                label=f"monitor pos   x = {x_sc_mon_pi*1e3:.1f} mm")
axes[0].set_ylabel("Instantaneous amplitude [a.u.]", fontsize=10)
axes[0].legend(fontsize=8, ncol=2, loc='upper right')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(x_lo, x_hi)

# ---- Panel 1: phase profiles at z_sc  (faded full + solid masked) ----------
# faded background: full unmasked phase
axes[1].plot(x_b, prof_phi_base,  color='b', lw=0.9, alpha=0.25)
axes[1].plot(x_b, prof_phi_mon,   color='r', lw=0.9, alpha=0.25)
axes[1].plot(x_b, prof_phi_delta, color='k', lw=0.9, alpha=0.25)
# solid foreground: amplitude-masked phase
axes[1].plot(x_b, prof_phi_base_msk,  color='b', lw=2.2, label=r"$\varphi_{base}$")
axes[1].plot(x_b, prof_phi_mon_msk,   color='r', lw=2.2, ls='--', label=f"$\\varphi_{{mon}}$  (shift {shift_frac}λ)")
axes[1].plot(x_b, prof_phi_delta_msk, color='k', lw=2.2, label=r"$\Delta\varphi = \varphi_{mon} - \varphi_{base}$  (wrapped)")
axes[1].axvline(x_sc0,       color='g', ls='--', lw=1.2)
axes[1].axvline(x_sc_mon_pi, color='r', ls=':',  lw=1.2)
axes[1].axhline(0, color='k', lw=0.5, alpha=0.5)
axes[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
axes[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'], fontsize=10)
axes[1].set_ylabel("Instantaneous phase [rad]", fontsize=10)
axes[1].set_xlabel("Position [m]", fontsize=10)
axes[1].legend(fontsize=9, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print phase values at scatterer positions
ix_base = np.argmin(np.abs(x_b - x_sc0))
ix_mon  = np.argmin(np.abs(x_b - x_sc_mon_pi))
print(f"At baseline scatterer (x = {x_sc0*1e3:.1f} mm):")
print(f"  phi_base  = {prof_phi_base[ix_base]:.3f} rad")
print(f"  phi_mon   = {prof_phi_mon[ix_base]:.3f} rad")
print(f"  delta_phi = {prof_phi_delta[ix_base]:.3f} rad")
print(f"At monitor scatterer (x = {x_sc_mon_pi*1e3:.1f} mm):")
print(f"  phi_base  = {prof_phi_base[ix_mon]:.3f} rad")
print(f"  phi_mon   = {prof_phi_mon[ix_mon]:.3f} rad")
print(f"  delta_phi = {prof_phi_delta[ix_mon]:.3f} rad")

# ── Zoomed window around scatterer positions ──────────────────────────────────
x_win_lo = min(x_sc0, x_sc_mon_pi) - 1/2 * lam
x_win_hi = max(x_sc0, x_sc_mon_pi) + 1/2 * lam

fig2, axes2 = plt.subplots(2, 1, figsize=(10, 18), sharex=True)
fig2.suptitle(
    f"Phase cross-section — zoomed window  (\u00b12\u03bb around scatterers)\n"
    f"shift = {shift_frac}\u03bb = {dx_shift*1e3:.1f} mm",
    fontsize=12, fontweight='bold',
)

# Panel 0: amplitude zoom
axes2[0].plot(x_b, prof_A_base, 'b-',  lw=1.8, label='baseline')
axes2[0].plot(x_b, prof_A_mon,  'r--', lw=1.8, label=f'monitor ({shift_frac}\u03bb)')
axes2[0].axhline(thresh_frac * A_base.max(), color='b', ls=':', lw=1.0, alpha=0.7)
axes2[0].axhline(thresh_frac * A_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7)
axes2[0].axvline(x_sc0,       color='g', ls='--', lw=1.5,
                 label=f'baseline  x = {x_sc0*1e3:.1f} mm')
axes2[0].axvline(x_sc_mon_pi, color='r', ls=':',  lw=1.5,
                 label=f'monitor   x = {x_sc_mon_pi*1e3:.1f} mm')
axes2[0].set_xlim(x_win_lo, x_win_hi)
axes2[0].set_ylabel("Instantaneous amplitude [a.u.]", fontsize=10)
axes2[0].legend(fontsize=9)
axes2[0].grid(True, alpha=0.3)

# Panel 1: phase zoom
axes2[1].plot(x_b, prof_phi_base,  color='b', lw=0.9, alpha=0.25)
axes2[1].plot(x_b, prof_phi_mon,   color='r', lw=0.9, alpha=0.25)
axes2[1].plot(x_b, prof_phi_delta, color='k', lw=0.9, alpha=0.25)
axes2[1].plot(x_b, prof_phi_base_msk,  color='b', lw=2.5,
              label=r"$\varphi_{base}$")
axes2[1].plot(x_b, prof_phi_mon_msk,   color='r', lw=2.5, ls='--',
              label=f"$\\varphi_{{mon}}$  (shift {shift_frac}\u03bb)")
axes2[1].plot(x_b, prof_phi_delta_msk, color='k', lw=2.5,
              label=r"$\Delta\varphi$  (wrapped)")
axes2[1].axvline(x_sc0,       color='g', ls='--', lw=1.5)
axes2[1].axvline(x_sc_mon_pi, color='r', ls=':',  lw=1.5)
axes2[1].axhline(0, color='k', lw=0.5, alpha=0.5)
axes2[1].set_xlim(x_win_lo, x_win_hi)
axes2[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
axes2[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes2[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'],
                         fontsize=10)

# annotate phase values at the two scatterer positions
for ix, x_pos, col in [(ix_base, x_sc0, 'g'), (ix_mon, x_sc_mon_pi, 'darkred')]:
    for phi_arr, marker, lbl in [
        (prof_phi_base_msk, 'o', r'$\varphi_{base}$'),
        (prof_phi_mon_msk,  's', r'$\varphi_{mon}$'),
    ]:
        val = phi_arr[ix]
        if not np.isnan(val):
            axes2[1].plot(x_pos, val, marker, color=col, ms=8, zorder=6)
            axes2[1].annotate(
                f'{val:.2f} rad',
                xy=(x_pos, val), xytext=(6, 6), textcoords='offset points',
                fontsize=8, color=col,
                arrowprops=dict(arrowstyle='->', color=col, lw=0.8),
            )

axes2[1].set_ylabel("Instantaneous phase [rad]", fontsize=10)
axes2[1].set_xlabel("Position [m]", fontsize=10)
axes2[1].legend(fontsize=9, loc='upper right')
axes2[1].grid(True, alpha=0.3)
# ── Zero crossing of Delta-phi in the zoom window ────────────────────────────
# Find positions where the masked phase difference crosses zero, using linear
# interpolation between the two bracketing samples for sub-sample accuracy.
mask_win  = (x_b >= x_win_lo) & (x_b <= x_win_hi)
x_win     = x_b[mask_win]
delta_win = prof_phi_delta_msk[mask_win]

# Keep only non-NaN pairs; collect (x_zero, slope) per crossing
x_zc     = []
slope_zc = []
for i in range(len(delta_win) - 1):
    a, b = delta_win[i], delta_win[i + 1]
    if np.isnan(a) or np.isnan(b):
        continue
    if a * b < 0:                          # sign change -> zero crossing
        t = a / (a - b)                    # linear interpolation weight
        x_zc.append(x_win[i] + t * (x_win[i + 1] - x_win[i]))
        slope_zc.append((b - a) / (x_win[i + 1] - x_win[i]))  # rad / m

if x_zc:
    dx_tan = lam                           # half-length of tangent segment
    for xz, sl in zip(x_zc, slope_zc):
        axes2[0].axvline(xz, color='purple', ls='-', lw=2.0, alpha=0.8,
                         label=f'\u0394\u03c6 zero crossing  x = {xz*1e3:.2f} mm')
        axes2[1].axvline(xz, color='purple', ls='-', lw=2.0, alpha=0.8,
                         label=f'\u0394\u03c6 = 0  at x = {xz*1e3:.2f} mm')
        # tangent line: Delta-phi = sl * (x - xz)
        x_tan = np.array([xz - dx_tan, xz + dx_tan])
        axes2[1].plot(x_tan, sl * (x_tan - xz), color='orange', lw=1.8,
                      ls='--', label=f'slope = {sl:.1f} rad/m')
        axes2[1].annotate(
            f'slope = {sl:.1f} rad/m',
            xy=(xz, 0), xytext=(8, 18), textcoords='offset points',
            fontsize=9, color='darkorange', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.0),
        )
    axes2[0].legend(fontsize=9)
    axes2[1].legend(fontsize=9, loc='upper right')
    print(f"Delta-phi zero crossing(s) in zoom window:")
    for xz, sl in zip(x_zc, slope_zc):
        print(f"  x_zero = {xz*1e3:.3f} mm   slope = {sl:.2f} rad/m  "
              f"({sl * lam:.3f} rad/lambda)")
        print(f"  baseline at {x_sc0*1e3:.3f} mm,  monitor at {x_sc_mon_pi*1e3:.3f} mm")
else:
    print("No zero crossing of Delta-phi found in zoom window.")

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Phase-gradient enhanced timelapse image
#
# Key insight (from cross-section cell):
#   slope of d(Delta-phi)/dx at the zero crossing  proportional to  shift
#
# Consequence: the 2-D spatial gradient of Delta-phi
#   - peaks at the scatterer position  (vs. a dipole in the difference image)
#   - has amplitude proportional to the shift magnitude
#   - sign encodes direction of movement
#
# Circular gradient is used (via the complex exponential) to avoid wrap
# artefacts at the +/-pi boundary.
# =============================================================================

# -- 2-D circular gradient of phi_delta along x --------------------------------
# d(phi)/dx via Im[conj(e^{i*phi}) * d(e^{i*phi})/dx] -- wrap-safe
exp_phi      = np.exp(1j * phi_delta)
dexp_dx      = np.gradient(exp_phi, x_b, axis=1)   # complex gradient
dphi_dx_full = np.imag(np.conj(exp_phi) * dexp_dx) # rad / m

dphi_dx_msk  = np.where(mask_delta, np.nan, dphi_dx_full)

# -- Cross-sections at z = z_sc -----------------------------------------------
iz_sc    = np.argmin(np.abs(z_mig - z_sc))
sec_diff = mig_pi_diff[iz_sc, :]
sec_dphi = dphi_dx_msk[iz_sc, :]

# -- Figure: 2 x 2 comparison -------------------------------------------------
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]

x_mid2 = 0.5 * (x_sc0 + x_sc_mon_pi)
x_lo2  = x_mid2 - 5 * lam;  x_hi2 = x_mid2 + 5 * lam
z_lo2  = z_sc - 0.08;       z_hi2 = z_sc + 0.08

def zoom_enh(ax):
    ax.set_xlim(x_lo2, x_hi2);  ax.set_ylim(z_hi2, z_lo2)
    ax.plot(x_sc0,         z_sc, 'g*', ms=12, zorder=7, label='baseline')
    ax.plot(x_sc_mon_pi,   z_sc, 'r^', ms=10, zorder=7, label=f'monitor ({shift_frac}\u03bb)')

clip_diff2 = np.percentile(np.abs(mig_pi_diff), 99)
clip_dphi  = np.nanpercentile(np.abs(dphi_dx_msk), 99)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Phase-gradient enhanced image  \u2014  shift = {shift_frac}\u03bb = {dx_shift*1e3:.1f} mm\n"
    r"$\partial(\Delta\varphi)/\partial x$ converts dipole \u2192 monopole",
    fontsize=12, fontweight='bold',
)

# (0,0): migrated difference (dipole)
im0 = axes[0, 0].imshow(mig_pi_diff, aspect='auto', origin='upper', extent=ext_mig,
                         cmap='RdBu', vmin=-clip_diff2, vmax=clip_diff2,
                         interpolation='bilinear')
axes[0, 0].set_title("Migrated difference  (dipole)", fontsize=11)
axes[0, 0].set_xlabel("Position [m]");  axes[0, 0].set_ylabel("Depth [m]")
zoom_enh(axes[0, 0]);  axes[0, 0].legend(fontsize=8)
fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

# (0,1): phase gradient image (monopole)
im1 = axes[0, 1].imshow(dphi_dx_msk, aspect='auto', origin='upper', extent=ext_mig,
                         cmap='RdBu', vmin=-clip_dphi, vmax=clip_dphi,
                         interpolation='none')
axes[0, 1].set_title(r"Phase gradient  $\partial(\Delta\varphi)/\partial x$  (monopole)",
                     fontsize=11)
axes[0, 1].set_xlabel("Position [m]");  axes[0, 1].set_ylabel("Depth [m]")
zoom_enh(axes[0, 1]);  axes[0, 1].legend(fontsize=8)
fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04, label='rad / m')

# (1,0): cross-section of migrated difference
axes[1, 0].plot(x_b, sec_diff, 'b-', lw=2.0, label='mig diff')
axes[1, 0].axvline(x_sc0,       color='g', ls='--', lw=1.2, label='baseline pos')
axes[1, 0].axvline(x_sc_mon_pi, color='r', ls=':',  lw=1.2, label='monitor pos')
axes[1, 0].axhline(0, color='k', lw=0.5)
axes[1, 0].set_xlim(x_lo2, x_hi2)
axes[1, 0].set_xlabel("Position [m]");  axes[1, 0].set_ylabel("Amplitude [a.u.]")
axes[1, 0].set_title(f"Cross-section z = {z_sc*1e3:.0f} mm  \u2014  difference image", fontsize=10)
axes[1, 0].legend(fontsize=8);  axes[1, 0].grid(True, alpha=0.3)

# (1,1): cross-section of phase gradient
axes[1, 1].plot(x_b, sec_dphi, 'r-', lw=2.0, label=r'$\partial(\Delta\varphi)/\partial x$')
axes[1, 1].axvline(x_sc0,       color='g', ls='--', lw=1.2, label='baseline pos')
axes[1, 1].axvline(x_sc_mon_pi, color='r', ls=':',  lw=1.2, label='monitor pos')
axes[1, 1].axhline(0, color='k', lw=0.5)
axes[1, 1].set_xlim(x_lo2, x_hi2)
axes[1, 1].set_xlabel("Position [m]");  axes[1, 1].set_ylabel("rad / m")
axes[1, 1].set_title(r"Cross-section z = {:.0f} mm  \u2014  phase gradient".format(z_sc*1e3),
                     fontsize=10)
axes[1, 1].legend(fontsize=8);  axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -- Summary -------------------------------------------------------------------
peak_diff2  = np.abs(sec_diff).max()
peak_dphi_s = np.nanmax(np.abs(sec_dphi))
ix_dphi_pk  = np.nanargmax(np.abs(sec_dphi))
print(f"shift = {shift_frac}\u03bb = {dx_shift*1e3:.2f} mm")
print(f"  Peak |mig_diff|     at z_sc = {peak_diff2:.4f}  (dipole amplitude)")
print(f"  Peak |d(\u0394\u03c6)/dx| at z_sc = {peak_dphi_s:.2f} rad/m  (gradient peak)")
print(f"  Gradient peak position       = {x_b[ix_dphi_pk]*1e3:.2f} mm"
      f"  (baseline at {x_sc0*1e3:.2f} mm)")


In [ ]:
# =============================================================================
# Phase-informed position recovery: baseline + monitor from difference alone
#
# Corrected geometry:
#   For a symmetric PSF,  diff = PSF(x - x0) - PSF(x - x0 - delta)
#   is zero exactly at the MIDPOINT  x_mid = x0 + delta/2.
#
# Key chain:
#   Δφ zero crossing  →  midpoint between the two scatterers
#   slope at zero crossing  ∝  delta  →  half-separation = delta/2
#   baseline  = x_mid - delta/2
#   monitor   = x_mid + delta/2
#
# Sharpening (small-shift Taylor expansion):
#   diff(x) ≈ −delta · ∂PSF/∂x
#   ⟹  −∫diff dx / delta  ≈  PSF_baseline   (monopole, not dipole)
#   monitor image = baseline shifted by delta
# =============================================================================

if not x_zc:
    raise RuntimeError("No zero crossing found — run the cross-section cell first.")

iz_sc = np.argmin(np.abs(z_mig - z_sc))

# ── Step 1: midpoint and shift from zero crossing + slope ────────────────────
x_mid_est = x_zc[0]                           # zero crossing = midpoint

k_calib      = slope_zc[0] / dx_shift         # rad/m²  (from calibration survey)
dx_shift_est = slope_zc[0] / k_calib          # estimated lateral shift

x_base_est = x_mid_est - dx_shift_est / 2     # baseline = midpoint - half-shift
x_mon_est  = x_mid_est + dx_shift_est / 2     # monitor  = midpoint + half-shift

x_mid_true = (x_sc0 + x_sc_mon_pi) / 2

print("Phase-informed position recovery from difference image:")
print(f"  Midpoint : {x_mid_est*1e3:.3f} mm"
      f"  (true: {x_mid_true*1e3:.3f} mm,"
      f"  err: {abs(x_mid_est - x_mid_true)*1e6:.0f} \u00b5m)")
print(f"  dx_shift : {dx_shift_est*1e3:.3f} mm"
      f"  (true: {dx_shift*1e3:.3f} mm,"
      f"  err: {abs(dx_shift_est - dx_shift)*1e6:.0f} \u00b5m)")
print(f"  Baseline : {x_base_est*1e3:.3f} mm"
      f"  (true: {x_sc0*1e3:.3f} mm,"
      f"  err: {abs(x_base_est - x_sc0)*1e6:.0f} \u00b5m)")
print(f"  Monitor  : {x_mon_est*1e3:.3f} mm"
      f"  (true: {x_sc_mon_pi*1e3:.3f} mm,"
      f"  err: {abs(x_mon_est - x_sc_mon_pi)*1e6:.0f} \u00b5m)")

# ── Step 2: integration sharpening ───────────────────────────────────────────
# diff ≈ −delta · ∂PSF/∂x  ⟹  −∫diff dx / delta  ≈  PSF_baseline
mig_diff_int = np.cumsum(mig_pi_diff, axis=1) * dx_b
img_base_rec = -mig_diff_int / dx_shift_est            # recovered baseline (monopole)

# Monitor: shift baseline by delta (sub-sample via scipy interpolation)
from scipy.ndimage import shift as nd_shift
img_mon_rec = nd_shift(img_base_rec, (0, dx_shift_est / dx_b), mode='nearest')

pk_b = np.abs(img_base_rec).max() + 1e-30
pk_m = np.abs(img_mon_rec).max()  + 1e-30

# ── Figure: 2 × 3 ────────────────────────────────────────────────────────────
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo3 = x_mid_est - 5 * lam;  x_hi3 = x_mid_est + 5 * lam
z_lo3 = z_sc - 0.08;           z_hi3 = z_sc + 0.08

def zoom_loc(ax):
    ax.set_xlim(x_lo3, x_hi3);  ax.set_ylim(z_hi3, z_lo3)
    ax.axvline(x_mid_est,  color='purple', ls='-',  lw=1.5,
               label=f'\u0394\u03c6 = 0  (midpoint  {x_mid_est*1e3:.2f} mm)')
    ax.axvline(x_base_est, color='g',      ls='--', lw=2.0,
               label=f'base est  {x_base_est*1e3:.2f} mm')
    ax.axvline(x_mon_est,  color='darkred', ls=':',  lw=2.0,
               label=f'mon est   {x_mon_est*1e3:.2f} mm')

clip_d = np.percentile(np.abs(mig_pi_diff), 99)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"Phase-informed position recovery  \u2014  shift = {shift_frac}\u03bb = {dx_shift*1e3:.1f} mm\n"
    "Zero crossing = midpoint; slope gives half-separation",
    fontsize=12, fontweight='bold',
)

# Row 0: migrated difference | recovered baseline | recovered monitor ----------
im0 = axes[0, 0].imshow(mig_pi_diff, aspect='auto', origin='upper', extent=ext_mig,
                         cmap='RdBu', vmin=-clip_d, vmax=clip_d, interpolation='bilinear')
axes[0, 0].set_title("Migrated difference (dipole, input)", fontsize=10)
axes[0, 0].set_xlabel("Position [m]");  axes[0, 0].set_ylabel("Depth [m]")
zoom_loc(axes[0, 0]);  axes[0, 0].legend(fontsize=7)
fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

for ax, img, pk, title in [
    (axes[0, 1], img_base_rec, pk_b, "Recovered baseline  (\u2212\u222bdiff\u00b7dx / \u03b4)"),
    (axes[0, 2], img_mon_rec,  pk_m, "Recovered monitor  (baseline shifted by \u03b4)"),
]:
    im = ax.imshow(img / pk, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='RdBu', vmin=-1, vmax=1, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    zoom_loc(ax);  ax.legend(fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 1: cross-sections --------------------------------------------------------
sec_d = mig_pi_diff[iz_sc, :] / (np.abs(mig_pi_diff[iz_sc, :]).max() + 1e-30)
sec_b = img_base_rec[iz_sc, :] / pk_b
sec_m = img_mon_rec[iz_sc, :]  / pk_m

# (1,0): dipole vs both recovered monopoles
axes[1, 0].plot(x_b, sec_d, color='steelblue', lw=1.5, alpha=0.5, label="diff (dipole, norm.)")
axes[1, 0].plot(x_b, sec_b, color='g',         lw=2.2, label="baseline recovered")
axes[1, 0].plot(x_b, sec_m, color='darkred',   lw=2.2, ls='--', label="monitor recovered")
axes[1, 0].axvline(x_mid_est,  color='purple',  ls='-',  lw=1.2, label='midpoint')
axes[1, 0].axvline(x_base_est, color='g',       ls='--', lw=1.2)
axes[1, 0].axvline(x_mon_est,  color='darkred', ls=':',  lw=1.2)
axes[1, 0].axhline(0, color='k', lw=0.5)
axes[1, 0].set_xlim(x_lo3, x_hi3);  axes[1, 0].set_ylim(-1.4, 1.4)
axes[1, 0].set_xlabel("Position [m]");  axes[1, 0].set_ylabel("Norm. amplitude")
axes[1, 0].set_title(f"Cross-section at z = {z_sc*1e3:.0f} mm", fontsize=10)
axes[1, 0].legend(fontsize=8);  axes[1, 0].grid(True, alpha=0.3)

# (1,1): baseline only
axes[1, 1].plot(x_b, sec_b, color='g', lw=2.2, label="baseline recovered (monopole)")
axes[1, 1].plot(x_b, sec_d, color='steelblue', lw=1.2, alpha=0.4, label="diff (dipole)")
axes[1, 1].axvline(x_base_est, color='g',      ls='--', lw=2.0)
axes[1, 1].axhline(0, color='k', lw=0.5)
axes[1, 1].set_xlim(x_lo3, x_hi3);  axes[1, 1].set_ylim(-1.4, 1.4)
axes[1, 1].set_xlabel("Position [m]");  axes[1, 1].set_ylabel("Norm. amplitude")
axes[1, 1].set_title("Baseline cross-section (monopole vs dipole)", fontsize=10)
axes[1, 1].legend(fontsize=8);  axes[1, 1].grid(True, alpha=0.3)

# (1,2): monitor only
axes[1, 2].plot(x_b, sec_m, color='darkred', lw=2.2, ls='--', label="monitor recovered (monopole)")
axes[1, 2].plot(x_b, sec_d, color='steelblue', lw=1.2, alpha=0.4, label="diff (dipole)")
axes[1, 2].axvline(x_mon_est, color='darkred', ls=':',  lw=2.0)
axes[1, 2].axhline(0, color='k', lw=0.5)
axes[1, 2].set_xlim(x_lo3, x_hi3);  axes[1, 2].set_ylim(-1.4, 1.4)
axes[1, 2].set_xlabel("Position [m]");  axes[1, 2].set_ylabel("Norm. amplitude")
axes[1, 2].set_title("Monitor cross-section (monopole vs dipole)", fontsize=10)
axes[1, 2].legend(fontsize=8);  axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Advanced Timelapse Image: Lateral Displacement Map
#
# The phase gradient ∂(Δφ)/∂x [rad/m] is proportional to the lateral shift.
# Dividing by k_calib [rad/m²] converts it to a physical displacement image:
#
#   displacement[z, x] = ∂(Δφ)/∂x / k_calib   [m]
#
# Properties vs standard migrated difference (dipole):
#   ✓  Monopole  — single focused peak, not a two-lobe dipole
#   ✓  Quantitative — peak value ≈ true lateral shift in metres
#   ✓  Signed — positive = rightward shift, negative = leftward
#   ✓  Sub-wavelength — resolves shifts far below the Rayleigh limit
# =============================================================================

# ── Displacement map ──────────────────────────────────────────────────────────
displacement_map = dphi_dx_full / k_calib              # (n_z, n_x)  [m]
displacement_msk = np.where(mask_delta, np.nan, displacement_map)

# ── Peak measurement at z = z_sc ─────────────────────────────────────────────
iz_sc       = np.argmin(np.abs(z_mig - z_sc))
disp_sec    = displacement_msk[iz_sc, :]
ix_pk       = np.nanargmax(np.abs(disp_sec))
disp_peak   = disp_sec[ix_pk]
x_pk        = x_b[ix_pk]

print("Advanced Timelapse: Lateral Displacement Map")
print(f"  True shift              : {dx_shift*1e3:.3f} mm  ({shift_frac}\u03bb)")
print(f"  Peak displacement       : {disp_peak*1e3:.3f} mm  at x = {x_pk*1e3:.3f} mm")
print(f"  Midpoint estimate       : {x_mid_est*1e3:.3f} mm")
print(f"  Measurement error       : {abs(disp_peak - dx_shift)*1e6:.1f} \u00b5m")
print(f"  k_calib                 : {k_calib:.1f} rad/m\u00b2")

# ── Figure: 2 × 2 ────────────────────────────────────────────────────────────
ext_mig  = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo4    = x_mid_est - 5 * lam;  x_hi4 = x_mid_est + 5 * lam
z_lo4    = z_sc - 0.08;           z_hi4 = z_sc + 0.08

def zoom_adv(ax):
    ax.set_xlim(x_lo4, x_hi4);  ax.set_ylim(z_hi4, z_lo4)
    ax.axvline(x_base_est, color='g',       ls='--', lw=1.8,
               label=f'base  {x_base_est*1e3:.2f} mm')
    ax.axvline(x_mon_est,  color='darkred', ls=':',  lw=1.8,
               label=f'mon   {x_mon_est*1e3:.2f} mm')

clip_d    = np.percentile(np.abs(mig_pi_diff), 99)
clip_disp = np.nanpercentile(np.abs(displacement_msk), 99)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Advanced Timelapse Image: Lateral Displacement Map\n"
    f"shift = {shift_frac}\u03bb = {dx_shift*1e3:.1f} mm  |  "
    f"peak measured = {disp_peak*1e3:.2f} mm  |  "
    f"error = {abs(disp_peak-dx_shift)*1e6:.0f} \u00b5m",
    fontsize=12, fontweight='bold',
)

# (0,0): standard migrated difference for comparison
im0 = axes[0, 0].imshow(
    mig_pi_diff, aspect='auto', origin='upper', extent=ext_mig,
    cmap='RdBu', vmin=-clip_d, vmax=clip_d, interpolation='bilinear',
)
axes[0, 0].set_title("Standard timelapse: migrated difference (dipole)", fontsize=10)
axes[0, 0].set_xlabel("Position [m]");  axes[0, 0].set_ylabel("Depth [m]")
zoom_adv(axes[0, 0]);  axes[0, 0].legend(fontsize=8)
fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04, label='[a.u.]')

# (0,1): advanced timelapse: displacement map in mm
im1 = axes[0, 1].imshow(
    displacement_msk * 1e3, aspect='auto', origin='upper', extent=ext_mig,
    cmap='RdBu', vmin=-clip_disp*1e3, vmax=clip_disp*1e3, interpolation='none',
)
axes[0, 1].set_title(
    r"Advanced timelapse: displacement map  $\partial(\Delta\varphi)/\partial x\,/\,k_{calib}$",
    fontsize=10,
)
axes[0, 1].set_xlabel("Position [m]");  axes[0, 1].set_ylabel("Depth [m]")
zoom_adv(axes[0, 1]);  axes[0, 1].legend(fontsize=8)
fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04, label='displacement [mm]')

# (1,0): cross-section — dipole vs displacement (both normalised for shape comparison)
sec_d4  = mig_pi_diff[iz_sc, :] / (np.abs(mig_pi_diff[iz_sc, :]).max() + 1e-30)
sec_adv = disp_sec / (clip_disp + 1e-30)

axes[1, 0].fill_between(x_b, sec_adv, alpha=0.15, color='tomato')
axes[1, 0].plot(x_b, sec_adv, color='tomato',   lw=2.5, label='displacement (norm.)')
axes[1, 0].plot(x_b, sec_d4,  color='steelblue', lw=1.5, ls='--', alpha=0.7,
                label='diff / dipole (norm.)')
axes[1, 0].axvline(x_base_est, color='g',       ls='--', lw=1.2, label='base est')
axes[1, 0].axvline(x_mon_est,  color='darkred', ls=':',  lw=1.2, label='mon est')
axes[1, 0].axhline(0, color='k', lw=0.5)
axes[1, 0].set_xlim(x_lo4, x_hi4);  axes[1, 0].set_ylim(-1.4, 1.4)
axes[1, 0].set_xlabel("Position [m]");  axes[1, 0].set_ylabel("Normalised amplitude")
axes[1, 0].set_title(
    f"Cross-section at z = {z_sc*1e3:.0f} mm — shape: monopole vs dipole", fontsize=10,
)
axes[1, 0].legend(fontsize=8);  axes[1, 0].grid(True, alpha=0.3)

# (1,1): displacement in physical units with true-shift reference line
axes[1, 1].fill_between(x_b, disp_sec * 1e3, alpha=0.15, color='tomato')
axes[1, 1].plot(x_b, disp_sec * 1e3, color='tomato', lw=2.5,
                label='measured displacement')
axes[1, 1].axhline( dx_shift * 1e3, color='k', ls='--', lw=1.8,
                    label=f'true shift  {dx_shift*1e3:.2f} mm')
axes[1, 1].axhline(-dx_shift * 1e3, color='k', ls='--', lw=1.0, alpha=0.4)
axes[1, 1].axhline(0, color='k', lw=0.5)
axes[1, 1].axvline(x_base_est, color='g',       ls='--', lw=1.2, label='base est')
axes[1, 1].axvline(x_mon_est,  color='darkred', ls=':',  lw=1.2, label='mon est')
axes[1, 1].set_xlim(x_lo4, x_hi4)
axes[1, 1].set_xlabel("Position [m]")
axes[1, 1].set_ylabel("Estimated displacement [mm]")
axes[1, 1].set_title(
    f"Displacement profile at z = {z_sc*1e3:.0f} mm  (true = {dx_shift*1e3:.2f} mm)",
    fontsize=10,
)
axes[1, 1].legend(fontsize=8);  axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Phase Informed Timelapse Migration: Vertical Movement

In [ ]:
# -- Acquisition --------------------------------------------------------------
dx_b   = 0.005 / 4
n_x_b  = 160 * 4
x_b    = np.arange(n_x_b) * dx_b       # 0 -- 0.795 m profile
x_sc0  = x_b.mean()

dt_b   = 0.004
n_t_b  = 1300                           # t_max = 5.196 ns > TWT(0.30 m) = 3.57 ns
t_b    = np.arange(n_t_b) * dt_b

z_sc   = 0.30
z_mig  = np.arange(0, 0.42, dx_b)      # 84 depth steps

# Vertical Shift — Phase Informed Timelapse Migration

In [ ]:
# =============================================================================
# Vertical Shift — Phase Informed Timelapse Migration
#
# Scatterer moves downward by dz_shift = shift_frac_v * lam.
# Everything mirrors the lateral section; gradient direction is now ∂/∂z.
# =============================================================================

# ── Parameters ───────────────────────────────────────────────────────────────
# v_ice, f_c, lam, x_b, t_b, z_mig, dx_b, x_sc0, z_sc  from earlier cells
dz_b = float(z_mig[1] - z_mig[0])

shift_frac_v = 1/4          # vertical shift as fraction of wavelength  <-- tune
dz_shift     = shift_frac_v * lam
z_sc_mon_v   = z_sc + dz_shift   # monitor scatterer depth

print(f"lambda      = {lam*1e3:.2f} mm")
print(f"vert shift  = {shift_frac_v} x lambda = {dz_shift*1e3:.2f} mm  ({dz_shift/dz_b:.1f} samples)")
print(f"Baseline:   x = {x_sc0*1e3:.1f} mm,  z = {z_sc*1e3:.1f} mm")
print(f"Monitor:    x = {x_sc0*1e3:.1f} mm,  z = {z_sc_mon_v*1e3:.1f} mm")

# ── 1. Synthesise B-scans ─────────────────────────────────────────────────────
print("\nSynthesising B-scans...")
B_v_base = point_bscan(x_sc0, z_sc,       x_b, t_b, v_ice, f_c)
B_v_mon  = point_bscan(x_sc0, z_sc_mon_v, x_b, t_b, v_ice, f_c)

# ── 2. Migrate both ───────────────────────────────────────────────────────────
print("\nMigrating baseline...")
t0 = _time.time()
mig_v_base = gazdag_migration(B_v_base, x_b, t_b, z_mig, v_ice)
print(f"  done in {_time.time()-t0:.1f} s")

print("\nMigrating monitor...")
t0 = _time.time()
mig_v_mon  = gazdag_migration(B_v_mon,  x_b, t_b, z_mig, v_ice)
print(f"  done in {_time.time()-t0:.1f} s")

# ── 3. Migrated difference ────────────────────────────────────────────────────
mig_v_diff = mig_v_base - mig_v_mon

# ── 4. Instantaneous amplitude & phase (Riesz-z) ─────────────────────────────
R_v_base = riesz_z(mig_v_base)
R_v_mon  = riesz_z(mig_v_mon)

A_v_base = np.sqrt(mig_v_base**2 + R_v_base**2)
A_v_mon  = np.sqrt(mig_v_mon**2  + R_v_mon**2)

phi_v_base  = np.arctan2(R_v_base, mig_v_base)
phi_v_mon   = np.arctan2(R_v_mon,  mig_v_mon)
phi_v_delta = np.angle(np.exp(1j * (phi_v_mon - phi_v_base)))


In [ ]:

# ── 5. Amplitude-masked phase images ─────────────────────────────────────────
thresh_frac_v = 0.10

mask_v_base  = A_v_base < thresh_frac_v * A_v_base.max()
mask_v_mon   = A_v_mon  < thresh_frac_v * A_v_mon.max()
mask_v_delta = mask_v_base | mask_v_mon

phi_v_base_msk  = np.where(mask_v_base,  np.nan, phi_v_base)
phi_v_mon_msk   = np.where(mask_v_mon,   np.nan, phi_v_mon)
phi_v_delta_msk = np.where(mask_v_delta, np.nan, phi_v_delta)

print(f"\nAmplitude mask ({thresh_frac_v*100:.0f}% of peak):")
print(f"  baseline: {mask_v_base.mean()*100:.1f}%  monitor: {mask_v_mon.mean()*100:.1f}%  delta: {mask_v_delta.mean()*100:.1f}%")

# ── 6. QC display (4 x 3) ────────────────────────────────────────────────────
ext_b   = [x_b[0], x_b[-1], t_b[-1], t_b[0]]
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]

x_mid_v = x_sc0
z_mid_v = 0.5 * (z_sc + z_sc_mon_v)
x_lo_v  = x_mid_v - 5 * lam;  x_hi_v = x_mid_v + 5 * lam
z_lo_v  = z_mid_v - 0.10;     z_hi_v = z_mid_v + 0.10

def mig_zoom_v(ax):
    ax.set_xlim(x_lo_v, x_hi_v);  ax.set_ylim(z_hi_v, z_lo_v)
    ax.plot(x_sc0, z_sc,       'g*', ms=12, zorder=7, label='baseline')
    ax.plot(x_sc0, z_sc_mon_v, 'r^', ms=10, zorder=7, label=f'monitor ({shift_frac_v}\u03bb)')

clip_b_v   = np.percentile(np.abs(B_v_base), 99)
clip_diff_v = np.percentile(np.abs(mig_v_diff), 99)
hsv_cmap_v  = plt.get_cmap('hsv').copy();  hsv_cmap_v.set_bad('lightgrey')

fig, axes = plt.subplots(4, 3, figsize=(18, 18))
fig.suptitle(
    f"Vertical Shift — Phase Informed Timelapse Migration\n"
    f"shift = {shift_frac_v}\u03bb = {dz_shift*1e3:.1f} mm downward",
    fontsize=13, fontweight='bold',
)

# Row 0: B-scans
for ax, data, title in [
    (axes[0,0], B_v_base, "Baseline B-scan"),
    (axes[0,1], B_v_mon,  f"Monitor B-scan (shift {shift_frac_v}\u03bb down)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_b,
                   cmap='RdBu', vmin=-clip_b_v, vmax=clip_b_v, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("TWT [ns]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

B_v_diff = B_v_mon - B_v_base
im = axes[0,2].imshow(B_v_diff, aspect='auto', origin='upper', extent=ext_b,
                       cmap='RdBu', vmin=-np.percentile(np.abs(B_v_diff),99),
                       vmax=np.percentile(np.abs(B_v_diff),99), interpolation='bilinear')
axes[0,2].set_title("Difference B-scan", fontsize=10)
axes[0,2].set_xlabel("Position [m]");  axes[0,2].set_ylabel("TWT [ns]")
fig.colorbar(im, ax=axes[0,2], fraction=0.046, pad=0.04)

# Row 1: Migrated images
for ax, data, clip, title in [
    (axes[1,0], mig_v_base, 1.0,         "Migrated baseline"),
    (axes[1,1], mig_v_mon,  1.0,         f"Migrated monitor ({shift_frac_v}\u03bb down)"),
    (axes[1,2], mig_v_diff, clip_diff_v, "Migrated difference (baseline - monitor)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='RdBu', vmin=-clip, vmax=clip, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom_v(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 2: Unmasked instantaneous phase
for ax, data, title in [
    (axes[2,0], phi_v_base,  r"Inst. phase $\varphi_{base}$"),
    (axes[2,1], phi_v_mon,   r"Inst. phase $\varphi_{mon}$"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom_v(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

im = axes[2,2].imshow(phi_v_delta, aspect='auto', origin='upper', extent=ext_mig,
                      cmap='hsv', vmin=-np.pi, vmax=np.pi, interpolation='bilinear')
axes[2,2].set_title(r"Phase diff $\Delta\varphi$ (wrapped)", fontsize=10)
axes[2,2].set_xlabel("Position [m]");  axes[2,2].set_ylabel("Depth [m]")
mig_zoom_v(axes[2,2]);  axes[2,2].legend(fontsize=8)
fig.colorbar(im, ax=axes[2,2], fraction=0.046, pad=0.04, label='[rad]')

# Row 3: Amplitude-masked phase
for ax, data, title in [
    (axes[3,0], phi_v_base_msk,  f"$\\varphi_{{base}}$ masked ({thresh_frac_v*100:.0f}%)"),
    (axes[3,1], phi_v_mon_msk,   f"$\\varphi_{{mon}}$ masked ({thresh_frac_v*100:.0f}%)"),
    (axes[3,2], phi_v_delta_msk, f"$\\Delta\\varphi$ masked (either < {thresh_frac_v*100:.0f}%)"),
]:
    im = ax.imshow(data, aspect='auto', origin='upper', extent=ext_mig,
                   cmap=hsv_cmap_v, vmin=-np.pi, vmax=np.pi, interpolation='none')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    mig_zoom_v(ax);  ax.legend(fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='[rad]')

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Phase cross-section along depth at x = x_sc0
# =============================================================================

ix_sc0 = np.argmin(np.abs(x_b - x_sc0))   # column index of scatterer

# Vertical profiles at x = x_sc0
prof_v_A_base       = A_v_base[:, ix_sc0]
prof_v_A_mon        = A_v_mon[:, ix_sc0]
prof_v_phi_base     = phi_v_base[:, ix_sc0]
prof_v_phi_mon      = phi_v_mon[:, ix_sc0]
prof_v_phi_delta    = phi_v_delta[:, ix_sc0]
prof_v_phi_base_msk = phi_v_base_msk[:, ix_sc0]
prof_v_phi_mon_msk  = phi_v_mon_msk[:, ix_sc0]
prof_v_phi_delta_msk= phi_v_delta_msk[:, ix_sc0]

# Average delta_phi between the two scatterer depth markers
mask_btw_v  = (z_mig >= min(z_sc, z_sc_mon_v)) & (z_mig <= max(z_sc, z_sc_mon_v))
mean_dphi_v = np.nanmean(prof_v_phi_delta_msk[mask_btw_v])

# -- Full cross-section figure (amplitude + phase vs depth) ------------------
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    f"Phase cross-section at x = {x_sc0*1e3:.1f} mm  (scatterer column)\n"
    f"Vertical shift = {shift_frac_v}λ = {dz_shift*1e3:.1f} mm",
    fontsize=12, fontweight='bold',
)

# Panel 0: amplitude
axes[0].plot(z_mig, prof_v_A_base, 'b-',  lw=1.8, label='baseline')
axes[0].plot(z_mig, prof_v_A_mon,  'r--', lw=1.8, label=f'monitor ({shift_frac_v}λ)')
axes[0].axvline(z_sc,       color='g', ls='--', lw=1.2, label=f'baseline z = {z_sc*1e3:.1f} mm')
axes[0].axvline(z_sc_mon_v, color='r', ls=':',  lw=1.2, label=f'monitor z = {z_sc_mon_v*1e3:.1f} mm')
axes[0].axhline(thresh_frac_v * A_v_base.max(), color='b', ls=':', lw=1.0, alpha=0.7)
axes[0].axhline(thresh_frac_v * A_v_mon.max(),  color='r', ls=':', lw=1.0, alpha=0.7)
axes[0].set_ylabel("Instantaneous amplitude [a.u.]")
axes[0].set_xlim(0.2,0.4)
axes[0].legend(fontsize=8);  axes[0].grid(True, alpha=0.3)

# Panel 1: phase
axes[1].plot(z_mig, prof_v_phi_base,      color='b', lw=0.9, alpha=0.25)
axes[1].plot(z_mig, prof_v_phi_mon,       color='r', lw=0.9, alpha=0.25)
axes[1].plot(z_mig, prof_v_phi_delta,     color='k', lw=0.9, alpha=0.25)
axes[1].plot(z_mig, prof_v_phi_base_msk,  color='b', lw=2.2,
             label=r"$\varphi_{base}$")
axes[1].plot(z_mig, prof_v_phi_mon_msk,   color='r', lw=2.2, ls='--',
             label=f"$\\varphi_{{mon}}$ ({shift_frac_v}λ)")
axes[1].plot(z_mig, prof_v_phi_delta_msk, color='k', lw=2.2,
             label=r"$\Delta\varphi$ (wrapped)")
axes[1].axvline(z_sc,       color='g', ls='--', lw=1.2)
axes[1].axvline(z_sc_mon_v, color='r', ls=':',  lw=1.2)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].axhline(mean_dphi_v, color='purple', ls='--', lw=0.9, alpha=0.7,
                label=fr"$\langle\Delta\varphi\rangle$ = {mean_dphi_v:.3f} rad")
axes[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
axes[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'], fontsize=10)
axes[1].set_ylabel("Instantaneous phase [rad]")
axes[1].set_xlabel("Depth [m]")
axes[1].set_xlim(0.2,0.4)
axes[1].legend(fontsize=9);  axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -- Zoomed window around scatterer depths -----------------------------------
z_mid_est_v0 = 0.5 * (z_sc + z_sc_mon_v)
z_win_lo_v   = min(z_sc, z_sc_mon_v) - 2 * lam
z_win_hi_v   = max(z_sc, z_sc_mon_v) + 2 * lam

fig2, axes2 = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
fig2.suptitle(
    f"Phase cross-section – zoomed (±2λ around scatterers)\n"
    f"vertical shift = {shift_frac_v}λ = {dz_shift*1e3:.1f} mm",
    fontsize=12, fontweight='bold',
)

for ax, prof, col, ls, label in [
    (axes2[0], prof_v_A_base, 'b', '-',  'baseline'),
    (axes2[0], prof_v_A_mon,  'r', '--', f'monitor ({shift_frac_v}λ)'),
]:
    ax.plot(z_mig, prof, color=col, lw=1.8, ls=ls, label=label)
axes2[0].axvline(z_sc,       color='g', ls='--', lw=1.5, label=f'base z = {z_sc*1e3:.1f} mm')
axes2[0].axvline(z_sc_mon_v, color='r', ls=':',  lw=1.5, label=f'mon z = {z_sc_mon_v*1e3:.1f} mm')
axes2[0].set_xlim(z_win_lo_v, z_win_hi_v)
axes2[0].set_ylabel("Instantaneous amplitude [a.u.]")
axes2[0].legend(fontsize=8);  axes2[0].grid(True, alpha=0.3)

axes2[1].plot(z_mig, prof_v_phi_base,      color='b', lw=0.9, alpha=0.25)
axes2[1].plot(z_mig, prof_v_phi_mon,       color='r', lw=0.9, alpha=0.25)
axes2[1].plot(z_mig, prof_v_phi_delta,     color='k', lw=0.9, alpha=0.25)
axes2[1].plot(z_mig, prof_v_phi_base_msk,  color='b', lw=2.5, label=r"$\varphi_{base}$")
axes2[1].plot(z_mig, prof_v_phi_mon_msk,   color='r', lw=2.5, ls='--',
              label=f"$\\varphi_{{mon}}$ ({shift_frac_v}λ)")
axes2[1].plot(z_mig, prof_v_phi_delta_msk, color='k', lw=2.5, label=r"$\Delta\varphi$")
axes2[1].axvline(z_sc,       color='g', ls='--', lw=1.5)
axes2[1].axvline(z_sc_mon_v, color='r', ls=':',  lw=1.5)
axes2[1].axhline(0, color='k', lw=0.5)
axes2[1].axhline(mean_dphi_v, color='purple', ls='--', lw=0.9, alpha=0.7,
                 label=fr"$\langle\Delta\varphi\rangle$ = {mean_dphi_v:.3f} rad")

# -- Zero crossing of Delta-phi in the zoom window ---------------------------
mask_win_v   = (z_mig >= z_win_lo_v) & (z_mig <= z_win_hi_v)
z_win_v      = z_mig[mask_win_v]
delta_win_v  = prof_v_phi_delta_msk[mask_win_v]

# z_zc_v     = []
# slope_zc_v = []
# for i in range(len(delta_win_v) - 1):
#     a, b = delta_win_v[i], delta_win_v[i + 1]
#     if np.isnan(a) or np.isnan(b):
#         continue
#     if a * b < 0:
#         t = a / (a - b)
#         z_zc_v.append(z_win_v[i] + t * (z_win_v[i + 1] - z_win_v[i]))
#         slope_zc_v.append((b - a) / (z_win_v[i + 1] - z_win_v[i]))   # rad / m

# if z_zc_v:
#     dx_tan_v = lam
#     for zz, sl in zip(z_zc_v, slope_zc_v):
#         axes2[0].axvline(zz, color='purple', ls='-', lw=2.0, alpha=0.8,
#                          label=f'Δφ=0  z={zz*1e3:.2f} mm')
#         axes2[1].axvline(zz, color='purple', ls='-', lw=2.0, alpha=0.8,
#                          label=f'Δφ=0  z={zz*1e3:.2f} mm')
#         z_tan = np.array([zz - dx_tan_v, zz + dx_tan_v])
#         axes2[1].plot(z_tan, sl * (z_tan - zz), color='orange', lw=1.8, ls='--',
#                       label=f'slope = {sl:.1f} rad/m')
#         axes2[1].annotate(
#             f'slope = {sl:.1f} rad/m',
#             xy=(zz, 0), xytext=(8, 6), textcoords='offset points',
#             fontsize=9, color='darkorange', fontweight='bold',
#             arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.0),
#         )
#     print(f"\nΔφ zero crossing(s) in zoom window:")
#     for zz, sl in zip(z_zc_v, slope_zc_v):
#         print(f"  z_zero = {zz*1e3:.3f} mm   slope = {sl:.2f} rad/m  ({sl*lam:.3f} rad/lambda)")
#         print(f"  baseline z = {z_sc*1e3:.3f} mm   monitor z = {z_sc_mon_v*1e3:.3f} mm")
# else:
#     print("No zero crossing found in zoom window.")

axes2[1].set_ylim(-np.pi - 0.3, np.pi + 0.3)
axes2[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes2[1].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])
axes2[1].set_ylabel("Instantaneous phase [rad]")
axes2[1].set_xlabel("Depth [m]")
for ax in axes2:
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Phase values at scatterer depths
iz_base_v = np.argmin(np.abs(z_mig - z_sc))
iz_mon_v  = np.argmin(np.abs(z_mig - z_sc_mon_v))
print(f"\nAt baseline z = {z_sc*1e3:.1f} mm:")
print(f"  phi_base={prof_v_phi_base[iz_base_v]:.3f}  phi_mon={prof_v_phi_mon[iz_base_v]:.3f}  delta={prof_v_phi_delta[iz_base_v]:.3f} rad")
print(f"At monitor z = {z_sc_mon_v*1e3:.1f} mm:")
print(f"  phi_base={prof_v_phi_base[iz_mon_v]:.3f}  phi_mon={prof_v_phi_mon[iz_mon_v]:.3f}  delta={prof_v_phi_delta[iz_mon_v]:.3f} rad")
print(f"\nMean Δφ between z_sc and z_sc_mon_v: {mean_dphi_v:.4f} rad")

In [ ]:
# =============================================================================
# Phase-gradient enhanced image (vertical) — ∂(Δφ)/∂z
# Circular gradient along z (axis=0) to avoid wrap artefacts.
# =============================================================================

exp_phi_v       = np.exp(1j * phi_v_delta)
dexp_dz         = np.gradient(exp_phi_v, z_mig, axis=0)
dphi_dz_full    = np.imag(np.conj(exp_phi_v) * dexp_dz)   # rad / m

dphi_dz_msk     = np.where(mask_v_delta, np.nan, dphi_dz_full)

# Cross-sections at x = x_sc0
ix_sc0_v    = np.argmin(np.abs(x_b - x_sc0))
sec_v_diff  = mig_v_diff[:, ix_sc0_v]
sec_v_dphi  = dphi_dz_msk[:, ix_sc0_v]

ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
clip_dv  = np.percentile(np.abs(mig_v_diff), 99)
clip_gpv = np.nanpercentile(np.abs(dphi_dz_msk), 99)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Phase-gradient enhanced image (vertical)  \u2014  shift = {shift_frac_v}\u03bb = {dz_shift*1e3:.1f} mm\n"
    r"$\partial(\Delta\varphi)/\partial z$  converts vertical dipole \u2192 monopole",
    fontsize=12, fontweight='bold',
)

im0 = axes[0,0].imshow(mig_v_diff, aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-clip_dv, vmax=clip_dv, interpolation='bilinear')
axes[0,0].set_title("Migrated difference (vertical dipole)", fontsize=11)
axes[0,0].set_xlabel("Position [m]");  axes[0,0].set_ylabel("Depth [m]")
mig_zoom_v(axes[0,0]);  axes[0,0].legend(fontsize=8)
fig.colorbar(im0, ax=axes[0,0], fraction=0.046, pad=0.04)

im1 = axes[0,1].imshow(dphi_dz_msk, aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-clip_gpv, vmax=clip_gpv, interpolation='none')
axes[0,1].set_title(r"Phase gradient  $\partial(\Delta\varphi)/\partial z$  (monopole)", fontsize=11)
axes[0,1].set_xlabel("Position [m]");  axes[0,1].set_ylabel("Depth [m]")
mig_zoom_v(axes[0,1]);  axes[0,1].legend(fontsize=8)
fig.colorbar(im1, ax=axes[0,1], fraction=0.046, pad=0.04, label='rad / m')

# Cross-section: depth profiles at x = x_sc0
axes[1,0].plot(z_mig, sec_v_diff, 'b-', lw=2.0, label='mig diff (vertical dipole)')
axes[1,0].axvline(z_sc,       color='g', ls='--', lw=1.2, label='baseline z')
axes[1,0].axvline(z_sc_mon_v, color='r', ls=':',  lw=1.2, label='monitor z')
axes[1,0].axhline(0, color='k', lw=0.5)
axes[1,0].set_xlim(z_win_lo_v, z_win_hi_v)
axes[1,0].set_xlabel("Depth [m]");  axes[1,0].set_ylabel("Amplitude [a.u.]")
axes[1,0].set_title(f"Depth profile at x = {x_sc0*1e3:.1f} mm — diff", fontsize=10)
axes[1,0].legend(fontsize=8);  axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(z_mig, sec_v_dphi, 'r-', lw=2.0,
               label=r'$\partial(\Delta\varphi)/\partial z$')
axes[1,1].axvline(z_sc,       color='g', ls='--', lw=1.2, label='baseline z')
axes[1,1].axvline(z_sc_mon_v, color='r', ls=':',  lw=1.2, label='monitor z')
axes[1,1].axhline(0, color='k', lw=0.5)
axes[1,1].set_xlim(z_win_lo_v, z_win_hi_v)
axes[1,1].set_xlabel("Depth [m]");  axes[1,1].set_ylabel("rad / m")
axes[1,1].set_title(r"Depth profile at x — $\partial(\Delta\varphi)/\partial z$", fontsize=10)
axes[1,1].legend(fontsize=8);  axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

peak_dphi_v = np.nanmax(np.abs(sec_v_dphi))
print(f"Phase gradient peak at x = {x_sc0*1e3:.1f} mm: {peak_dphi_v:.2f} rad/m")


In [ ]:
# =============================================================================
# Phase-informed vertical position recovery
#
# Zero crossing of Δφ = midpoint in z  →  z_mid = (z_base + z_mon) / 2
# Slope at zero crossing ∝ dz_shift   →  z_base = z_mid - dz/2
#                                         z_mon  = z_mid + dz/2
# Sharpening: diff(z) ≈ -dz_shift · ∂PSF/∂z
#   ⟹  -∫diff dz / dz_shift  ≈  PSF_baseline
# =============================================================================

if not z_zc_v:
    raise RuntimeError("No vertical zero crossing — run cross-section cell first.")

iz_sc0_col = np.argmin(np.abs(x_b - x_sc0))

# ── Step 1: positions ─────────────────────────────────────────────────────────
z_mid_est_v  = z_zc_v[0]
k_calib_v    = slope_zc_v[0] / dz_shift          # rad/m per m  (calibration)
dz_shift_est = slope_zc_v[0] / k_calib_v
z_base_est_v = z_mid_est_v - dz_shift_est / 2
z_mon_est_v  = z_mid_est_v + dz_shift_est / 2

z_mid_true_v = (z_sc + z_sc_mon_v) / 2

print("Phase-informed vertical position recovery:")
print(f"  Midpoint : {z_mid_est_v*1e3:.3f} mm  (true: {z_mid_true_v*1e3:.3f} mm,  err: {abs(z_mid_est_v-z_mid_true_v)*1e6:.0f} \u00b5m)")
print(f"  dz_shift : {dz_shift_est*1e3:.3f} mm  (true: {dz_shift*1e3:.3f} mm,  err: {abs(dz_shift_est-dz_shift)*1e6:.0f} \u00b5m)")
print(f"  Baseline : {z_base_est_v*1e3:.3f} mm  (true: {z_sc*1e3:.3f} mm,  err: {abs(z_base_est_v-z_sc)*1e6:.0f} \u00b5m)")
print(f"  Monitor  : {z_mon_est_v*1e3:.3f} mm  (true: {z_sc_mon_v*1e3:.3f} mm,  err: {abs(z_mon_est_v-z_sc_mon_v)*1e6:.0f} \u00b5m)")

# ── Step 2: integration sharpening (along z, axis=0) ─────────────────────────
mig_v_diff_int = np.cumsum(mig_v_diff, axis=0) * dz_b
img_v_base_rec = -mig_v_diff_int / dz_shift_est

from scipy.ndimage import shift as nd_shift
img_v_mon_rec  = nd_shift(img_v_base_rec, (dz_shift_est / dz_b, 0), mode='nearest')

pk_vb = np.abs(img_v_base_rec).max() + 1e-30
pk_vm = np.abs(img_v_mon_rec).max()  + 1e-30

# ── Figure: 2 × 3 ─────────────────────────────────────────────────────────────
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo_r  = x_sc0 - 5 * lam;  x_hi_r = x_sc0 + 5 * lam
z_lo_r  = z_mid_est_v - 0.10;  z_hi_r = z_mid_est_v + 0.10

def zoom_v_rec(ax):
    ax.set_xlim(x_lo_r, x_hi_r);  ax.set_ylim(z_hi_r, z_lo_r)
    ax.axhline(z_mid_est_v,  color='purple', ls='-',  lw=1.5,
               label=f'\u0394\u03c6=0  z={z_mid_est_v*1e3:.2f} mm')
    ax.axhline(z_base_est_v, color='g',       ls='--', lw=2.0,
               label=f'base est  {z_base_est_v*1e3:.2f} mm')
    ax.axhline(z_mon_est_v,  color='darkred', ls=':',  lw=2.0,
               label=f'mon est   {z_mon_est_v*1e3:.2f} mm')

clip_dv2 = np.percentile(np.abs(mig_v_diff), 99)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"Vertical position recovery  \u2014  shift = {shift_frac_v}\u03bb = {dz_shift*1e3:.1f} mm\n"
    "Zero crossing = midpoint in z; slope gives half-separation",
    fontsize=12, fontweight='bold',
)

# Row 0: images
im0 = axes[0,0].imshow(mig_v_diff, aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-clip_dv2, vmax=clip_dv2, interpolation='bilinear')
axes[0,0].set_title("Migrated difference (vertical dipole)", fontsize=10)
axes[0,0].set_xlabel("Position [m]");  axes[0,0].set_ylabel("Depth [m]")
zoom_v_rec(axes[0,0]);  axes[0,0].legend(fontsize=7)
fig.colorbar(im0, ax=axes[0,0], fraction=0.046, pad=0.04)

for ax, img, pk, title in [
    (axes[0,1], img_v_base_rec, pk_vb, "Recovered baseline  (\u2212\u222bdiff\u00b7dz / \u03b4z)"),
    (axes[0,2], img_v_mon_rec,  pk_vm, "Recovered monitor  (baseline shifted by \u03b4z)"),
]:
    im = ax.imshow(img / pk, aspect='auto', origin='upper', extent=ext_mig,
                   cmap='RdBu', vmin=-1, vmax=1, interpolation='bilinear')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Position [m]");  ax.set_ylabel("Depth [m]")
    zoom_v_rec(ax);  ax.legend(fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Row 1: depth profiles at x = x_sc0
sec_vd  = mig_v_diff[:, iz_sc0_col] / (np.abs(mig_v_diff[:, iz_sc0_col]).max() + 1e-30)
sec_vb  = img_v_base_rec[:, iz_sc0_col] / pk_vb
sec_vm  = img_v_mon_rec[:, iz_sc0_col]  / pk_vm

for ax, secs, labels, colors, title in [
    (axes[1,0],
     [(sec_vd,'steelblue',1.5,'--',0.7),(sec_vb,'g',2.2,'-',1.0),(sec_vm,'darkred',2.2,'--',1.0)],
     "Overlay: dipole vs recovered", None, f"Depth profile at x={x_sc0*1e3:.0f} mm"),
    (axes[1,1],
     [(sec_vd,'steelblue',1.2,'--',0.4),(sec_vb,'g',2.2,'-',1.0)],
     "Baseline", None, "Baseline recovered vs dipole"),
    (axes[1,2],
     [(sec_vd,'steelblue',1.2,'--',0.4),(sec_vm,'darkred',2.2,'--',1.0)],
     "Monitor", None, "Monitor recovered vs dipole"),
]:
    lbl_map = {'steelblue': 'diff (dipole)', 'g': 'baseline rec', 'darkred': 'monitor rec'}
    for s, col, lw, ls, alpha in secs:
        ax.plot(z_mig, s, color=col, lw=lw, ls=ls, alpha=alpha, label=lbl_map[col])
    ax.axvline(z_mid_est_v,  color='purple', ls='-',  lw=1.2)
    ax.axvline(z_base_est_v, color='g',       ls='--', lw=1.2)
    ax.axvline(z_mon_est_v,  color='darkred', ls=':',  lw=1.2)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlim(z_win_lo_v, z_win_hi_v);  ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel("Depth [m]");  ax.set_ylabel("Norm. amplitude")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Advanced Timelapse Image (vertical): Depth Displacement Map
#
#   displacement_z[z, x] = ∂(Δφ)/∂z / k_calib_v   [m]
#
#   Peak = dz_shift  (quantitative vertical displacement)
#   Sign encodes downward (+) vs upward (-) movement
# =============================================================================

displacement_z_map = dphi_dz_full / k_calib_v
displacement_z_msk = np.where(mask_v_delta, np.nan, displacement_z_map)

# Peak measurement along depth profile at x = x_sc0
iz_sc0_col_v  = np.argmin(np.abs(x_b - x_sc0))
disp_z_sec    = displacement_z_msk[:, iz_sc0_col_v]
iz_pk_v       = np.nanargmax(np.abs(disp_z_sec))
disp_z_peak   = disp_z_sec[iz_pk_v]
z_pk_v        = z_mig[iz_pk_v]

print("Advanced Timelapse: Vertical Displacement Map")
print(f"  True shift              : {dz_shift*1e3:.3f} mm  ({shift_frac_v}\u03bb)")
print(f"  Peak displacement       : {disp_z_peak*1e3:.3f} mm  at z = {z_pk_v*1e3:.3f} mm")
print(f"  Midpoint estimate       : {z_mid_est_v*1e3:.3f} mm")
print(f"  Measurement error       : {abs(disp_z_peak - dz_shift)*1e6:.1f} \u00b5m")

ext_mig  = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo_a   = x_sc0 - 5 * lam;  x_hi_a = x_sc0 + 5 * lam
z_lo_a   = z_mid_est_v - 0.10;  z_hi_a = z_mid_est_v + 0.10

def zoom_adv_v(ax):
    ax.set_xlim(x_lo_a, x_hi_a);  ax.set_ylim(z_hi_a, z_lo_a)
    ax.axhline(z_base_est_v, color='g',       ls='--', lw=1.8, label=f'base  {z_base_est_v*1e3:.2f} mm')
    ax.axhline(z_mon_est_v,  color='darkred', ls=':',  lw=1.8, label=f'mon   {z_mon_est_v*1e3:.2f} mm')

clip_dv3  = np.percentile(np.abs(mig_v_diff), 99)
clip_dispv = np.nanpercentile(np.abs(displacement_z_msk), 99)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Advanced Timelapse Image: Vertical Displacement Map\n"
    f"shift = {shift_frac_v}\u03bb = {dz_shift*1e3:.1f} mm  |  "
    f"peak measured = {disp_z_peak*1e3:.2f} mm  |  "
    f"error = {abs(disp_z_peak-dz_shift)*1e6:.0f} \u00b5m",
    fontsize=12, fontweight='bold',
)

im0 = axes[0,0].imshow(mig_v_diff, aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-clip_dv3, vmax=clip_dv3, interpolation='bilinear')
axes[0,0].set_title("Standard timelapse: migrated difference (vertical dipole)", fontsize=10)
axes[0,0].set_xlabel("Position [m]");  axes[0,0].set_ylabel("Depth [m]")
zoom_adv_v(axes[0,0]);  axes[0,0].legend(fontsize=8)
fig.colorbar(im0, ax=axes[0,0], fraction=0.046, pad=0.04, label='[a.u.]')

im1 = axes[0,1].imshow(displacement_z_msk * 1e3, aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-clip_dispv*1e3, vmax=clip_dispv*1e3, interpolation='none')
axes[0,1].set_title(r"Advanced timelapse: depth displacement  $\partial(\Delta\varphi)/\partial z\,/\,k_{calib}$",
                    fontsize=10)
axes[0,1].set_xlabel("Position [m]");  axes[0,1].set_ylabel("Depth [m]")
zoom_adv_v(axes[0,1]);  axes[0,1].legend(fontsize=8)
fig.colorbar(im1, ax=axes[0,1], fraction=0.046, pad=0.04, label='displacement [mm]')

sec_vd3  = mig_v_diff[:, iz_sc0_col_v] / (np.abs(mig_v_diff[:, iz_sc0_col_v]).max() + 1e-30)
sec_vadv = disp_z_sec / (clip_dispv + 1e-30)

axes[1,0].fill_between(z_mig, sec_vadv, alpha=0.15, color='tomato')
axes[1,0].plot(z_mig, sec_vadv, color='tomato',    lw=2.5, label='displacement (norm.)')
axes[1,0].plot(z_mig, sec_vd3,  color='steelblue', lw=1.5, ls='--', alpha=0.7, label='diff / dipole (norm.)')
axes[1,0].axvline(z_base_est_v, color='g',       ls='--', lw=1.2, label='base est')
axes[1,0].axvline(z_mon_est_v,  color='darkred', ls=':',  lw=1.2, label='mon est')
axes[1,0].axhline(0, color='k', lw=0.5)
axes[1,0].set_xlim(z_win_lo_v, z_win_hi_v);  axes[1,0].set_ylim(-1.4, 1.4)
axes[1,0].set_xlabel("Depth [m]");  axes[1,0].set_ylabel("Normalised amplitude")
axes[1,0].set_title(f"Depth profile at x = {x_sc0*1e3:.0f} mm — monopole vs dipole", fontsize=10)
axes[1,0].legend(fontsize=8);  axes[1,0].grid(True, alpha=0.3)

axes[1,1].fill_between(z_mig, disp_z_sec * 1e3, alpha=0.15, color='tomato')
axes[1,1].plot(z_mig, disp_z_sec * 1e3, color='tomato', lw=2.5, label='measured displacement')
axes[1,1].axhline( dz_shift * 1e3, color='k', ls='--', lw=1.8, label=f'true shift  {dz_shift*1e3:.2f} mm')
axes[1,1].axhline(-dz_shift * 1e3, color='k', ls='--', lw=1.0, alpha=0.4)
axes[1,1].axhline(0, color='k', lw=0.5)
axes[1,1].axvline(z_base_est_v, color='g',       ls='--', lw=1.2, label='base est')
axes[1,1].axvline(z_mon_est_v,  color='darkred', ls=':',  lw=1.2, label='mon est')
axes[1,1].set_xlim(z_win_lo_v, z_win_hi_v)
axes[1,1].set_xlabel("Depth [m]");  axes[1,1].set_ylabel("Estimated displacement [mm]")
axes[1,1].set_title(f"Displacement profile  (true = {dz_shift*1e3:.2f} mm)", fontsize=10)
axes[1,1].legend(fontsize=8);  axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Vertical shift detection via phase offset (not gradient)
#
# Key insight:
#   Lateral shift  → Δφ forms a zero-crossing with slope ∝ δx
#                    → use  ∂(Δφ)/∂x  (gradient approach)
#
#   Vertical shift → wavelet slides along z (its own oscillation direction)
#                    → Δφ ≈ constant offset ≈ −2π·δz/λ everywhere near scatterer
#                    → gradient ∂(Δφ)/∂z ≈ 0  (no slope to measure)
#                    → use  Δφ VALUE directly
#
# Vertical displacement estimate:
#   δz ≈ −Δφ × λ / (2π)
#
# This gives a 2-D depth-displacement map in physical units [m].
# =============================================================================

ix_sc0_v = np.argmin(np.abs(x_b - x_sc0))

# ── 1. Displacement map from phase offset ─────────────────────────────────────
dz_from_phase_map = -phi_v_delta * lam / (2 * np.pi)            # (n_z, n_x) [m]
dz_from_phase_msk = np.where(mask_v_delta, np.nan, dz_from_phase_map)

# ── 2. Profiles at x = x_sc0 ─────────────────────────────────────────────────
prof_dphi_v   = phi_v_delta_msk[:, ix_sc0_v]           # Δφ masked profile
prof_dz_phase = dz_from_phase_msk[:, ix_sc0_v]         # δz estimate profile

# Robust estimate: median of amplitude-masked pixels near scatterer
near_sc = (z_mig > z_sc - 0.05) & (z_mig < z_sc_mon_v + 0.05)
dz_est_median = np.nanmedian(prof_dz_phase[near_sc])
dz_est_at_base = prof_dz_phase[np.argmin(np.abs(z_mig - z_sc))]

print("Vertical displacement from phase offset  (δz ≈ −Δφ · λ / 2π):")
print(f"  True dz_shift           : {dz_shift*1e3:.3f} mm")
print(f"  δz at z_sc  (point)     : {dz_est_at_base*1e3:.3f} mm  "
      f"(err: {abs(dz_est_at_base - dz_shift)*1e6:.0f} µm)")
print(f"  δz near scatterer (med) : {dz_est_median*1e3:.3f} mm  "
      f"(err: {abs(dz_est_median - dz_shift)*1e6:.0f} µm)")
print(f"  Implied Δφ at z_sc      : {prof_dphi_v[np.argmin(np.abs(z_mig-z_sc))]:.3f} rad  "
      f"(expected: {-2*np.pi*dz_shift/lam:.3f} rad)")

# ── 3. Figure: 2 × 2 ─────────────────────────────────────────────────────────
ext_mig  = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo_op  = x_sc0 - 5 * lam;   x_hi_op = x_sc0 + 5 * lam
z_lo_op  = z_mid_est_v - 0.10;  z_hi_op = z_mid_est_v + 0.10

def zoom_op(ax):
    ax.set_xlim(x_lo_op, x_hi_op);  ax.set_ylim(z_hi_op, z_lo_op)
    ax.axhline(z_sc,       color='g',       ls='--', lw=1.8,
               label=f'base z = {z_sc*1e3:.2f} mm')
    ax.axhline(z_sc_mon_v, color='darkred', ls=':',  lw=1.8,
               label=f'mon z  = {z_sc_mon_v*1e3:.2f} mm')

hsv_op = plt.get_cmap('hsv').copy()
hsv_op.set_bad('lightgrey')

clip_dphi_op = np.nanpercentile(np.abs(phi_v_delta_msk), 99)
clip_dz_op   = np.nanpercentile(np.abs(dz_from_phase_msk), 99)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Vertical Shift: Phase Offset → Displacement Map"
    f"δz ≈ −Δφ · λ/2π  |  "
    f"shift = {shift_frac_v}λ = {dz_shift*1e3:.1f} mm  |  "
    f"measured = {dz_est_median*1e3:.2f} mm",
    fontsize=12, fontweight='bold',
)

# (0,0): masked Δφ image — should be roughly constant near scatterer
im0 = axes[0, 0].imshow(
    np.where(mask_v_delta, np.nan, phi_v_delta),
    aspect='auto', origin='upper', extent=ext_mig,
    cmap=hsv_op, vmin=-np.pi, vmax=np.pi, interpolation='none',
)
axes[0, 0].set_title(r"Phase offset $\Delta \phi$ (masked)  — constant near scatterer",
                     fontsize=10)
axes[0, 0].set_xlabel("Position [m]");  axes[0, 0].set_ylabel("Depth [m]")
zoom_op(axes[0, 0]);  axes[0, 0].legend(fontsize=8)
fig.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04, label='[rad]')

# (0,1): depth-displacement map in mm
im1 = axes[0, 1].imshow(
    dz_from_phase_msk * 1e3,
    aspect='auto', origin='upper', extent=ext_mig,
    cmap='RdBu', vmin=-clip_dz_op * 1e3, vmax=clip_dz_op * 1e3, interpolation='none',
)
axes[0, 1].set_title(
    r"Vertical displacement map  $-\Delta \phi \cdot \lambda / 2\pi$",
    fontsize=10,
)
axes[0, 1].set_xlabel("Position [m]");  axes[0, 1].set_ylabel("Depth [m]")
zoom_op(axes[0, 1]);  axes[0, 1].legend(fontsize=8)
fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04, label='displacement [mm]')

# (1,0): Δφ depth profile at x = x_sc0
axes[1, 0].plot(z_mig, prof_dphi_v, color='k', lw=2.2, label=r'$\Delta \phi$ masked')
axes[1, 0].axhline(-2*np.pi*dz_shift/lam, color='purple', ls='--', lw=1.8,
                   label=f'expected  {-2*np.pi*dz_shift/lam:.3f} rad')
axes[1, 0].axhline(0, color='k', lw=0.5)
axes[1, 0].axvline(z_sc,       color='g',       ls='--', lw=1.2)
axes[1, 0].axvline(z_sc_mon_v, color='darkred', ls=':',  lw=1.2)
axes[1, 0].set_xlim(z_win_lo_v, z_win_hi_v)
axes[1, 0].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
axes[1, 0].set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])
axes[1, 0].set_xlabel("Depth [m]");  axes[1, 0].set_ylabel("Phase [rad]")
axes[1, 0].set_title(f"Δφ depth profile at x = {x_sc0*1e3:.0f} mm", fontsize=10)
axes[1, 0].legend(fontsize=8);  axes[1, 0].grid(True, alpha=0.3)

# (1,1): displacement depth profile in mm vs true shift
axes[1, 1].fill_between(z_mig, prof_dz_phase * 1e3, alpha=0.15, color='steelblue')
axes[1, 1].plot(z_mig, prof_dz_phase * 1e3, color='steelblue', lw=2.2,
                label='measured δz')
axes[1, 1].axhline(dz_shift * 1e3,  color='k', ls='--', lw=1.8,
                   label=f'true shift  {dz_shift*1e3:.2f} mm')
axes[1, 1].axhline(-dz_shift * 1e3, color='k', ls='--', lw=1.0, alpha=0.4)
axes[1, 1].axhline(dz_est_median * 1e3, color='tomato', ls=':', lw=1.8,
                   label=f'median est  {dz_est_median*1e3:.2f} mm')
axes[1, 1].axhline(0, color='k', lw=0.5)
axes[1, 1].axvline(z_sc,       color='g',       ls='--', lw=1.2)
axes[1, 1].axvline(z_sc_mon_v, color='darkred', ls=':',  lw=1.2)
axes[1, 1].set_xlim(z_win_lo_v, z_win_hi_v)
axes[1, 1].set_xlabel("Depth [m]")
axes[1, 1].set_ylabel("Estimated δz [mm]")
axes[1, 1].set_title(
    f"Displacement profile  (true = {dz_shift*1e3:.2f} mm,  "
    f"median = {dz_est_median*1e3:.2f} mm)", fontsize=10,
)
axes[1, 1].legend(fontsize=8);  axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Vertical displacement — three robust methods
#
# Why pointwise Δφ failed:
#   The Riesz-z instantaneous phase cycles with the wavelet (2π per λ).
#   Two overlapping wavelets at different depths create interference; the
#   phase difference is dominated by the local wavelet structure, not δz.
#
# Better alternatives:
#   1. Amplitude peak tracking       — trivial; works for any δz >= 1 sample
#   2. Envelope cross-correlation    — robust; lag gives δz directly
#   3. Windowed complex cross-corr.  — sub-wavelength; global phase of the
#                                      inner product ⟨f_base, f_mon⟩ avoids
#                                      per-pixel phase cycling
#                                      angle(C0) ≈ 2π·δz/λ  → δz = angle(C0)·λ/2π
# =============================================================================

from scipy.signal import correlate

ix_sc0_v = np.argmin(np.abs(x_b - x_sc0))

# Amplitude and analytic signal columns at x = x_sc0
A_col_b  = A_v_base[:, ix_sc0_v]
A_col_m  = A_v_mon[:, ix_sc0_v]
sig_b    = mig_v_base[:, ix_sc0_v]
sig_m    = mig_v_mon[:,  ix_sc0_v]
f_b_col  = sig_b + 1j * R_v_base[:, ix_sc0_v]   # analytic signal (baseline)
f_m_col  = sig_m + 1j * R_v_mon[:,  ix_sc0_v]   # analytic signal (monitor)

# ── Method 1: amplitude peak tracking ────────────────────────────────────────
iz_b_pk = np.argmax(A_col_b)
iz_m_pk = np.argmax(A_col_m)
dz_amp  = z_mig[iz_m_pk] - z_mig[iz_b_pk]

# ── Method 2: cross-correlation ───────────────────────────────────────────────
# 2a: envelope (amplitude) cross-correlation
cc_env  = correlate(A_col_m, A_col_b, mode='full')   # monitor * baseline → lag = +dz_shift
lags_z  = (np.arange(len(cc_env)) - (len(A_col_b) - 1)) * dz_b
iz_cc_env = np.argmax(cc_env)
dz_cc_env = lags_z[iz_cc_env]

# 2b: waveform (real) cross-correlation
cc_wav    = correlate(sig_m, sig_b, mode='full')   # monitor * baseline → lag = +dz_shift
dz_cc_wav = lags_z[np.argmax(cc_wav)]

# ── Method 3: windowed complex cross-correlation ──────────────────────────────
# Gaussian window centred between baseline and monitor depths
z_win_cen = 0.5 * (z_sc + z_sc_mon_v)
win_z = np.exp(-0.5 * ((z_mig - z_win_cen) / (1.5 * lam))**2)   # ~3λ FWHM

f_b_w = win_z * f_b_col
f_m_w = win_z * f_m_col
C0    = np.dot(f_b_w, np.conj(f_m_w))           # inner product  ⟨f_base, f_mon⟩

# angle(C0) ≈ 2π·δz/λ for small δz
dz_xcorr = np.angle(C0) * lam / (2 * np.pi)

# 2D displacement map using sliding-window complex cross-correlation
win_half = max(1, int(round(lam / dz_b)))        # ±1λ window half-width
dz_map_2d = np.full((len(z_mig), len(x_b)), np.nan)
for iz in range(win_half, len(z_mig) - win_half):
    sl  = slice(iz - win_half, iz + win_half + 1)
    fb2 = (mig_v_base + 1j * R_v_base)[sl, :]
    fm2 = (mig_v_mon  + 1j * R_v_mon)[sl, :]
    C2  = np.sum(fb2 * np.conj(fm2), axis=0)
    dz_map_2d[iz, :] = np.angle(C2) * lam / (2 * np.pi)
dz_map_msk = np.where(mask_v_delta, np.nan, dz_map_2d)

# ── Print summary ─────────────────────────────────────────────────────────────
print("Vertical displacement — method comparison:")
print(f"  True dz_shift                : {dz_shift*1e3:.3f} mm")
print(f"  1.  Amplitude peak tracking  : {dz_amp*1e3:.3f} mm"
      f"  (err {abs(dz_amp - dz_shift)*1e6:.0f} µm)")
print(f"  2a. Envelope cross-corr.     : {dz_cc_env*1e3:.3f} mm"
      f"  (err {abs(dz_cc_env - dz_shift)*1e6:.0f} µm)")
print(f"  2b. Waveform cross-corr.     : {dz_cc_wav*1e3:.3f} mm"
      f"  (err {abs(dz_cc_wav - dz_shift)*1e6:.0f} µm)")
print(f"  3.  Windowed complex xcorr   : {dz_xcorr*1e3:.3f} mm"
      f"  (err {abs(dz_xcorr - dz_shift)*1e6:.0f} µm)")

# ── Figure: 2 × 3 ─────────────────────────────────────────────────────────────
ext_mig = [x_b[0], x_b[-1], z_mig[-1], z_mig[0]]
x_lo_xc = x_sc0 - 5*lam;  x_hi_xc = x_sc0 + 5*lam
z_lo_xc = z_win_cen - 0.10;  z_hi_xc = z_win_cen + 0.10

def zoom_xc(ax):
    ax.set_xlim(x_lo_xc, x_hi_xc);  ax.set_ylim(z_hi_xc, z_lo_xc)
    ax.axhline(z_sc,       color='g',       ls='--', lw=1.8, label=f'base z={z_sc*1e3:.1f} mm')
    ax.axhline(z_sc_mon_v, color='darkred', ls=':',  lw=1.8, label=f'mon z={z_sc_mon_v*1e3:.1f} mm')

clip_dz2d = np.nanpercentile(np.abs(dz_map_msk), 99)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"Vertical shift — robust displacement methods  "
    f"(true = {dz_shift*1e3:.1f} mm = {shift_frac_v}\u03bb)\n"
    f"Peak tracking: {dz_amp*1e3:.2f} mm  |  "
    f"Xcorr (env): {dz_cc_env*1e3:.2f} mm  |  "
    f"Complex xcorr: {dz_xcorr*1e3:.2f} mm",
    fontsize=11, fontweight='bold',
)

# (0,0): amplitude depth profiles
axes[0,0].plot(A_col_b / A_col_b.max(), z_mig, 'b-',  lw=2.0, label='baseline envelope')
axes[0,0].plot(A_col_m / A_col_m.max(), z_mig, 'r--', lw=2.0, label='monitor envelope')
axes[0,0].plot(win_z,                   z_mig, 'k:',  lw=1.2, alpha=0.6, label='xcorr window')
axes[0,0].axhline(z_mig[iz_b_pk], color='g',       ls='--', lw=1.5, label=f'base peak {z_mig[iz_b_pk]*1e3:.1f} mm')
axes[0,0].axhline(z_mig[iz_m_pk], color='darkred', ls=':',  lw=1.5, label=f'mon peak  {z_mig[iz_m_pk]*1e3:.1f} mm')
axes[0,0].set_ylim(z_hi_xc, z_lo_xc)
axes[0,0].set_xlabel("Norm. amplitude");  axes[0,0].set_ylabel("Depth [m]")
axes[0,0].set_title(f"Method 1: Amplitude peak tracking  \u03b4z = {dz_amp*1e3:.2f} mm", fontsize=10)
axes[0,0].legend(fontsize=7);  axes[0,0].grid(True, alpha=0.3)

# (0,1): envelope cross-correlation function
axes[0,1].plot(lags_z * 1e3, cc_env / cc_env.max(), 'b-',  lw=2.0, label='envelope xcorr')
axes[0,1].plot(lags_z * 1e3, cc_wav / cc_wav.max(), 'r--', lw=1.5, alpha=0.7, label='waveform xcorr')
axes[0,1].axvline(dz_cc_env * 1e3, color='b', ls='--', lw=1.5, label=f'env peak {dz_cc_env*1e3:.2f} mm')
axes[0,1].axvline(dz_cc_wav * 1e3, color='r', ls=':',  lw=1.5, label=f'wav peak {dz_cc_wav*1e3:.2f} mm')
axes[0,1].axvline(dz_shift * 1e3,  color='k', ls='-',  lw=1.2, alpha=0.5, label=f'true {dz_shift*1e3:.2f} mm')
axes[0,1].set_xlim(-3*lam*1e3, 3*lam*1e3)
axes[0,1].set_xlabel("Lag [mm]");  axes[0,1].set_ylabel("Norm. cross-correlation")
axes[0,1].set_title(f"Method 2: Cross-correlation  env={dz_cc_env*1e3:.2f} mm  wav={dz_cc_wav*1e3:.2f} mm",
                    fontsize=10)
axes[0,1].legend(fontsize=7);  axes[0,1].grid(True, alpha=0.3)

# (0,2): sliding-window complex xcorr displacement map
im2 = axes[0,2].imshow(dz_map_msk * 1e3, aspect='auto', origin='upper', extent=ext_mig,
                        cmap='RdBu', vmin=-clip_dz2d*1e3, vmax=clip_dz2d*1e3, interpolation='none')
axes[0,2].set_title(f"Method 3: Sliding-window complex xcorr map  [mm]", fontsize=10)
axes[0,2].set_xlabel("Position [m]");  axes[0,2].set_ylabel("Depth [m]")
zoom_xc(axes[0,2]);  axes[0,2].legend(fontsize=7)
fig.colorbar(im2, ax=axes[0,2], fraction=0.046, pad=0.04, label='\u03b4z [mm]')

# (1,0): waveform comparison at x = x_sc0
axes[1,0].plot(sig_b / np.abs(sig_b).max(), z_mig, 'b-',  lw=2.0, label='baseline waveform')
axes[1,0].plot(sig_m / np.abs(sig_m).max(), z_mig, 'r--', lw=2.0, label='monitor waveform')
axes[1,0].axhline(z_sc,       color='g',       ls='--', lw=1.2)
axes[1,0].axhline(z_sc_mon_v, color='darkred', ls=':',  lw=1.2)
axes[1,0].axvline(0, color='k', lw=0.5)
axes[1,0].set_ylim(z_hi_xc, z_lo_xc)
axes[1,0].set_xlabel("Norm. amplitude");  axes[1,0].set_ylabel("Depth [m]")
axes[1,0].set_title(f"Waveforms at x = {x_sc0*1e3:.0f} mm (vertical dipole visible)", fontsize=10)
axes[1,0].legend(fontsize=8);  axes[1,0].grid(True, alpha=0.3)

# (1,1): depth profile from sliding-window complex xcorr at x = x_sc0
dz_map_col = dz_map_msk[:, ix_sc0_v]
axes[1,1].fill_between(z_mig, dz_map_col * 1e3, alpha=0.2, color='steelblue')
axes[1,1].plot(z_mig, dz_map_col * 1e3, 'steelblue', lw=2.2, label='sliding xcorr \u03b4z')
axes[1,1].axhline(dz_shift * 1e3,  color='k',   ls='--', lw=1.8, label=f'true {dz_shift*1e3:.2f} mm')
axes[1,1].axhline(dz_xcorr * 1e3,  color='tomato', ls=':',  lw=1.8,
                  label=f'global xcorr {dz_xcorr*1e3:.2f} mm')
axes[1,1].axhline(0, color='k', lw=0.5)
axes[1,1].axvline(z_sc,       color='g',       ls='--', lw=1.2)
axes[1,1].axvline(z_sc_mon_v, color='darkred', ls=':',  lw=1.2)
axes[1,1].set_xlim(z_win_lo_v, z_win_hi_v)
axes[1,1].set_xlabel("Depth [m]");  axes[1,1].set_ylabel("\u03b4z [mm]")
axes[1,1].set_title(f"Depth profile at x = {x_sc0*1e3:.0f} mm", fontsize=10)
axes[1,1].legend(fontsize=8);  axes[1,1].grid(True, alpha=0.3)

# (1,2): bar comparison of all methods vs true
methods = ['True', 'Peak\ntrack.', 'Env.\nxcorr', 'Wav.\nxcorr', 'Complex\nxcorr']
values  = [dz_shift, dz_amp, dz_cc_env, dz_cc_wav, dz_xcorr]
colors  = ['k', 'steelblue', 'royalblue', 'cornflowerblue', 'tomato']
bars    = axes[1,2].bar(methods, [v*1e3 for v in values], color=colors, alpha=0.8, edgecolor='k', lw=0.8)
axes[1,2].axhline(dz_shift * 1e3, color='k', ls='--', lw=1.5, label=f'true = {dz_shift*1e3:.2f} mm')
for bar, v in zip(bars, values):
    axes[1,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   f'{v*1e3:.2f}', ha='center', va='bottom', fontsize=9)
axes[1,2].set_ylabel("Estimated \u03b4z [mm]")
axes[1,2].set_title("Method comparison", fontsize=10)
axes[1,2].legend(fontsize=8);  axes[1,2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Vertical shift: overlaid traces + XC function + frequency-domain estimator
#
# Sub-sample estimator uses the cross-spectrum phase slope (broadband-safe).
# The windowed zero-lag complex XC only works for narrowband signals; Ricker
# wavelets have ~100% fractional bandwidth so that method is unreliable.
#
# Cross-spectrum: angle(FFT_b * conj(FFT_m))[k] = kz[k] * delta_z  (exact).
# Slope of phase vs kz gives delta_z without any narrowband assumption.
# =============================================================================
from scipy.signal import hilbert as sp_hilbert

dz_mig  = z_mig[1] - z_mig[0]          # depth sample spacing [m]
ix_apex = np.argmin(np.abs(x_b - x_sc0))
trace_b = mig_v_base[:, ix_apex]
trace_m = mig_v_mon[:,  ix_apex]

ana_b = sp_hilbert(trace_b)
ana_m = sp_hilbert(trace_m)
A_b   = np.abs(ana_b)
A_m   = np.abs(ana_m)

# ---- Real-valued XC: coarse integer-sample estimate ------------------------
xc   = np.correlate(trace_m, trace_b, mode='full')
lags = (np.arange(len(xc)) - (len(trace_b) - 1)) * dz_mig * 1e3   # mm
i_peak      = np.argmax(np.abs(xc))
lag_peak_mm = lags[i_peak]

# ---- Frequency-domain cross-spectrum slope: sub-sample estimate -----------
kz_ax = np.fft.rfftfreq(len(trace_b), d=dz_mig) * 2 * np.pi   # rad/m
TB    = np.fft.rfft(trace_b)
TM    = np.fft.rfft(trace_m)
XS    = TB * np.conj(TM)                 # cross-spectrum: phase = kz * dz
w_k   = np.abs(XS)                       # weight by coherent energy
phi_k = np.unwrap(np.angle(XS))          # unwrapped phase [rad]

# Fit only over the signal band (above 10 % of peak cross-spectral energy)
snr_mask = (w_k > 0.1 * w_k.max()) & (kz_ax > 0)
k_s, p_s, ws_s = kz_ax[snr_mask], phi_k[snr_mask], w_k[snr_mask]
A_f    = np.column_stack([k_s, np.ones(len(k_s))])
c_fit  = np.linalg.lstsq(A_f * ws_s[:, None], p_s * ws_s, rcond=None)[0]
dz_fdom, phi0 = float(c_fit[0]), float(c_fit[1])

kz       = 2 * np.pi / lam
z_cm     = z_mig * 100
z_win_cm = 4 * lam * 100

# ---- Figure -----------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle(
    f'Vertical shift -- shift_frac={shift_frac_v}  '
    f'(dz = {dz_shift*1e3:.2f} mm = lambda/{int(round(1/shift_frac_v))})'
    f'\nXC peak: {lag_peak_mm:.2f} mm  |  '
    f'Freq-domain: {dz_fdom*1e3:.3f} mm  |  '
    f'True: {dz_shift*1e3:.3f} mm',
    fontsize=11
)

# Panel 1 -- overlaid traces with envelopes
ax  = axes[0]
norm = max(np.max(np.abs(trace_b)), np.max(np.abs(trace_m)))
ax.plot(trace_b / norm, z_cm, color='steelblue', lw=1.5, label='Baseline')
ax.plot(trace_m / norm, z_cm, color='tomato',    lw=1.5, label='Monitor', alpha=0.85)
for col, A in [('steelblue', A_b), ('tomato', A_m)]:
    ax.plot( A / norm, z_cm, color=col, lw=0.9, ls='--', alpha=0.4)
    ax.plot(-A / norm, z_cm, color=col, lw=0.9, ls='--', alpha=0.4)
ax.axhline(z_sc * 100,       color='steelblue', ls=':', lw=0.9, alpha=0.7)
ax.axhline(z_sc_mon_v * 100, color='tomato',    ls=':', lw=0.9, alpha=0.7)
ax.set_xlabel('Normalised amplitude')
ax.set_ylabel('Depth [cm]')
ax.set_title('Central traces (apex column)')
ax.set_ylim(z_sc_mon_v * 100 + z_win_cm, z_sc * 100 - z_win_cm)
ax.invert_yaxis()
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)

# Panel 2 -- real-valued cross-correlation
ax2    = axes[1]
win_mm = 5 * lam * 1e3
m      = np.abs(lags) < win_mm
ax2.plot(lags[m], xc[m] / np.max(np.abs(xc[m])), color='mediumpurple', lw=1.5)
ax2.axvline(0,              color='k',      lw=0.8, ls='--')
ax2.axvline(dz_shift * 1e3, color='gray',   lw=1.4, ls=':',
            label=f'True = {dz_shift*1e3:.2f} mm')
ax2.axvline(lag_peak_mm,    color='orange', lw=1.2, ls='--',
            label=f'Peak = {lag_peak_mm:.2f} mm')
ax2.axvline(dz_fdom * 1e3,  color='green',  lw=1.2, ls='-.',
            label=f'Freq-dom = {dz_fdom*1e3:.3f} mm')
ax2.set_xlabel('Lag [mm]')
ax2.set_ylabel('Normalised XC')
ax2.set_title('Cross-correlation (monitor * baseline)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.25)

# Panel 3 -- cross-spectrum phase vs kz (should be linear, slope = dz)
ax3 = axes[2]
kz_plot = k_s * 1e-3   # rad/mm for readability
ax3.scatter(kz_plot, np.degrees(p_s), s=8, color='teal',
            alpha=0.7, label='Measured phase (signal band)')
kz_line = np.array([kz_plot.min(), kz_plot.max()])
ax3.plot(kz_line, np.degrees(dz_fdom * kz_line * 1e3 + phi0),
         color='orange', lw=1.8, ls='--',
         label=f'Fit slope: dz = {dz_fdom*1e3:.3f} mm')
ax3.axhline(0, color='k', lw=0.7, ls='--')
ax3.set_xlabel('kz [rad/mm]')
ax3.set_ylabel('angle(XS) [deg]')
ax3.set_title('Cross-spectrum phase (slope = dz)')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

print(f"XC peak-lag estimate   : {lag_peak_mm:.3f} mm")
print(f"Freq-domain estimate   : {dz_fdom*1e3:.3f} mm")
print(f"True dz                : {dz_shift*1e3:.3f} mm")
print(f"Error (freq-domain)    : {(dz_fdom*1e3 - dz_shift*1e3):+.4f} mm")


In [ ]:
# =============================================================================
# 2D cross-spectrum shift estimation — windowed-patch version
#
# Key improvement over the full-image Hanning approach:
#   The Hanning taper is applied to a small patch CENTRED on the scatterer.
#   At the window centre the taper derivative is exactly zero, so the taper
#   does not introduce a spurious phase slope that biases the slope estimate.
#   (Full-image Hanning with an off-centre scatterer causes ~1% relative error.)
# =============================================================================
from scipy.ndimage  import shift as nd_shift
from scipy.signal   import hilbert as sp_hilbert

dz_mig = (z_mig[1] - z_mig[0])
dx_mig = (x_b[1]   - x_b[0])

sf    = 1/32 #0.25
kz_c  = 2 * np.pi / lam

cases = [
    ('Vertical',   sf * lam,              0.0                  ),
    ('Horizontal', 0.0,                   sf * lam             ),
    ('Diagonal',   sf * lam / np.sqrt(2), sf * lam / np.sqrt(2)),
]


# def estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent, win_lam=3):
#     """
#     Estimate (dz, dx) from the 2D cross-spectrum phase slope.

#     Extracts a Hanning-windowed patch CENTRED on the strongest scatterer in
#     `base` before computing the FFT.  The taper derivative is zero at the
#     patch centre so the taper cannot introduce a proportional phase bias.

#     Band-limited to |kz|, |kx| < 1.5*kz_cent to prevent phase wrapping for
#     shifts up to lambda/3.

#     Returns (dz_est, dx_est, XS_patch, kz_ax, kx_ax).
#     """
#     Nz, Nx = base.shape
#     lam_g  = 2 * np.pi / kz_cent          # wavelength [m]

#     # ---- Find scatterer apex (peak of 2-D Hilbert envelope) -----------------
#     env2d         = np.abs(sp_hilbert(base, axis=0))
#     iz_c, ix_c    = np.unravel_index(np.argmax(env2d), (Nz, Nx))

#     # ---- Extract patch ±win_lam wavelengths around apex ---------------------
#     Nhz = max(int(win_lam * lam_g / dz_g), 8)
#     Nhx = max(int(win_lam * lam_g / dx_g), 8)
#     iz0 = max(0, iz_c - Nhz);  iz1 = min(Nz, iz_c + Nhz + 1)
#     ix0 = max(0, ix_c - Nhx);  ix1 = min(Nx, ix_c + Nhx + 1)

#     p_b = base[iz0:iz1, ix0:ix1]
#     p_m = mon[ iz0:iz1, ix0:ix1]

#     Nzp, Nxp = p_b.shape
#     kz_ax    = np.fft.fftfreq(Nzp, d=dz_g) * 2 * np.pi
#     kx_ax    = np.fft.fftfreq(Nxp, d=dx_g) * 2 * np.pi
#     KZ, KX   = np.meshgrid(kz_ax, kx_ax, indexing='ij')

#     # Hanning centred on the patch = centred on the scatterer
#     taper = np.outer(np.hanning(Nzp), np.hanning(Nxp))
#     XS    = np.fft.fft2(p_b * taper) * np.conj(np.fft.fft2(p_m * taper))

#     w   = np.abs(XS)
#     phi = np.angle(XS)

#     band = (np.abs(KZ) < 1.5 * kz_cent) & (np.abs(KX) < 1.5 * kz_cent)
#     mask = (w > 0.1 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

#     if mask.sum() < 3:
#         return 0.0, 0.0, XS, kz_ax, kx_ax

#     A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
#     W = w[mask]
#     c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
#     return c[0], c[1], XS, kz_ax, kx_ax

import numpy as np
from scipy.signal.windows import tukey

def estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent):
    """
    Fit a 2D phase plane phi(kz,kx) = kz*dz + kx*dx + c to the cross-spectrum.
    
    Updates:
    - Uses a Tukey window to preserve core wavelet amplitudes.
    - Dynamically bounds the fitting to where the cross-spectrum energy is 
      strong, preventing phase-wrapping errors safely.
    """
    Nz, Nx = base.shape
    
    # 1. Compute 2D wavenumber axes
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    # 2. Apply Tukey window (tapers outer 15%, leaves inner 85% pristine)
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    
    # 3. Compute Cross-Spectrum
    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon * taper)
    XS       = base_fft * np.conj(mon_fft)

    # 4. Extract weights (magnitude) and phase
    w   = np.abs(XS)
    phi = np.angle(XS)

    # 5. Dynamic Band Limit Mask
    # Limits evaluation to a stable band around center frequencies to avoid 
    # phase wrapping, while filtering out low-energy noise floors (<10% peak)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    # 6. Linear Regression (Weighted Least Squares)
    A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
    W = w[mask]  # Amplitude weights ensure reliable bins dominate the fit
    
    # Solve system: (A * W) * c = (phi * W)
    c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
    
    return c[0], c[1], XS, kz_ax, kx_ax

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
fig.suptitle(
    f'2D cross-spectrum shift estimation  |  '
    f'||shift|| = lambda/4 = {sf * lam * 1e3:.1f} mm  (all cases)',
    fontsize=12
)

for row, (name, true_dz, true_dx) in enumerate(cases):

    mon = nd_shift(mig_v_base, (true_dz / dz_mig, true_dx / dx_mig), order=3)

    dz_est, dx_est, XS, kz_ax, kx_ax = estimate_shift_2d(
        mig_v_base, mon, dz_mig, dx_mig, kz_c
    )

    XS_s   = np.fft.fftshift(XS)
    kz_s   = np.fft.fftshift(kz_ax)
    kx_s   = np.fft.fftshift(kx_ax)
    energy = np.abs(XS_s)
    phi_show = np.where(energy > 0.05 * energy.max(),
                        np.degrees(np.angle(XS_s)), np.nan)

    # Col 0: difference image
    ax   = axes[row, 0]
    diff = mon - mig_v_base
    vmax = np.max(np.abs(diff))
    ax.pcolormesh(x_b * 100, z_mig * 100, diff,
                  cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
    ax.set_title(
        f'{name}  (dz={true_dz*1e3:.1f} mm, dx={true_dx*1e3:.1f} mm)\n'
        'Difference image (mon - base)', fontsize=9
    )
    ax.set_xlabel('x [cm]');  ax.set_ylabel('z [cm]')
    ax.invert_yaxis()

    # Col 1: cross-spectrum phase (patch FFT)
    ax2 = axes[row, 1]
    im2 = ax2.pcolormesh(kx_s, kz_s, phi_show,
                         cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
    klim = 1.5 * kz_c
    for sgn in [-1, 1]:
        ax2.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax2.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
    ax2.set_title('Cross-spectrum phase [deg]  (patch FFT)\ndashed = fit band', fontsize=9)
    ax2.set_xlabel('kx [rad/m]');  ax2.set_ylabel('kz [rad/m]')
    plt.colorbar(im2, ax=ax2, fraction=0.046)

    # Col 2: results
    ax3 = axes[row, 2]
    ax3.axis('off')
    txt = (
        f'True:  dz = {true_dz * 1e3:+7.3f} mm\n'
        f'       dx = {true_dx * 1e3:+7.3f} mm\n\n'
        f'Est:   dz = {dz_est  * 1e3:+7.3f} mm\n'
        f'       dx = {dx_est  * 1e3:+7.3f} mm\n\n'
        f'Err:   dz = {(dz_est - true_dz) * 1e3:+.5f} mm\n'
        f'       dx = {(dx_est - true_dx) * 1e3:+.5f} mm'
    )
    ax3.text(0.05, 0.92, txt, transform=ax3.transAxes,
             fontsize=10, family='monospace', verticalalignment='top')
    ax3.set_title('Results', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# 2D cross-spectrum estimator: error vs shift magnitude
# Row 0: absolute error |est - true| [mm]
# Row 1: relative error |est - true| / ||true shift||  [%]
#         normalised by the total applied shift magnitude so that small and
#         large shifts are on a comparable footing.
# =============================================================================
from scipy.ndimage import shift as nd_shift

shift_fracs = np.linspace(0.02, 0.60, 40)

case_defs = [
    ('Vertical',
     lambda s: (s * lam,              0.0              ),
     1.0 / 3.0,
     'steelblue', 'cornflowerblue'),
    ('Horizontal',
     lambda s: (0.0,                  s * lam          ),
     1.0 / 3.0,
     'tomato', 'lightsalmon'),
    ('Diagonal',
     lambda s: (s * lam / np.sqrt(2), s * lam / np.sqrt(2)),
     1.0 / (3.0 * np.sqrt(2)),
     'seagreen', 'mediumaquamarine'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey='row')
fig.suptitle(
    f'2D cross-spectrum: error vs shift magnitude  '
    f'(noiseless, band-limited to 1.5*kz_c)\n'
    f'lambda = {lam*1e3:.1f} mm  |  '
    f'dz_sample = {dz_mig*1e3:.2f} mm  |  '
    f'dx_sample = {dx_mig*1e3:.2f} mm',
    fontsize=10
)

for col, (name, shift_fn, wrap_lim, col_z, col_x) in enumerate(case_defs):
    ax_abs = axes[0, col]
    ax_rel = axes[1, col]

    err_z_abs, err_x_abs = [], []
    err_z_rel, err_x_rel = [], []

    for sf in shift_fracs:
        true_dz, true_dx = shift_fn(sf)
        norm_mm = sf * lam * 1e3          # total shift magnitude [mm]

        mon = nd_shift(mig_v_base,
                       (true_dz / dz_mig, true_dx / dx_mig), order=3)
        dz_est, dx_est, *_ = estimate_shift_2d(
            mig_v_base, mon, dz_mig, dx_mig, kz_c
        )

        ez = abs(dz_est - true_dz) * 1e3   # mm
        ex = abs(dx_est - true_dx) * 1e3

        err_z_abs.append(max(ez, 1e-5))
        err_x_abs.append(max(ex, 1e-5))
        err_z_rel.append(max(ez / norm_mm * 100, 1e-4))   # percent
        err_x_rel.append(max(ex / norm_mm * 100, 1e-4))

    err_z_abs = np.array(err_z_abs)
    err_x_abs = np.array(err_x_abs)
    err_z_rel = np.array(err_z_rel)
    err_x_rel = np.array(err_x_rel)

    for ax, y_z, y_x, ylabel, ref_lines in [
        (ax_abs, err_z_abs, err_x_abs,
         'Absolute error [mm]',
         [(dz_mig * 1e3,   'dimgray',    f'Sample spacing ({dz_mig*1e3:.2f} mm)'),
          (lam * 1e3 / 10, 'darkorange', f'lambda/10 ({lam*1e3/10:.1f} mm)'),
          (lam * 1e3 / 100,'goldenrod',  f'lambda/100 ({lam*1e3/100:.2f} mm)')]),
        (ax_rel, err_z_rel, err_x_rel,
         'Relative error [%]',
         [(100,   'dimgray',    '100 % (error = shift)'),
          (10,    'darkorange', '10 %'),
          (1,     'goldenrod',  '1 %'),
          (0.1,   'lightgreen', '0.1 %')]),
    ]:
        ax.semilogy(shift_fracs, y_z, '-o',  ms=4, lw=1.5,
                    color=col_z, label='|err dz|')
        ax.semilogy(shift_fracs, y_x, '--s', ms=4, lw=1.5,
                    color=col_x, label='|err dx|')

        for level, color, label in ref_lines:
            ax.axhline(level, color=color, ls=':', lw=1.1, alpha=0.8,
                       label=label)

        ax.axvline(wrap_lim, color='red', ls='--', lw=1.3,
                   label=f'Wrap limit ({wrap_lim:.2f} lambda)')

        ax.set_xlabel('Shift magnitude [lambda]')
        if col == 0:
            ax.set_ylabel(ylabel)
        ax.set_xlim(shift_fracs[0] - 0.01, 0.62)
        ax.legend(fontsize=7, loc='upper left')
        ax.grid(True, alpha=0.2, which='both')

    ax_abs.set_title(f'{name} shift')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from scipy.signal import hilbert, correlate
import matplotlib.pyplot as plt


# ─────────────────────────────────────────────────────────────────────────────
# Public API
# ─────────────────────────────────────────────────────────────────────────────

def analyze_vertical_shift(
    mig_base: np.ndarray,
    mig_mon:  np.ndarray,
    z: np.ndarray,
    x: np.ndarray,
    fc: float,
    v:  float,
    thresh_frac:    float = 0.10,
    win_width_lam:  float = 2.0,
    plot: bool = True,
) -> dict:
    """
    Estimate sub-wavelength vertical shift δz between two migrated GPR B-scans.

    Parameters
    ----------
    mig_base, mig_mon : (Nz, Nx) ndarray
        Migrated baseline and monitor images (real-valued).
    z : (Nz,) ndarray  — depth axis [m]
    x : (Nx,) ndarray  — lateral axis [m]
    fc : float         — GPR centre frequency [Hz]
    v  : float         — medium velocity [m/s]
    thresh_frac : float
        Amplitude masking threshold (fraction of peak joint amplitude).
    win_width_lam : float
        Half-width of Gaussian correlation window in wavelengths.
    plot : bool        — produce diagnostic figure if True

    Returns
    -------
    dict
        dz_env_cc        : δz from amplitude-envelope cross-correlation [m]
        dz_wav_cc        : δz from waveform cross-correlation [m]
        dz_xcorr         : δz from windowed complex cross-correlation [m]  ← primary
        dz_regression    : δz from linear regression on phase profile [m]
        dz_phase_peak    : δz from pointwise phase at peak (QC only) [m]
        kz_effective     : measured kz from regression [rad/m]
        dz_map_2d        : (Nz, Nx) 2D displacement map [m]
        phi_delta_masked : (Nz, Nx) masked phase-difference image [rad]
        diff             : (Nz, Nx) monitor − baseline [same units as input]
        mask             : (Nz, Nx) bool — True where amplitude below threshold
    """
    lam = v / fc
    kz  = 2.0 * np.pi / lam          # theoretical wavenumber [rad/m]
    dz  = float(z[1] - z[0])
    Nz, Nx = mig_base.shape

    # 1. Analytic signals via Hilbert transform along depth axis ──────────────
    ana_base = hilbert(mig_base, axis=0)   # (Nz, Nx) complex
    ana_mon  = hilbert(mig_mon,  axis=0)

    A_base = np.abs(ana_base)
    A_mon  = np.abs(ana_mon)

    # 2. Dipole image ─────────────────────────────────────────────────────────
    diff = mig_mon - mig_base

    # 3. Amplitude mask ───────────────────────────────────────────────────────
    A_joint = 0.5 * (A_base + A_mon)
    mask    = A_joint < thresh_frac * np.nanmax(A_joint)   # True → masked

    # 4. Phase difference (wrap-safe) ─────────────────────────────────────────
    phi_delta      = np.angle(ana_mon * np.conj(ana_base))
    phi_delta_msk  = np.where(mask, np.nan, phi_delta)

    # 5. Apex location ────────────────────────────────────────────────────────
    # iz_pk, ix_pk = np.unravel_index(np.nanargmax(A_joint), A_joint.shape)
    # Use BASELINE apex, not joint apex
    iz_pk_b, ix_pk = np.unravel_index(np.argmax(A_base), A_base.shape)
    iz_pk   = iz_pk_b     # rename for clarity; keep for all later uses
    # dphi_peak     = phi_delta_msk[iz_pk, ix_pk]
    # dz_phase_peak = _safe_dz(dphi_peak, kz)
    # Phase at peak (sign is correct as-is via _safe_dz)
    dphi_peak     = phi_delta_msk[iz_pk, ix_pk]
    dz_phase_peak = _safe_dz(dphi_peak, kz)       # still -dphi/kz = +δz ✓

    # 6. 1-D cross-correlations at apex column ────────────────────────────────
    A_col_b = A_base[:, ix_pk]
    A_col_m = A_mon[:,  ix_pk]
    col_b   = mig_base[:, ix_pk]
    col_m   = mig_mon[:,  ix_pk]

    # Convention: correlate(monitor, baseline) → peak at lag = +δz for downward shift
    cc_env  = correlate(A_col_m, A_col_b, mode='full')
    cc_wav  = correlate(col_m,   col_b,   mode='full')
    lags_z  = (np.arange(len(cc_env)) - (Nz - 1)) * dz

    dz_env_cc = lags_z[np.argmax(cc_env)]
    # Waveform CC: take argmax of abs (half-cycle ambiguity)
    dz_wav_cc = lags_z[np.argmax(np.abs(cc_wav))]

    # 7. Windowed complex cross-correlation (1-D at apex column) ──────────────
    win_sig  = win_width_lam * lam
    gauss_w  = np.exp(-0.5 * ((z - z[iz_pk]) / win_sig) ** 2)

    # C = Σ_z  w · f̃_base · conj(f̃_mon)  →  angle(C) = −kz·δz
    # C0       = np.sum(gauss_w * ana_base[:, ix_pk] * np.conj(ana_mon[:, ix_pk]))
    # dz_xcorr = _safe_dz(np.angle(C0), kz)
    # Complex XC — use +angle, NOT _safe_dz
    C0       = np.sum(gauss_w * ana_base[:, ix_pk] * np.conj(ana_mon[:, ix_pk]))
    dz_xcorr = float(np.angle(C0) / kz)           # +angle → +δz ✓

    print(f"angle(C0)     = {np.angle(C0)*180/np.pi:.2f} deg  (expect {360 * shift_frac:.1f} deg)")
    print(f"A_base apex z = {z[iz_pk_b]*100:.1f} cm  (expect {z_sc*100:.1f} cm)")
    print(f"phi_delta peak = {np.degrees(dphi_peak):.1f} deg  (expect {-360*shift_frac:.1f} deg)")

    # 8. Linear regression calibration on phase profile at apex column ────────
    dz_regression, kz_effective = _phase_regression(
        phi_delta_msk[:, ix_pk], z, iz_pk, kz
    )

    # 9. 2-D sliding-window displacement map ──────────────────────────────────
    # dz_map_2d = _sliding_window_map(ana_base, ana_mon, z, kz, win_sig, dz, mask)
    # ── In _sliding_window_map ─────────────────────────────────────────────────
    dz_map_2d[iz, :] = np.angle(C2) / kz          # was: -np.angle(C2)/kz

    # 10. Diagnostic figure ───────────────────────────────────────────────────
    if plot:
        _plot(
            mig_base, mig_mon, diff, phi_delta_msk, dz_map_2d,
            A_col_b, A_col_m, lags_z, cc_env, cc_wav,
            z, x, iz_pk, ix_pk, lam,
            dz_env_cc, dz_wav_cc, dz_xcorr, dz_regression, dz_phase_peak,
        )

    return dict(
        dz_env_cc        = dz_env_cc,
        dz_wav_cc        = dz_wav_cc,
        dz_xcorr         = dz_xcorr,
        dz_regression    = dz_regression,
        dz_phase_peak    = dz_phase_peak,
        kz_effective     = kz_effective,
        dz_map_2d        = dz_map_2d,
        phi_delta_masked  = phi_delta_msk,
        diff             = diff,
        mask             = mask,
    )


# ─────────────────────────────────────────────────────────────────────────────
# Private helpers
# ─────────────────────────────────────────────────────────────────────────────

def _safe_dz(dphi: float, kz: float) -> float:
    """δz = −Δφ / kz; returns NaN if Δφ is NaN."""
    return float(-dphi / kz) if not np.isnan(dphi) else np.nan


# def _phase_regression(
#     phi_col: np.ndarray,
#     z: np.ndarray,
#     iz_pk: int,
#     kz_theory: float,
# ) -> tuple[float, float]:
#     """
#     Fit Δφ(z) = a + b·z over the valid (unmasked) depth samples near the apex.

#     For a pure rigid translation, the slope b ≈ 0 and the intercept a = −kz·δz.
#     A non-zero slope b indicates dispersive or chirped phase behaviour.

#     Returns
#     -------
#     dz_regression : δz inferred from the fit evaluated at apex depth [m]
#     kz_effective  : measured kz = 2π/λ_eff from the phase profile slope [rad/m]
#                     (equals kz_theory when b=0)
#     """
#     valid = ~np.isnan(phi_col)
#     if valid.sum() < 3:
#         return np.nan, kz_theory

#     z_v   = z[valid]
#     phi_v = phi_col[valid]

#     # Unwrap phase before fitting to handle potential wrapping near ±π
#     phi_v_uw = np.unwrap(phi_v)

#     # Linear fit:  Δφ(z) ≈ a + b·z
#     b, a = np.polyfit(z_v, phi_v_uw, 1)

#     phi_at_apex   = a + b * z[iz_pk]          # fitted value at scatterer depth
#     dz_regression = float(-phi_at_apex / kz_theory)

#     # Effective kz: if phase were −kz·(z − z_ref), slope = −kz
#     # Here we report |b| as the measured chirp/wavenumber offset
#     kz_effective = kz_theory   # b ≈ 0 for a clean point scatterer

#     return dz_regression, kz_effective

def _phase_regression(phi_col, z, iz_pk, kz_theory):
    valid = ~np.isnan(phi_col)
    if valid.sum() < 3:
        return np.nan, kz_theory
    phi_median    = float(np.nanmedian(phi_col[valid]))
    dz_regression = -phi_median / kz_theory    # -dphi/kz (same sign as dz_phase_peak)
    return dz_regression, kz_theory



def _sliding_window_map(
    ana_base: np.ndarray,
    ana_mon:  np.ndarray,
    z: np.ndarray,
    kz: float,
    win_sig: float,
    dz: float,
    mask: np.ndarray,
) -> np.ndarray:
    """2-D sliding Gaussian-window complex cross-correlation map."""
    Nz, Nx = ana_base.shape
    win_half  = max(1, int(round(win_sig / dz)))
    dz_map    = np.full((Nz, Nx), np.nan)

    for iz in range(win_half, Nz - win_half):
        z_sl  = z[iz - win_half : iz + win_half + 1]
        gauss = np.exp(-0.5 * ((z_sl - z[iz]) / win_sig) ** 2)[:, None]  # (W,1)
        sl    = slice(iz - win_half, iz + win_half + 1)

        # C[x] = Σ_z  w · f̃_base · conj(f̃_mon)
        C = np.sum(gauss * ana_base[sl, :] * np.conj(ana_mon[sl, :]), axis=0)
        dz_map[iz, :] = -np.angle(C) / kz

    dz_map[mask] = np.nan
    return dz_map


# ─────────────────────────────────────────────────────────────────────────────
# Diagnostic figure
# ─────────────────────────────────────────────────────────────────────────────

def _plot(
    mig_base, mig_mon, diff, phi_delta_msk, dz_map_2d,
    A_col_b, A_col_m, lags_z, cc_env, cc_wav,
    z, x, iz_pk, ix_pk, lam,
    dz_env_cc, dz_wav_cc, dz_xcorr, dz_regression, dz_phase_peak,
):
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle('Vertical Shift Analysis — Phase-Informed Timelapse Migration', fontsize=13)

    ext = [x[0] * 100, x[-1] * 100, z[-1] * 100, z[0] * 100]  # cm

    # ── Row 0: migrated images + dipole ──────────────────────────────────────
    vmax   = max(np.nanmax(np.abs(mig_base)), np.nanmax(np.abs(mig_mon)))
    vmax_d = np.nanmax(np.abs(diff))
    for ax, img, title, vr in zip(
        axes[0],
        [mig_base, mig_mon, diff],
        ['Migrated — Baseline', 'Migrated — Monitor', 'Dipole (Mon − Base)'],
        [vmax, vmax, vmax_d],
    ):
        ax.imshow(img, aspect='auto', extent=ext, cmap='seismic', vmin=-vr, vmax=vr)
        ax.axhline(z[iz_pk] * 100, color='lime', lw=0.8, ls='--', alpha=0.7)
        ax.axvline(x[ix_pk] * 100, color='lime', lw=0.8, ls='--', alpha=0.7)
        ax.set(title=title, xlabel='x [cm]', ylabel='z [cm]')

    # ── Row 1: phase difference, 2-D displacement map, amplitude profiles ────
    im_phi = axes[1, 0].imshow(
        phi_delta_msk, aspect='auto', extent=ext, cmap='hsv', vmin=-np.pi, vmax=np.pi
    )
    axes[1, 0].set(title='Phase Difference Δφ (masked)', xlabel='x [cm]', ylabel='z [cm]')
    plt.colorbar(im_phi, ax=axes[1, 0], label='Δφ [rad]', shrink=0.85)

    vd = np.nanmax(np.abs(dz_map_2d[~np.isnan(dz_map_2d)])) * 1e3 \
         if np.any(~np.isnan(dz_map_2d)) else 1.0
    im_map = axes[1, 1].imshow(
        dz_map_2d * 1e3, aspect='auto', extent=ext, cmap='RdBu_r', vmin=-vd, vmax=vd
    )
    axes[1, 1].set(title='2-D Displacement Map δz [mm]', xlabel='x [cm]', ylabel='z [cm]')
    plt.colorbar(im_map, ax=axes[1, 1], label='δz [mm]', shrink=0.85)

    axes[1, 2].plot(A_col_b / A_col_b.max(), z * 100, color='steelblue', label='Baseline')
    axes[1, 2].plot(A_col_m / A_col_m.max(), z * 100, color='tomato',    label='Monitor')
    axes[1, 2].axhline(z[iz_pk] * 100, color='grey', ls='--', lw=0.8)
    axes[1, 2].invert_yaxis()
    axes[1, 2].set(title='Envelope Profiles at x_apex', xlabel='Norm. amplitude', ylabel='z [cm]')
    axes[1, 2].legend(fontsize=8)

    # ── Row 2: cross-correlations, summary bar, pointwise phase profile ───────
    axes[2, 0].plot(lags_z * 1e3, cc_env / cc_env.max(),          'steelblue',  label='Envelope CC')
    axes[2, 0].plot(lags_z * 1e3, cc_wav / np.abs(cc_wav).max(),  'tomato',     label='Waveform CC', ls='--')
    for val, col, label in [
        (dz_env_cc, 'steelblue', f'δz_env = {dz_env_cc*1e3:.1f} mm'),
        (dz_wav_cc, 'tomato',    f'δz_wav = {dz_wav_cc*1e3:.1f} mm'),
    ]:
        axes[2, 0].axvline(val * 1e3, color=col, ls=':', lw=1.2, label=label)
    axes[2, 0].set(title='1-D Cross-Correlations (apex col)', xlabel='Lag [mm]', ylabel='Norm. CC')
    axes[2, 0].legend(fontsize=7)

    methods = ['Env\nCC', 'Wav\nCC', 'Complex\nXC', 'Regression', 'Phase\nPeak']
    vals_mm = np.array([dz_env_cc, dz_wav_cc, dz_xcorr, dz_regression, dz_phase_peak]) * 1e3
    colors  = ['steelblue', 'tomato', 'seagreen', 'mediumpurple', 'goldenrod']
    bars    = axes[2, 1].bar(methods, vals_mm, color=colors, width=0.6)
    axes[2, 1].axhline(0, color='k', lw=0.8)
    axes[2, 1].set(title='δz Estimates by Method', ylabel='δz [mm]')
    for bar, val in zip(bars, vals_mm):
        if not np.isnan(val):
            axes[2, 1].text(
                bar.get_x() + bar.get_width() / 2, val,
                f'{val:.1f}', ha='center',
                va='bottom' if val >= 0 else 'top', fontsize=8
            )

    phi_col = phi_delta_msk[:, ix_pk]
    dz_col  = -phi_col / (2 * np.pi / lam) * 1e3   # mm
    axes[2, 2].plot(dz_col, z * 100, color='purple', label='Pointwise Δφ·λ/2π')
    axes[2, 2].axvline(dz_xcorr * 1e3, color='seagreen', ls='--',
                       label=f'Complex XC: {dz_xcorr*1e3:.1f} mm')
    axes[2, 2].axvline(dz_regression * 1e3, color='mediumpurple', ls=':',
                       label=f'Regression: {dz_regression*1e3:.1f} mm')
    axes[2, 2].invert_yaxis()
    axes[2, 2].set(title='Pointwise δz Profile (apex col)', xlabel='δz [mm]', ylabel='z [cm]')
    axes[2, 2].legend(fontsize=7)

    plt.tight_layout()
    plt.show()


In [ ]:
# -- Acquisition --------------------------------------------------------------
dx_b   = 0.005 / 4
n_x_b  = 160 * 4
x_b    = np.arange(n_x_b) * dx_b       # 0 -- 0.795 m profile
x_sc0  = x_b.mean()

dt_b   = 0.004
n_t_b  = 1300                           # t_max = 5.196 ns > TWT(0.30 m) = 3.57 ns
t_b    = np.arange(n_t_b) * dt_b

z_sc   = 0.30
z_mig  = np.arange(0, 0.42, dx_b)      # 84 depth steps

In [ ]:

x_rec = x_b

In [ ]:
# ── Synthetic example (mirrors notebook geometry) ──────────────────────────
fc  = 1e9          # 1 GHz
v   = 0.1          # 0.1 m/ns = 10 cm/ns
lam = v / fc       # 10 cm

# Supply your migrated arrays and axes — e.g. from gazdag_migration():
results = analyze_vertical_shift(
    mig_base   = mig_v_base,        # (Nz, Nx) real array
    mig_mon    = mig_v_mon,         # (Nz, Nx) real array
    z          = z_mig,             # depth axis [m]
    x          = x_rec,             # lateral axis [m]
    fc         = fc,
    v          = v,
    thresh_frac    = 0.10,          # mask below 10% of peak
    win_width_lam  = 1.0,           # 2λ Gaussian window
    plot           = True,
)

print(f"δz (complex XC)   : {results['dz_xcorr']*1e3:+.2f} mm")
print(f"δz (env CC)       : {results['dz_env_cc']*1e3:+.2f} mm")
print(f"δz (regression)   : {results['dz_regression']*1e3:+.2f} mm")
print(f"δz (phase peak)   : {results['dz_phase_peak']*1e3:+.2f} mm  [QC only]")

# ── Multi-shift calibration curve ──────────────────────────────────────────
shift_fracs = np.array([1/16, 1/8, 1/4, 3/8, 1/2])
dz_true  = shift_fracs * lam
dz_meas  = []   # fill with results['dz_xcorr'] for each pair

# Calibration slope (should equal 1.0 for perfect recovery):
# cal = np.polyfit(dz_true, dz_meas, 1)
# print(f"Calibration slope: {cal[0]:.4f}  (ideal = 1.0)")
